# CountGD Open-World Object Counting Pipeline — DIMER E2E counting tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/countgd-object-counting-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/countgd-object-counting-pipeline/blob/main/tutorials/countgd_object_counting_colab.ipynb) [![Hugging Face Space](https://img.shields.io/badge/%F0%9F%A4%97%20Space-nikigoli%2Fcountgd-ffcc4d?style=flat)](https://huggingface.co/spaces/nikigoli/countgd) [![Upstream](https://img.shields.io/badge/Upstream-niki--amini--naieni%2FCountGD-181717?style=flat&logo=github&logoColor=white)](https://github.com/niki-amini-naieni/CountGD) [![arXiv](https://img.shields.io/badge/arXiv-2407.04619-b31b1b.svg)](https://arxiv.org/abs/2407.04619)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** open-world object counting from a text prompt, up to three exemplar boxes, or both — a count, one box and one point per counted object — with count, point-localisation and box-IoU evaluation and a bounded counting fine-tune, using the pinned CountGD checkpoint

**This notebook is standalone.** It carries the repository's package (8 modules under `src/countgd_pipeline/`, at revision `6ec56df6692e`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable Space revision `6e82e59569a84ee5c6aafa35d396f2d2bee57be2` (~1251 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the authors' pinned CountGD snapshot (a 1.25 GB pickle checkpoint, statically audited and converted once in this kernel into the 938 MB `countgd.safetensors` that is actually loaded) and the BERT tokenizer snapshot, draws the seeded synthetic counting scenes (24 / 8 / 12 training, validation and test scenes with known object boxes) and fetches the 80 pinned FSC-147 test photographs (about 2.7 MB, each refused on any byte-size or SHA-256 mismatch), validates and splits both without leakage, counts the demo scene by text, by exemplar boxes and by both through the inference contract with an input manifest and a rejection probe, scores the frozen model beside a mean-count baseline and a template matcher on the held-out scenes (count error, point localisation and box IoU) and on 24 FSC-147 photographs, runs a bounded counting fine-tune of the last two decoder layers and the shared box head on the training scenes with validation-MAE epoch selection, scores both held-out sets again, re-counts the demo scene, exports the adapter as safetensors with a manifest and reloads it into a fresh pipeline to verify count and box parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). A CUDA runtime is used automatically when present (a Kaggle T4 finishes the model work in minutes); on CPU the path is about 20 minutes of model time on the build workstation, several times longer on a 2-vCPU hosted runtime.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload one zip (or also set `BYOD_ZIP_PATH` to a zip already in the runtime, which skips the upload dialog) holding a `labels.csv` (columns `id`, `file`, `label`, `count`; optional `points`, `exemplars` and `boxes` as JSON lists) beside the image files — at least eight images. Your records replace the synthetic scenes and pass through the same validation, seeded split, baselines, fine-tune, held-out evaluation, artifact export and reload-parity cells; the FSC-147 check is skipped. Records with `boxes` train on their object extents and are scored by box IoU; records with points only train on upstream's point boxes and are scored by point localisation. The schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

CountGD (Amini-Naieni, Han and Zisserman, NeurIPS 2024) counts the objects in an image that a prompt describes: a **text prompt** naming the category, up to three **exemplar boxes** drawn around single instances, or both. It is GroundingDINO's open-set detector — a Swin-B image backbone, a BERT text encoder, a feature-enhancer that fuses them, and a six-layer decoder with 900 queries — extended so that each exemplar box is pooled from the image features and entered as an extra prompt token beside the text. Every query predicts a box and a similarity to each prompt token; the queries whose best score exceeds a threshold (0.23, upstream's) are the counted objects, so the model returns a **count, one box and one point per counted object** — 233,362,816 parameters, fine-tuned on FSC-147, published by its authors under the **MIT** licence. The score is a sigmoid similarity, uncalibrated: the count is a threshold decision, not a probability.

This notebook reads the model three ways. The **count error** (MAE and RMSE — FSC-147's own metrics) says how many; **point localisation** (predicted points matched one-to-one to gold points within a radius) says whether it counted the right things; and **box IoU** (predicted boxes matched one-to-one to gold object boxes at IoU ≥ 0.5) says whether the boxes enclose them. FSC-147 annotates one point per object and three exemplar boxes per image, so it can score the first two but not the third; the tutorial therefore draws **synthetic counting scenes with known object boxes** — targets of one colour and shape among distractors of another — which score all three and are the data the fine-tune trains on, and keeps 24 real FSC-147 test photographs (MIT-licensed mirror; categories CountGD never saw in training) as the check that the fine-tune did not break counting on real images.

What this notebook adds to inference is a **bounded counting fine-tune**: the last two decoder layers, the decoder's final norm and the shared box head — 3,619,584 of 233,362,816 parameters — trained on upstream's own objective (token focal loss and L1 box loss after Hungarian matching) with the gold object boxes as targets, the epoch chosen on the validation scenes' count error. The honest question is narrow: does a few minutes of that move the count error and the boxes on held-out scenes, against a **mean-count baseline** and a **template matcher** built from the same exemplars, and does it leave the FSC-147 photographs alone? The build record's answer is in Sections 6 and 8. Nothing here is a quality claim about your images: it is one seeded draw of synthetic scenes and 24 photographs.

**Snapshot note:** the authors distribute the paper checkpoint as a 1.25 GB PyTorch pickle (`checkpoint_best_regular.pth` in their Space, byte-identical to the Google Drive file their README links). A pickle can execute code when it is opened, so the carried package audits it **statically** first — every global its `data.pkl` would import is listed without executing anything and must be one of five allowed names (four torch tensor builders and the `argparse.Namespace` of the training arguments) — then opens it once with torch's restricted unpickler, keeps the `model` state dict and writes `countgd.safetensors`, whose SHA-256 is pinned. Only that file is loaded. The GroundingDINO network is carried as vendored code with the multi-scale deformable attention in pure PyTorch (`grid_sample`), so no compiled CUDA extension is needed on CPU or GPU.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the immutable upstream snapshot and see a pickle checkpoint audited statically and converted into safetensors before anything loads it; draw seeded synthetic counting scenes with known boxes and fetch a digest-pinned set of FSC-147 photographs, validate both and split them without leakage; count one scene by text, by exemplar boxes and by both through the public API and read what each prompt mode counts; read a count, a set of points and a set of boxes correctly (uncalibrated scores, a threshold decision); measure the frozen model's count error, point localisation and box IoU beside a mean-count baseline and a template matcher; run a bounded counting fine-tune with explicit hyperparameters and validation-based epoch selection; evaluate on held-out scenes and on real photographs; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** object detection or segmentation as a product (the boxes are a reading of what was counted, not a detector to ship), counting by density map, video or tracking, dense scenes beyond one 800-pixel pass (FSC-147 images reach 3,701 objects; the upstream test-time cropping for those is not carried), the upstream SAM-based test-time normalisation, calibrated confidence, full fine-tuning or training from scratch, evaluation on the full FSC-147 benchmark (only 24 of its test photographs are scored here), and any claim that coloured shapes stand in for your images. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab, Kaggle or Jupyter, Python 3.12). CUDA is used automatically when available; the default path also runs on CPU (float32). The build record measured 4.6 s per image to count on the build workstation's CPU and 4.7 s per fine-tuning step, so the whole default path is about 20 minutes of model time there after the downloads (a 2-vCPU hosted runtime will be several times slower); a hosted T4 finishes it in minutes. The pinned `torch==2.14.0` install and the 1.25 GB checkpoint are the large downloads; the conversion writes a further 938 MB, so allow about 3 GB of disk.
- **Knowledge:** basic Python and PIL; what a bounding box and intersection-over-union are; what MAE and RMSE measure; why a score above a threshold is a decision and not a probability; what fine-tuning the last layers of a network changes and what it leaves alone.
- **Data contract:** records are `{id, image, label, count}` with optional `points` (`[x, y]`, one per counted object), `exemplars` (0..3 boxes `[x0, y0, x1, y1]` around single instances) and `boxes` (one full object box per counted object) in image pixels. Images are PIL images or files decodable by Pillow with sides between `MIN_IMAGE_SIDE` (32) and `MAX_IMAGE_SIDE` (4096) px; the label is the category to count, at most 64 plain characters; `count` agrees with the points and the boxes when they are given (box centres stand in for missing points). Ids match `[A-Za-z0-9_.:-]{1,64}` and are unique; a dataset needs 4..20,000 records. Every image is resized to a shortest side of 800 px (longest at most 1333) like upstream's test transform. Evaluation needs a gold count; adaptation needs points or boxes. BYOD accepts one zip of images plus a `labels.csv` in that shape.
- **Validation is structural, not semantic:** every image is decoded and every annotation checked against the image and against the count, but nothing checks that a label names what the boxes enclose — a mislabelled set is counted without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** besides the model and tokenizer snapshots, the default path fetches 80 JPEG files from the Hugging Face dataset mirror `isentropic/FSC147` at revision `3e420cb6537e…` (about 2.7 MB in total), each pinned by byte size and SHA-256 in the carried `samples.py` and refused on any mismatch. The mirror is published under MIT; FSC-147's authors collected the images from the web and state no per-image licence. The synthetic scenes are drawn in the kernel. Nothing is committed to the repository.
- **External access:** the Hugging Face Hub only, to fetch the pinned `nikigoli/countgd` snapshot (~1251 MB in total) at revision `6e82e59569a8…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `numpy` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'huggingface-hub==0.36.2',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'safetensors==0.8.0',
    'scipy==1.18.1',
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
]
NOTEBOOK_SOURCE = {
    'repository': 'countgd-object-counting-pipeline',
    'repository_revision': '6ec56df6692ecd703262b9159d5dc5bb160c9aab',
    'embedded_module': 'src/countgd_pipeline/config.py',
    'embedded_modules': ['src/countgd_pipeline/config.py', 'src/countgd_pipeline/metrics.py', 'src/countgd_pipeline/modeling.py', 'src/countgd_pipeline/synthetic.py', 'src/countgd_pipeline/model.py', 'src/countgd_pipeline/provenance.py', 'src/countgd_pipeline/samples.py', 'src/countgd_pipeline/pipeline.py'],
    'module_sha256': '106c90d8b9c0bab01dcaa843886353c7822011d8ab310c141765593e07e568da',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, numpy
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'numpy': numpy.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/countgd_pipeline/` @ `6ec56df6692e`)

The next 8 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (2 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/8:** `src/countgd_pipeline/config.py`

In [ ]:
# ruff: noqa: E501
from __future__ import annotations

# ------------------------------------------------------------------ the model snapshot (weights)
# The pinned snapshot is the authors' Hugging Face Space, which hosts the paper checkpoint byte-for-byte: its
# `checkpoint_best_regular.pth` has the SHA-256 of the file the upstream README links on Google Drive
# (`checkpoint_fsc147_best.pth`, 1.2 GB; both downloaded and hashed on 2026-09-24). The checkpoint is a pickle,
# so it is audited and converted once into the safetensors file the package serves (see docs/WEIGHTS.md).
MODEL_ID = "nikigoli/countgd"
MODEL_REPO_TYPE = "space"
MODEL_REVISION = "6e82e59569a84ee5c6aafa35d396f2d2bee57be2"
MODEL_LICENSE = "MIT"
DEFAULT_MODEL_KEY = "countgd"
SOURCE_CKPT_NAME = "checkpoint_best_regular.pth"
SOURCE_CKPT_SHA256 = "c1bab864b17db345b4c6e3aaabb5765bc2c0a90d0bc8defb5e664a74a50aa126"
SOURCE_CKPT_SIZE_BYTES = 1_250_122_522
SOURCE_DRIVE_ID = "1RbRcNLsOfeEbx6u39pBehqsgQiexHHrI"  # the upstream README's link; same bytes as the Space file
ALLOWED_CHECKPOINT_FILES = ("README.md", SOURCE_CKPT_NAME)  # the manifest-listed files of the Space snapshot
# Every GLOBAL the checkpoint's pickle names (static `pickletools` audit; nothing executed). `argparse.Namespace`
# is the training-arguments object; it is the one class beyond torch's default restricted-unpickler set.
PICKLE_ALLOWED_GLOBALS = (
    "argparse.Namespace",
    "collections.OrderedDict",
    "torch.FloatStorage",
    "torch.LongStorage",
    "torch._utils._rebuild_tensor_v2",
)
PICKLE_AUDIT_SHA256 = "4606eaf365d27d2fddd901bdc069218dabbd418f9856ed2d0e3707612ad4c527"  # sha256 of the sorted names, newline-joined
SOURCE_STATE_KEY = "model"  # the checkpoint also carries args, epoch (19), lr_scheduler and optimizer, which are not read
SOURCE_STATE_TENSORS = 1_146
DROPPED_PREFIX = "feature_map_encoder."  # 38 tensors of an exemplar encoder the forward pass never calls
# The converted serving file: the model's state dict with each tied box-head tensor stored once and the 66
# aliases recorded as one JSON metadata entry (safetensors writes a multi-key metadata map in hash order, so
# only a single key keeps the file's digest reproducible).
MODEL_FILENAME = "countgd.safetensors"
MODEL_SHA256 = "8e44867b951e3a4205d918e022b78bc5fea218fd17c1851b864a01c421d2d443"
MODEL_SIZE_BYTES = 937_560_480
ALIAS_METADATA_KEY = "countgd_aliases"
# The upstream code the vendored `modeling.py` is carried from.
UPSTREAM_REPOSITORY = "niki-amini-naieni/CountGD"
UPSTREAM_COMMIT = "b6f362b3f5cd20db4a171faa410dfed8f2f466d8"
UPSTREAM_LICENSE = "MIT"  # the GroundingDINO parts carry IDEA's Apache-2.0 header

# ------------------------------------------------------------------ the text-tokenizer snapshot (no weights)
TOKENIZER_MODEL_ID = "google-bert/bert-base-uncased"
TOKENIZER_REVISION = "86b5e0934494bd15c9632b12f734a8a67f723594"
TOKENIZER_LICENSE = "Apache-2.0"
TOKENIZER_KEY = "bert-base-uncased"
TOKENIZER_FILES = ("LICENSE", "README.md", "config.json", "tokenizer.json", "tokenizer_config.json", "vocab.txt")
# BERT's weights are *inside* the CountGD checkpoint (`bert.*`, 199 tensors); only its vocabulary, tokenizer
# configuration and architecture configuration come from this snapshot.

# ------------------------------------------------------------------ architecture (config/cfg_fsc147_vit_b_test.py upstream)
MODEL_CONFIG: dict[str, object] = {
    "modelname": "groundingdino",
    "backbone": "swin_B_384_22k",
    "position_embedding": "sine",
    "pe_temperatureH": 20,
    "pe_temperatureW": 20,
    "return_interm_indices": [1, 2, 3],
    "backbone_freeze_keywords": None,
    "enc_layers": 6,
    "dec_layers": 6,
    "pre_norm": False,
    "dim_feedforward": 2048,
    "hidden_dim": 256,
    "dropout": 0.0,
    "nheads": 8,
    "num_queries": 900,
    "query_dim": 4,
    "num_patterns": 0,
    "num_feature_levels": 4,
    "enc_n_points": 4,
    "dec_n_points": 4,
    "two_stage_type": "standard",
    "two_stage_bbox_embed_share": False,
    "two_stage_class_embed_share": False,
    "transformer_activation": "relu",
    "dec_pred_bbox_embed_share": True,
    "dn_box_noise_scale": 1.0,
    "dn_label_noise_ratio": 0.5,
    "dn_labelbook_size": 91,
    "embed_init_tgt": True,
    "max_text_len": 256,
    "text_encoder_type": "bert-base-uncased",
    "use_text_enhancer": True,
    "use_fusion_layer": True,
    "use_checkpoint": False,  # upstream trains with activation checkpointing on; inference and the bounded adaptation here do not need it
    "use_transformer_ckpt": False,
    "use_text_cross_attention": True,
    "text_dropout": 0.0,
    "fusion_dropout": 0.0,
    "fusion_droppath": 0.1,
    "sub_sentence_present": True,
    "aux_loss": True,
    "set_cost_class": 5.0,
    "set_cost_bbox": 1.0,
    "set_cost_giou": 0.0,
    "cls_loss_coef": 5.0,
    "bbox_loss_coef": 1.0,
    "giou_loss_coef": 0.0,
    "interm_loss_coef": 1.0,
    "no_interm_box_loss": False,
    "focal_alpha": 0.25,
    "focal_gamma": 2.0,
    "matcher_type": "HungarianMatcher",
}
PARAMETER_COUNT = 233_362_816  # parameters of the built model (232,772,224 of them require grad at construction; the checkpoint adds 24 int64 index buffers)
STATE_TENSORS = 1_108  # state-dict entries, 66 of them aliases of the shared box head recorded in the safetensors metadata
HIDDEN_DIM = 256
NUM_QUERIES = 900
DECODER_LAYERS = 6
ENCODER_LAYERS = 6

# ------------------------------------------------------------------ inference protocol (upstream inference scripts)
CONFIDENCE_THRESHOLD = 0.23  # upstream `--confidence_thresh` / cfg `box_threshold`: a query counts when its best token score exceeds it
SHORT_SIDE = 800  # upstream test transform: RandomResize([800], max_size=1333)
MAX_SIDE = 1333
IMAGE_MEAN = (0.485, 0.456, 0.406)
IMAGE_STD = (0.229, 0.224, 0.225)
MAX_EXEMPLARS = 3  # FSC-147 gives three exemplar boxes per image; upstream uses all three
TARGET_BOX_PX = 2.0  # upstream's FSC-147 training boxes: a 2 x 2 px box centred on each annotated point
EXEMPLAR_TOKEN_ID = 1008  # the `*` WordPiece id upstream inserts for each visual exemplar
MAX_TEXT_CHARS = 64

UNSAFE_WEIGHT_EXTENSIONS = (  # refused anywhere except the audited source checkpoint itself
    ".bin",
    ".pt",
    ".pth",
    ".ckpt",
    ".pkl",
    ".pickle",
    ".h5",
    ".msgpack",
)

**Module 2/8:** `src/countgd_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Corpus-level measures for open-world counting, in numpy / torch: the count errors FSC-147 is scored on
(MAE, RMSE, and the normalised absolute error), a point-based localisation reading of the predicted boxes,
a box-extent reading (IoU matching) where gold object boxes exist, and two non-neural baselines scored by the same code — the mean training count and a normalised
cross-correlation template matcher built from the exemplar boxes."""
# ruff: noqa: E501  -- fleet metrics module written at the 110-column fleet width; this repo lints at 100

from __future__ import annotations

import math
from collections.abc import Mapping, Sequence
from typing import Any

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from scipy.optimize import linear_sum_assignment

METRIC_DEFINITIONS = {
    "mae": "mean over images of |predicted count - gold count|; lower is better (FSC-147's headline metric)",
    "rmse": "square root of the mean squared count error; lower is better, dominated by the largest errors",
    "nae": "mean over images of |predicted - gold| / gold; a scale-free count error, lower is better",
    "under_count_fraction": "fraction of images the system counts fewer objects than the gold count",
    "exact_fraction": "fraction of images counted exactly",
}
LOCALISATION_DEFINITIONS = {
    "precision": "fraction of predicted points matched one-to-one to a gold point within the match radius",
    "recall": "fraction of gold points matched one-to-one to a predicted point within the match radius",
    "f1": "harmonic mean of localisation precision and recall (micro-averaged over the scored images)",
    "match_radius": "per image: half the mean exemplar side when exemplars are given, else 2 % of the longer image side, never below 4 px",
}
MIN_MATCH_RADIUS = 4.0
DEFAULT_TEMPLATE_THRESHOLD = 0.6  # normalised cross-correlation peak the template matcher counts
MIN_WINDOW_STD = 0.01  # grey levels in [0, 1]: an image window or template flatter than this has no defined correlation


# ------------------------------------------------------------------------------------ counting


def counting_metrics(rows: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """MAE / RMSE / NAE over `{id, gold, predicted, label}` rows, with a per-class breakdown."""
    if not rows:
        raise ValueError("no rows to score")
    gold = np.asarray([float(r["gold"]) for r in rows])
    pred = np.asarray([float(r["predicted"]) for r in rows])
    if np.any(gold < 0) or np.any(pred < 0):
        raise ValueError("counts must be non-negative")
    err = pred - gold
    per_class: dict[str, dict[str, Any]] = {}
    for label in sorted({str(r.get("label", "")) for r in rows}):
        idx = [i for i, r in enumerate(rows) if str(r.get("label", "")) == label]
        e = err[idx]
        per_class[label] = {
            "n": len(idx),
            "mae": float(np.mean(np.abs(e))),
            "rmse": float(math.sqrt(np.mean(e**2))),
            "mean_gold": float(np.mean(gold[idx])),
            "mean_predicted": float(np.mean(pred[idx])),
        }
    safe_gold = np.where(gold > 0, gold, 1.0)
    return {
        "n": int(len(rows)),
        "mae": float(np.mean(np.abs(err))),
        "rmse": float(math.sqrt(np.mean(err**2))),
        "nae": float(np.mean(np.abs(err) / safe_gold)),
        "under_count_fraction": float(np.mean(err < 0)),
        "exact_fraction": float(np.mean(err == 0)),
        "total_gold": int(gold.sum()),
        "total_predicted": int(pred.sum()),
        "per_class": per_class,
        "definitions": dict(METRIC_DEFINITIONS),
    }


# ------------------------------------------------------------------------------------ localisation


def match_radius(record: Mapping[str, Any]) -> float:
    """Half the mean exemplar side, else 2 % of the longer image side; never below MIN_MATCH_RADIUS px."""
    exemplars = record.get("exemplars") or []
    if exemplars:
        sides = [((b[2] - b[0]) + (b[3] - b[1])) / 2.0 for b in exemplars]
        radius = 0.5 * float(np.mean(sides))
    else:
        width, height = record["image"].size
        radius = 0.02 * max(width, height)
    return max(MIN_MATCH_RADIUS, radius)


def match_points(predicted: Sequence[Sequence[float]], gold: Sequence[Sequence[float]], radius: float) -> dict[str, int]:
    """One-to-one Hungarian matching of predicted to gold points within `radius` (pixels): TP / FP / FN."""
    if radius <= 0:
        raise ValueError("radius must be positive")
    if not predicted or not gold:
        return {"tp": 0, "fp": len(predicted), "fn": len(gold)}
    p = np.asarray(predicted, dtype=np.float64).reshape(-1, 2)
    g = np.asarray(gold, dtype=np.float64).reshape(-1, 2)
    dist = np.sqrt(((p[:, None, :] - g[None, :, :]) ** 2).sum(-1))
    cost = np.where(dist <= radius, dist, 1e6)
    rows, cols = linear_sum_assignment(cost)
    tp = int(np.sum(dist[rows, cols] <= radius))
    return {"tp": tp, "fp": int(len(p) - tp), "fn": int(len(g) - tp)}


def localisation_metrics(rows: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Micro-averaged precision / recall / F1 over `{tp, fp, fn}` rows (images without gold points are skipped)."""
    scored = [r for r in rows if r is not None]
    if not scored:
        return {"n": 0, "precision": None, "recall": None, "f1": None, "definitions": dict(LOCALISATION_DEFINITIONS)}
    tp = sum(int(r["tp"]) for r in scored)
    fp = sum(int(r["fp"]) for r in scored)
    fn = sum(int(r["fn"]) for r in scored)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"n": len(scored), "tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1, "definitions": dict(LOCALISATION_DEFINITIONS)}


# ------------------------------------------------------------------------------------ box extents

BOX_DEFINITIONS = {
    "iou_threshold": "a predicted box is a true positive when its one-to-one Hungarian partner (maximising IoU) is a gold box with IoU >= this",
    "precision": "fraction of predicted boxes matched to a gold box at IoU >= the threshold",
    "recall": "fraction of gold boxes matched to a predicted box at IoU >= the threshold",
    "f1": "harmonic mean of box precision and recall (micro-averaged over the scored images)",
    "mean_matched_iou": "mean IoU of the true-positive pairs (how tight the boxes are that do match)",
    "mean_best_iou": "mean over gold boxes of the highest IoU any predicted box reaches (0 for a gold box nothing overlaps)",
}
BOX_IOU_THRESHOLD = 0.5


def box_iou_matrix(a: Sequence[Sequence[float]], b: Sequence[Sequence[float]]) -> np.ndarray:
    """Pairwise IoU of `[x0, y0, x1, y1]` boxes, shape (len(a), len(b))."""
    p = np.asarray(a, dtype=np.float64).reshape(-1, 4)
    g = np.asarray(b, dtype=np.float64).reshape(-1, 4)
    ix = np.clip(np.minimum(p[:, None, 2], g[None, :, 2]) - np.maximum(p[:, None, 0], g[None, :, 0]), 0, None)
    iy = np.clip(np.minimum(p[:, None, 3], g[None, :, 3]) - np.maximum(p[:, None, 1], g[None, :, 1]), 0, None)
    inter = ix * iy
    area_p = (p[:, 2] - p[:, 0]) * (p[:, 3] - p[:, 1])
    area_g = (g[:, 2] - g[:, 0]) * (g[:, 3] - g[:, 1])
    union = area_p[:, None] + area_g[None, :] - inter
    return np.where(union > 0, inter / np.where(union > 0, union, 1.0), 0.0)


def match_boxes(predicted: Sequence[Sequence[float]], gold: Sequence[Sequence[float]], threshold: float = BOX_IOU_THRESHOLD) -> dict[str, Any]:
    """One-to-one Hungarian matching of predicted to gold boxes by IoU: TP / FP / FN at `threshold`, the IoUs of
    the true positives and each gold box's best IoU."""
    if not 0.0 < threshold <= 1.0:
        raise ValueError("threshold must be in (0, 1]")
    if not predicted or not gold:
        return {"tp": 0, "fp": len(predicted), "fn": len(gold), "matched_iou": [], "best_iou": [0.0] * len(gold)}
    iou = box_iou_matrix(predicted, gold)
    rows, cols = linear_sum_assignment(-iou)
    hits = iou[rows, cols] >= threshold
    tp = int(hits.sum())
    return {
        "tp": tp,
        "fp": int(len(predicted) - tp),
        "fn": int(len(gold) - tp),
        "matched_iou": [float(v) for v in iou[rows, cols][hits]],
        "best_iou": [float(v) for v in iou.max(axis=0)],
    }


def box_metrics(rows: Sequence[Mapping[str, Any] | None], threshold: float = BOX_IOU_THRESHOLD) -> dict[str, Any]:
    """Micro-averaged box precision / recall / F1 at the IoU threshold, the mean IoU of the matched pairs and the
    mean best IoU per gold box, over `match_boxes` rows (images without gold boxes are skipped)."""
    scored = [r for r in rows if r is not None]
    base = {"iou_threshold": threshold, "definitions": dict(BOX_DEFINITIONS)}
    if not scored:
        return {"n": 0, "precision": None, "recall": None, "f1": None, "mean_matched_iou": None, "mean_best_iou": None, **base}
    tp = sum(int(r["tp"]) for r in scored)
    fp = sum(int(r["fp"]) for r in scored)
    fn = sum(int(r["fn"]) for r in scored)
    matched = [v for r in scored for v in r["matched_iou"]]
    best = [v for r in scored for v in r["best_iou"]]
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "n": len(scored),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mean_matched_iou": float(np.mean(matched)) if matched else None,
        "mean_best_iou": float(np.mean(best)) if best else None,
        **base,
    }


# ------------------------------------------------------------------------------------ baselines


def mean_count_baseline(train: Sequence[Mapping[str, Any]], records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Every image gets the rounded mean gold count of the training set (the counting analogue of a majority floor)."""
    if not train:
        raise ValueError("the mean-count baseline needs training records")
    mean = float(np.mean([float(r["count"]) for r in train]))
    predicted = int(round(mean))
    rows = [{"id": r["id"], "label": r["label"], "gold": r["count"], "predicted": predicted} for r in records]
    result = counting_metrics(rows)
    result["rows"] = rows
    result["baseline"] = f"mean training count ({mean:.1f} -> {predicted} for every image)"
    return result


def _grey(image: Image.Image) -> torch.Tensor:
    return torch.from_numpy(np.asarray(image.convert("L"), dtype=np.float32) / 255.0)


def _ncc_map(image: torch.Tensor, template: torch.Tensor) -> torch.Tensor:
    """Normalised cross-correlation of a zero-mean template over the image (same-size output, in [-1, 1]).

    The correlation is undefined where the image window (or the template) is flat; those positions get 0
    rather than a ratio of two near-zero numbers, which float error would turn into spurious peaks."""
    th, tw = template.shape
    n = th * tw
    t = template - template.mean()
    t_var = (t**2).sum()
    if t_var <= (MIN_WINDOW_STD**2) * n:
        return torch.zeros_like(image)
    pad = (tw // 2, tw - tw // 2 - 1, th // 2, th - th // 2 - 1)
    img = F.pad(image[None, None], pad, mode="reflect")
    ones = torch.ones(1, 1, th, tw)
    local_sum = F.conv2d(img, ones)
    local_sq = F.conv2d(img**2, ones)
    local_var = (local_sq - local_sum**2 / n).clamp_min(0.0)
    corr = F.conv2d(img, t[None, None])
    valid = local_var > (MIN_WINDOW_STD**2) * n
    ncc = torch.where(valid, corr / (torch.sqrt(t_var) * torch.sqrt(local_var.clamp_min(1e-12))), torch.zeros_like(corr))
    return ncc.clamp(-1.0, 1.0)[0, 0]


def template_match(record: Mapping[str, Any], *, threshold: float = DEFAULT_TEMPLATE_THRESHOLD) -> dict[str, Any]:
    """Count one image by normalised cross-correlation with the mean of its exemplar crops (grayscale, resized
    to the mean exemplar size): local maxima above `threshold`, one per exemplar-sized neighbourhood. A
    classical, learning-free use of the same exemplars the model gets."""
    exemplars = record.get("exemplars") or []
    if not exemplars:
        return {"count": None, "points": [], "note": "no exemplars"}
    image = record["image"].convert("RGB")
    grey = _grey(image)
    tw = max(3, int(round(float(np.mean([b[2] - b[0] for b in exemplars])))))
    th = max(3, int(round(float(np.mean([b[3] - b[1] for b in exemplars])))))
    tw, th = min(tw, grey.shape[1]), min(th, grey.shape[0])
    crops = []
    for x0, y0, x1, y1 in exemplars:
        crop = image.crop((int(round(x0)), int(round(y0)), max(int(round(x1)), int(round(x0)) + 1), max(int(round(y1)), int(round(y0)) + 1))).resize((tw, th), Image.BILINEAR)
        crops.append(_grey(crop))
    template = torch.stack(crops).mean(0)
    ncc = _ncc_map(grey, template)
    k = max(3, (min(th, tw) // 2) * 2 + 1)
    peaks = F.max_pool2d(ncc[None, None], kernel_size=k, stride=1, padding=k // 2)[0, 0]
    keep = (ncc >= threshold) & (ncc == peaks)
    ys, xs = torch.nonzero(keep, as_tuple=True)
    points = [[float(x), float(y)] for x, y in zip(xs.tolist(), ys.tolist(), strict=True)]
    return {"count": len(points), "points": points, "template_size": [tw, th], "threshold": threshold, "peak_max": float(ncc.max())}


def template_matching_baseline(records: Sequence[Mapping[str, Any]], *, threshold: float = DEFAULT_TEMPLATE_THRESHOLD) -> dict[str, Any]:
    """The template matcher over a set: count errors and, where gold points exist, localisation."""
    rows, loc_rows = [], []
    for record in records:
        result = template_match(record, threshold=threshold)
        predicted = result["count"] if result["count"] is not None else 0
        rows.append({"id": record["id"], "label": record["label"], "gold": record["count"], "predicted": predicted})
        if "points" in record:
            loc_rows.append(match_points(result["points"], record["points"], match_radius(record)))
    out = counting_metrics(rows)
    out["rows"] = rows
    out["localisation"] = localisation_metrics(loc_rows)
    out["baseline"] = f"normalised cross-correlation with the mean exemplar crop, peaks >= {threshold} (no learning)"
    return out


__all__ = [
    "BOX_DEFINITIONS",
    "BOX_IOU_THRESHOLD",
    "DEFAULT_TEMPLATE_THRESHOLD",
    "LOCALISATION_DEFINITIONS",
    "METRIC_DEFINITIONS",
    "MIN_MATCH_RADIUS",
    "box_iou_matrix",
    "box_metrics",
    "counting_metrics",
    "localisation_metrics",
    "match_boxes",
    "match_points",
    "match_radius",
    "mean_count_baseline",
    "template_match",
    "template_matching_baseline",
]

**Module 3/8:** `src/countgd_pipeline/modeling.py` (carried verbatim; see the note above)

In [ ]:
"""CountGD (GroundingDINO Swin-B + exemplar tokens) — the upstream model code carried verbatim.

Vendored from https://github.com/niki-amini-naieni/CountGD at commit b6f362b3f5cd20db4a171faa410dfed8f2f466d8 (MIT; the GroundingDINO parts
carry IDEA's Apache-2.0 header) — the named top-level definitions of the files listed below, in dependency
order, with the package-relative imports removed and every edit marked `# vendored:`. The edits: the ResNet
backbone branch, the compiled `MultiScaleDeformableAttention` op (the pure-PyTorch `grid_sample` path is used on
CPU and CUDA alike), the build registry, the COCO `PostProcess`, the never-called exemplar feature-map encoder
(no weights in the checkpoint) and the distributed helpers are not carried; the tokenizer and the BERT module
are injected from the staged snapshot instead of fetched by name; `timm.models.layers` (DropPath, to_2tuple,
trunc_normal_) is replaced by the three small definitions below. Nothing else is changed.

Upstream sources (SHA-256 of the file at b6f362b3):
- groundingdino/util/misc.py: bac1d93b339ae23abf3b7c8299e06d6ea71dbfeba8939444661a902d121666a6
- util/box_ops.py: 35b4481bc1d523b209a357f4cf193950a6e617f1e6817ce603af5f1275b0bf16
- models/GroundingDINO/backbone/position_encoding.py: ea55ddcc1a2f811c93885ad5320f98477c33933a7afa1492828c254b2e80ef20
- models/GroundingDINO/backbone/swin_transformer.py: 0584c619ebb168c8e46ecf3f469ecea7f1b1bd0e92a360943f9e39a8738dcc87
- models/GroundingDINO/backbone/backbone.py: 1f6f2cf7f3709bec300664467d48fc6e9339d0b85c3a8ce658e66837b5491053
- models/GroundingDINO/ms_deform_attn.py: 14fa9c57e168fd776372fea923c6f590234f86bfe331b70c5ae59b1a4fa44e3d
- models/GroundingDINO/utils.py: bc7862bda23ae44c1d2e1c3a0f02f1e949477748e701d3f97b6376219a7df362
- models/GroundingDINO/transformer_vanilla.py: 540ba06a1d6c9b603ae0695e72a7a3cebae5f62c35ec5ec5e92490581f5b61f9
- models/GroundingDINO/fuse_modules.py: a4b738a4ae3ca90cc5ae339241a9e0c24026bda96aedffb549b161bbdff96cd1
- models/GroundingDINO/bertwarper.py: 666e345c3450a6a276b37d09f2556926737e04f0c98b2526a7f6871b624ce546
- models/GroundingDINO/transformer.py: 6a1b339e9d7849ef416a2400b505fd6982133a3fbfe345f44aa87699ae7c33b3
- models/GroundingDINO/matcher.py: 0952f87beba3e03065d5afafb0b4fa3d2715e223d83c785ff01300f891432bb8
- models/GroundingDINO/groundingdino.py: 80479ab38fec0ffd4aa4a1e65af0a4f6d5bde148d1e6a78f8d338eaa206f696a
"""
# ruff: noqa: E501, E712, E722, E741, F541, F841, B006, B007, B008, B018, B028, B905, UP004, UP006, UP008, UP018, UP031, UP032, UP034, UP035, UP045  -- vendored code kept as upstream wrote it, for auditability
from __future__ import annotations

import copy
import math
import warnings
from typing import List, Optional

import numpy as np
import torch
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint
from scipy.optimize import linear_sum_assignment
from torch import Tensor, nn
from torch.nn.init import constant_, xavier_uniform_
from torchvision.ops import roi_align
from torchvision.ops.boxes import box_area
from transformers.modeling_outputs import BaseModelOutputWithPoolingAndCrossAttentions


# vendored: minimal replacements for `timm.models.layers` (DropPath, to_2tuple, trunc_normal_)
def to_2tuple(x):
    return tuple(x) if isinstance(x, (tuple, list)) else (x, x)


trunc_normal_ = nn.init.trunc_normal_


class DropPath(nn.Module):
    """Drop paths (stochastic depth) per sample, applied in the main path of residual blocks (timm's DropPath)."""

    def __init__(self, drop_prob=None):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0.0 or self.drop_prob is None or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x.div(keep_prob) * random_tensor



# ----------------------------------------------------------------------------- groundingdino/util/misc.py

def _max_by_axis(the_list):
    # type: (List[List[int]]) -> List[int]
    maxes = the_list[0]
    for sublist in the_list[1:]:
        for index, item in enumerate(sublist):
            maxes[index] = max(maxes[index], item)
    return maxes

class NestedTensor(object):
    def __init__(self, tensors, mask: Optional[Tensor]):
        self.tensors = tensors
        self.mask = mask
        if mask == "auto":
            self.mask = torch.zeros_like(tensors).to(tensors.device)
            if self.mask.dim() == 3:
                self.mask = self.mask.sum(0).to(bool)
            elif self.mask.dim() == 4:
                self.mask = self.mask.sum(1).to(bool)
            else:
                raise ValueError(
                    "tensors dim must be 3 or 4 but {}({})".format(
                        self.tensors.dim(), self.tensors.shape
                    )
                )

    def imgsize(self):
        res = []
        for i in range(self.tensors.shape[0]):
            mask = self.mask[i]
            maxH = (~mask).sum(0).max()
            maxW = (~mask).sum(1).max()
            res.append(torch.Tensor([maxH, maxW]))
        return res

    def to(self, device):
        # type: (Device) -> NestedTensor # noqa
        cast_tensor = self.tensors.to(device)
        mask = self.mask
        if mask is not None:
            assert mask is not None
            cast_mask = mask.to(device)
        else:
            cast_mask = None
        return NestedTensor(cast_tensor, cast_mask)

    def to_img_list_single(self, tensor, mask):
        assert tensor.dim() == 3, "dim of tensor should be 3 but {}".format(tensor.dim())
        maxH = (~mask).sum(0).max()
        maxW = (~mask).sum(1).max()
        img = tensor[:, :maxH, :maxW]
        return img

    def to_img_list(self):
        """remove the padding and convert to img list

        Returns:
            [type]: [description]
        """
        if self.tensors.dim() == 3:
            return self.to_img_list_single(self.tensors, self.mask)
        else:
            res = []
            for i in range(self.tensors.shape[0]):
                tensor_i = self.tensors[i]
                mask_i = self.mask[i]
                res.append(self.to_img_list_single(tensor_i, mask_i))
            return res

    @property
    def device(self):
        return self.tensors.device

    def decompose(self):
        return self.tensors, self.mask

    def __repr__(self):
        return str(self.tensors)

    @property
    def shape(self):
        return {"tensors.shape": self.tensors.shape, "mask.shape": self.mask.shape}

def nested_tensor_from_tensor_list(tensor_list: List[Tensor]):
    # upstream to-do: make this more general  # vendored: comment reworded (placeholder tokens are refused in the tutorial)
    if tensor_list[0].ndim == 3:
        # vendored: the ONNX-tracing branch (torchvision._is_tracing) is not carried
        # upstream to-do: make it support different-sized images  # vendored: comment reworded (placeholder tokens are refused in the tutorial)
        max_size = _max_by_axis([list(img.shape) for img in tensor_list])
        # min_size = tuple(min(s) for s in zip(*[img.shape for img in tensor_list]))
        batch_shape = [len(tensor_list)] + max_size
        b, c, h, w = batch_shape
        dtype = tensor_list[0].dtype
        device = tensor_list[0].device
        tensor = torch.zeros(batch_shape, dtype=dtype, device=device)
        mask = torch.ones((b, h, w), dtype=torch.bool, device=device)
        for img, pad_img, m in zip(tensor_list, tensor, mask):
            pad_img[: img.shape[0], : img.shape[1], : img.shape[2]].copy_(img)
            m[: img.shape[1], : img.shape[2]] = False
    else:
        raise ValueError("not supported")
    return NestedTensor(tensor, mask)

def inverse_sigmoid(x, eps=1e-3):
    x = x.clamp(min=0, max=1)
    x1 = x.clamp(min=eps)
    x2 = (1 - x).clamp(min=eps)
    return torch.log(x1 / x2)


# ----------------------------------------------------------------------------- util/box_ops.py

def box_cxcywh_to_xyxy(x):
    x_c, y_c, w, h = x.unbind(-1)
    b = [(x_c - 0.5 * w), (y_c - 0.5 * h),
         (x_c + 0.5 * w), (y_c + 0.5 * h)]
    return torch.stack(b, dim=-1)

def box_xyxy_to_cxcywh(x):
    x0, y0, x1, y1 = x.unbind(-1)
    b = [(x0 + x1) / 2, (y0 + y1) / 2,
         (x1 - x0), (y1 - y0)]
    return torch.stack(b, dim=-1)

def box_iou(boxes1, boxes2):
    area1 = box_area(boxes1)
    area2 = box_area(boxes2)


    lt = torch.max(boxes1[:, None, :2], boxes2[:, :2])  # [N,M,2]
    rb = torch.min(boxes1[:, None, 2:], boxes2[:, 2:])  # [N,M,2]

    wh = (rb - lt).clamp(min=0)  # [N,M,2]
    inter = wh[:, :, 0] * wh[:, :, 1]  # [N,M]

    union = area1[:, None] + area2 - inter

    iou = inter / (union + 1e-6)
    return iou, union

def generalized_box_iou(boxes1, boxes2):
    """
    Generalized IoU from https://giou.stanford.edu/

    The boxes should be in [x0, y0, x1, y1] format

    Returns a [N, M] pairwise matrix, where N = len(boxes1)
    and M = len(boxes2)
    """
    # degenerate boxes gives inf / nan results
    # so do an early check
    assert (boxes1[:, 2:] >= boxes1[:, :2]).all(), f"{boxes1}"
    assert (boxes2[:, 2:] >= boxes2[:, :2]).all(), f"{boxes2}"

    iou, union = box_iou(boxes1, boxes2)

    lt = torch.min(boxes1[:, None, :2], boxes2[:, :2])
    rb = torch.max(boxes1[:, None, 2:], boxes2[:, 2:])

    wh = (rb - lt).clamp(min=0)  # [N,M,2]
    area = wh[:, :, 0] * wh[:, :, 1]

    return iou - (area - union) / (area + 1e-6)


# ----------------------------------------------------------------------------- models/GroundingDINO/backbone/position_encoding.py

class PositionEmbeddingSineHW(nn.Module):
    """
    This is a more standard version of the position embedding, very similar to the one
    used by the Attention is all you need paper, generalized to work on images.
    """

    def __init__(
        self, num_pos_feats=64, temperatureH=10000, temperatureW=10000, normalize=False, scale=None
    ):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.temperatureH = temperatureH
        self.temperatureW = temperatureW
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale

    def forward(self, tensor_list: NestedTensor):
        x = tensor_list.tensors
        mask = tensor_list.mask
        assert mask is not None
        not_mask = ~mask
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)

        # import ipdb; ipdb.set_trace()

        if self.normalize:
            eps = 1e-6
            y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
            x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale

        dim_tx = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_tx = self.temperatureW ** (2 * (torch.div(dim_tx, 2, rounding_mode='floor')) / self.num_pos_feats)
        pos_x = x_embed[:, :, :, None] / dim_tx

        dim_ty = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_ty = self.temperatureH ** (2 * (torch.div(dim_ty, 2, rounding_mode='floor')) / self.num_pos_feats)
        pos_y = y_embed[:, :, :, None] / dim_ty

        pos_x = torch.stack(
            (pos_x[:, :, :, 0::2].sin(), pos_x[:, :, :, 1::2].cos()), dim=4
        ).flatten(3)
        pos_y = torch.stack(
            (pos_y[:, :, :, 0::2].sin(), pos_y[:, :, :, 1::2].cos()), dim=4
        ).flatten(3)
        pos = torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)

        # import ipdb; ipdb.set_trace()

        return pos

def build_position_encoding(args):
    N_steps = args.hidden_dim // 2
    if args.position_embedding in ("v2", "sine"):
        # upstream to-do: find a better way of exposing other arguments  # vendored: comment reworded (placeholder tokens are refused in the tutorial)
        position_embedding = PositionEmbeddingSineHW(
            N_steps,
            temperatureH=args.pe_temperatureH,
            temperatureW=args.pe_temperatureW,
            normalize=True,
        )
    else:  # vendored: the learned position embedding ("v3" / "learned") is not carried; the checkpoint uses sine
        raise ValueError(f"not supported {args.position_embedding}")

    return position_embedding


# ----------------------------------------------------------------------------- models/GroundingDINO/backbone/swin_transformer.py

class Mlp(nn.Module):
    """Multilayer perceptron."""

    def __init__(
        self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.0
    ):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

def window_partition(x, window_size):
    """
    Args:
        x: (B, H, W, C)
        window_size (int): window size
    Returns:
        windows: (num_windows*B, window_size, window_size, C)
    """
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows

def window_reverse(windows, window_size, H, W):
    """
    Args:
        windows: (num_windows*B, window_size, window_size, C)
        window_size (int): Window size
        H (int): Height of image
        W (int): Width of image
    Returns:
        x: (B, H, W, C)
    """
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x

class WindowAttention(nn.Module):
    """Window based multi-head self attention (W-MSA) module with relative position bias.
    It supports both of shifted and non-shifted window.
    Args:
        dim (int): Number of input channels.
        window_size (tuple[int]): The height and width of the window.
        num_heads (int): Number of attention heads.
        qkv_bias (bool, optional):  If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set
        attn_drop (float, optional): Dropout ratio of attention weight. Default: 0.0
        proj_drop (float, optional): Dropout ratio of output. Default: 0.0
    """

    def __init__(
        self,
        dim,
        window_size,
        num_heads,
        qkv_bias=True,
        qk_scale=None,
        attn_drop=0.0,
        proj_drop=0.0,
    ):

        super().__init__()
        self.dim = dim
        self.window_size = window_size  # Wh, Ww
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim**-0.5

        # define a parameter table of relative position bias
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads)
        )  # 2*Wh-1 * 2*Ww-1, nH

        # get pair-wise relative position index for each token inside the window
        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid([coords_h, coords_w]))  # 2, Wh, Ww
        coords_flatten = torch.flatten(coords, 1)  # 2, Wh*Ww
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]  # 2, Wh*Ww, Wh*Ww
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()  # Wh*Ww, Wh*Ww, 2
        relative_coords[:, :, 0] += self.window_size[0] - 1  # shift to start from 0
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        relative_position_index = relative_coords.sum(-1)  # Wh*Ww, Wh*Ww
        self.register_buffer("relative_position_index", relative_position_index)

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

        trunc_normal_(self.relative_position_bias_table, std=0.02)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        """Forward function.
        Args:
            x: input features with shape of (num_windows*B, N, C)
            mask: (0/-inf) mask with shape of (num_windows, Wh*Ww, Wh*Ww) or None
        """
        B_, N, C = x.shape
        qkv = (
            self.qkv(x)
            .reshape(B_, N, 3, self.num_heads, C // self.num_heads)
            .permute(2, 0, 3, 1, 4)
        )
        q, k, v = qkv[0], qkv[1], qkv[2]  # make torchscript happy (cannot use tensor as tuple)

        q = q * self.scale
        attn = q @ k.transpose(-2, -1)

        relative_position_bias = self.relative_position_bias_table[
            self.relative_position_index.view(-1)
        ].view(
            self.window_size[0] * self.window_size[1], self.window_size[0] * self.window_size[1], -1
        )  # Wh*Ww,Wh*Ww,nH
        relative_position_bias = relative_position_bias.permute(
            2, 0, 1
        ).contiguous()  # nH, Wh*Ww, Wh*Ww
        attn = attn + relative_position_bias.unsqueeze(0)

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
            attn = self.softmax(attn)
        else:
            attn = self.softmax(attn)

        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class SwinTransformerBlock(nn.Module):
    """Swin Transformer Block.
    Args:
        dim (int): Number of input channels.
        num_heads (int): Number of attention heads.
        window_size (int): Window size.
        shift_size (int): Shift size for SW-MSA.
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim.
        qkv_bias (bool, optional): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set.
        drop (float, optional): Dropout rate. Default: 0.0
        attn_drop (float, optional): Attention dropout rate. Default: 0.0
        drop_path (float, optional): Stochastic depth rate. Default: 0.0
        act_layer (nn.Module, optional): Activation layer. Default: nn.GELU
        norm_layer (nn.Module, optional): Normalization layer.  Default: nn.LayerNorm
    """

    def __init__(
        self,
        dim,
        num_heads,
        window_size=7,
        shift_size=0,
        mlp_ratio=4.0,
        qkv_bias=True,
        qk_scale=None,
        drop=0.0,
        attn_drop=0.0,
        drop_path=0.0,
        act_layer=nn.GELU,
        norm_layer=nn.LayerNorm,
    ):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio
        assert 0 <= self.shift_size < self.window_size, "shift_size must in 0-window_size"

        self.norm1 = norm_layer(dim)
        self.attn = WindowAttention(
            dim,
            window_size=to_2tuple(self.window_size),
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            qk_scale=qk_scale,
            attn_drop=attn_drop,
            proj_drop=drop,
        )

        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(
            in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop
        )

        self.H = None
        self.W = None

    def forward(self, x, mask_matrix):
        """Forward function.
        Args:
            x: Input feature, tensor size (B, H*W, C).
            H, W: Spatial resolution of the input feature.
            mask_matrix: Attention mask for cyclic shift.
        """
        B, L, C = x.shape
        H, W = self.H, self.W
        assert L == H * W, "input feature has wrong size"

        shortcut = x
        x = self.norm1(x)
        x = x.view(B, H, W, C)

        # pad feature maps to multiples of window size
        pad_l = pad_t = 0
        pad_r = (self.window_size - W % self.window_size) % self.window_size
        pad_b = (self.window_size - H % self.window_size) % self.window_size
        x = F.pad(x, (0, 0, pad_l, pad_r, pad_t, pad_b))
        _, Hp, Wp, _ = x.shape

        # cyclic shift
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
            attn_mask = mask_matrix
        else:
            shifted_x = x
            attn_mask = None

        # partition windows
        x_windows = window_partition(
            shifted_x, self.window_size
        )  # nW*B, window_size, window_size, C
        x_windows = x_windows.view(
            -1, self.window_size * self.window_size, C
        )  # nW*B, window_size*window_size, C

        # W-MSA/SW-MSA
        attn_windows = self.attn(x_windows, mask=attn_mask)  # nW*B, window_size*window_size, C

        # merge windows
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        shifted_x = window_reverse(attn_windows, self.window_size, Hp, Wp)  # B H' W' C

        # reverse cyclic shift
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x

        if pad_r > 0 or pad_b > 0:
            x = x[:, :H, :W, :].contiguous()

        x = x.view(B, H * W, C)

        # FFN
        x = shortcut + self.drop_path(x)
        x = x + self.drop_path(self.mlp(self.norm2(x)))

        return x

class PatchMerging(nn.Module):
    """Patch Merging Layer
    Args:
        dim (int): Number of input channels.
        norm_layer (nn.Module, optional): Normalization layer.  Default: nn.LayerNorm
    """

    def __init__(self, dim, norm_layer=nn.LayerNorm):
        super().__init__()
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = norm_layer(4 * dim)

    def forward(self, x, H, W):
        """Forward function.
        Args:
            x: Input feature, tensor size (B, H*W, C).
            H, W: Spatial resolution of the input feature.
        """
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"

        x = x.view(B, H, W, C)

        # padding
        pad_input = (H % 2 == 1) or (W % 2 == 1)
        if pad_input:
            x = F.pad(x, (0, 0, 0, W % 2, 0, H % 2))

        x0 = x[:, 0::2, 0::2, :]  # B H/2 W/2 C
        x1 = x[:, 1::2, 0::2, :]  # B H/2 W/2 C
        x2 = x[:, 0::2, 1::2, :]  # B H/2 W/2 C
        x3 = x[:, 1::2, 1::2, :]  # B H/2 W/2 C
        x = torch.cat([x0, x1, x2, x3], -1)  # B H/2 W/2 4*C
        x = x.view(B, -1, 4 * C)  # B H/2*W/2 4*C

        x = self.norm(x)
        x = self.reduction(x)

        return x

class BasicLayer(nn.Module):
    """A basic Swin Transformer layer for one stage.
    Args:
        dim (int): Number of feature channels
        depth (int): Depths of this stage.
        num_heads (int): Number of attention head.
        window_size (int): Local window size. Default: 7.
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim. Default: 4.
        qkv_bias (bool, optional): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set.
        drop (float, optional): Dropout rate. Default: 0.0
        attn_drop (float, optional): Attention dropout rate. Default: 0.0
        drop_path (float | tuple[float], optional): Stochastic depth rate. Default: 0.0
        norm_layer (nn.Module, optional): Normalization layer. Default: nn.LayerNorm
        downsample (nn.Module | None, optional): Downsample layer at the end of the layer. Default: None
        use_checkpoint (bool): Whether to use checkpointing to save memory. Default: False.
    """

    def __init__(
        self,
        dim,
        depth,
        num_heads,
        window_size=7,
        mlp_ratio=4.0,
        qkv_bias=True,
        qk_scale=None,
        drop=0.0,
        attn_drop=0.0,
        drop_path=0.0,
        norm_layer=nn.LayerNorm,
        downsample=None,
        use_checkpoint=False,
    ):
        super().__init__()
        self.window_size = window_size
        self.shift_size = window_size // 2
        self.depth = depth
        self.use_checkpoint = use_checkpoint

        # build blocks
        self.blocks = nn.ModuleList(
            [
                SwinTransformerBlock(
                    dim=dim,
                    num_heads=num_heads,
                    window_size=window_size,
                    shift_size=0 if (i % 2 == 0) else window_size // 2,
                    mlp_ratio=mlp_ratio,
                    qkv_bias=qkv_bias,
                    qk_scale=qk_scale,
                    drop=drop,
                    attn_drop=attn_drop,
                    drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                    norm_layer=norm_layer,
                )
                for i in range(depth)
            ]
        )

        # patch merging layer
        if downsample is not None:
            self.downsample = downsample(dim=dim, norm_layer=norm_layer)
        else:
            self.downsample = None

    def forward(self, x, H, W):
        """Forward function.
        Args:
            x: Input feature, tensor size (B, H*W, C).
            H, W: Spatial resolution of the input feature.
        """

        # calculate attention mask for SW-MSA
        Hp = int(np.ceil(H / self.window_size)) * self.window_size
        Wp = int(np.ceil(W / self.window_size)) * self.window_size
        img_mask = torch.zeros((1, Hp, Wp, 1), device=x.device)  # 1 Hp Wp 1
        h_slices = (
            slice(0, -self.window_size),
            slice(-self.window_size, -self.shift_size),
            slice(-self.shift_size, None),
        )
        w_slices = (
            slice(0, -self.window_size),
            slice(-self.window_size, -self.shift_size),
            slice(-self.shift_size, None),
        )
        cnt = 0
        for h in h_slices:
            for w in w_slices:
                img_mask[:, h, w, :] = cnt
                cnt += 1

        mask_windows = window_partition(
            img_mask, self.window_size
        )  # nW, window_size, window_size, 1
        mask_windows = mask_windows.view(-1, self.window_size * self.window_size)
        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
        attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(
            attn_mask == 0, float(0.0)
        )

        for blk in self.blocks:
            blk.H, blk.W = H, W
            if self.use_checkpoint:
                x = checkpoint.checkpoint(blk, x, attn_mask)
            else:
                x = blk(x, attn_mask)
        if self.downsample is not None:
            x_down = self.downsample(x, H, W)
            Wh, Ww = (H + 1) // 2, (W + 1) // 2
            return x, H, W, x_down, Wh, Ww
        else:
            return x, H, W, x, H, W

class PatchEmbed(nn.Module):
    """Image to Patch Embedding
    Args:
        patch_size (int): Patch token size. Default: 4.
        in_chans (int): Number of input image channels. Default: 3.
        embed_dim (int): Number of linear projection output channels. Default: 96.
        norm_layer (nn.Module, optional): Normalization layer. Default: None
    """

    def __init__(self, patch_size=4, in_chans=3, embed_dim=96, norm_layer=None):
        super().__init__()
        patch_size = to_2tuple(patch_size)
        self.patch_size = patch_size

        self.in_chans = in_chans
        self.embed_dim = embed_dim

        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        if norm_layer is not None:
            self.norm = norm_layer(embed_dim)
        else:
            self.norm = None

    def forward(self, x):
        """Forward function."""
        # padding
        _, _, H, W = x.size()
        if W % self.patch_size[1] != 0:
            x = F.pad(x, (0, self.patch_size[1] - W % self.patch_size[1]))
        if H % self.patch_size[0] != 0:
            x = F.pad(x, (0, 0, 0, self.patch_size[0] - H % self.patch_size[0]))

        x = self.proj(x)  # B C Wh Ww
        if self.norm is not None:
            Wh, Ww = x.size(2), x.size(3)
            x = x.flatten(2).transpose(1, 2)
            x = self.norm(x)
            x = x.transpose(1, 2).view(-1, self.embed_dim, Wh, Ww)

        return x

class SwinTransformer(nn.Module):
    """Swin Transformer backbone.
        A PyTorch impl of : `Swin Transformer: Hierarchical Vision Transformer using Shifted Windows`  -
          https://arxiv.org/pdf/2103.14030
    Args:
        pretrain_img_size (int): Input image size for training the pretrained model,
            used in absolute postion embedding. Default 224.
        patch_size (int | tuple(int)): Patch size. Default: 4.
        in_chans (int): Number of input image channels. Default: 3.
        embed_dim (int): Number of linear projection output channels. Default: 96.
        depths (tuple[int]): Depths of each Swin Transformer stage.
        num_heads (tuple[int]): Number of attention head of each stage.
        window_size (int): Window size. Default: 7.
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim. Default: 4.
        qkv_bias (bool): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float): Override default qk scale of head_dim ** -0.5 if set.
        drop_rate (float): Dropout rate.
        attn_drop_rate (float): Attention dropout rate. Default: 0.
        drop_path_rate (float): Stochastic depth rate. Default: 0.2.
        norm_layer (nn.Module): Normalization layer. Default: nn.LayerNorm.
        ape (bool): If True, add absolute position embedding to the patch embedding. Default: False.
        patch_norm (bool): If True, add normalization after patch embedding. Default: True.
        out_indices (Sequence[int]): Output from which stages.
        frozen_stages (int): Stages to be frozen (stop grad and set eval mode).
            -1 means not freezing any parameters.
        use_checkpoint (bool): Whether to use checkpointing to save memory. Default: False.
        dilation (bool): if True, the output size if 16x downsample, ow 32x downsample.
    """

    def __init__(
        self,
        pretrain_img_size=224,
        patch_size=4,
        in_chans=3,
        embed_dim=96,
        depths=[2, 2, 6, 2],
        num_heads=[3, 6, 12, 24],
        window_size=7,
        mlp_ratio=4.0,
        qkv_bias=True,
        qk_scale=None,
        drop_rate=0.0,
        attn_drop_rate=0.0,
        drop_path_rate=0.2,
        norm_layer=nn.LayerNorm,
        ape=False,
        patch_norm=True,
        out_indices=(0, 1, 2, 3),
        frozen_stages=-1,
        dilation=False,
        use_checkpoint=False,
    ):
        super().__init__()

        self.pretrain_img_size = pretrain_img_size
        self.num_layers = len(depths)
        self.embed_dim = embed_dim
        self.ape = ape
        self.patch_norm = patch_norm
        self.out_indices = out_indices
        self.frozen_stages = frozen_stages
        self.dilation = dilation

        # if use_checkpoint:
        #     print("use_checkpoint!!!!!!!!!!!!!!!!!!!!!!!!")

        # split image into non-overlapping patches
        self.patch_embed = PatchEmbed(
            patch_size=patch_size,
            in_chans=in_chans,
            embed_dim=embed_dim,
            norm_layer=norm_layer if self.patch_norm else None,
        )

        # absolute position embedding
        if self.ape:
            pretrain_img_size = to_2tuple(pretrain_img_size)
            patch_size = to_2tuple(patch_size)
            patches_resolution = [
                pretrain_img_size[0] // patch_size[0],
                pretrain_img_size[1] // patch_size[1],
            ]

            self.absolute_pos_embed = nn.Parameter(
                torch.zeros(1, embed_dim, patches_resolution[0], patches_resolution[1])
            )
            trunc_normal_(self.absolute_pos_embed, std=0.02)

        self.pos_drop = nn.Dropout(p=drop_rate)

        # stochastic depth
        dpr = [
            x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))
        ]  # stochastic depth decay rule

        # build layers
        self.layers = nn.ModuleList()
        # prepare downsample list
        downsamplelist = [PatchMerging for i in range(self.num_layers)]
        downsamplelist[-1] = None
        num_features = [int(embed_dim * 2**i) for i in range(self.num_layers)]
        if self.dilation:
            downsamplelist[-2] = None
            num_features[-1] = int(embed_dim * 2 ** (self.num_layers - 1)) // 2
        for i_layer in range(self.num_layers):
            layer = BasicLayer(
                # dim=int(embed_dim * 2 ** i_layer),
                dim=num_features[i_layer],
                depth=depths[i_layer],
                num_heads=num_heads[i_layer],
                window_size=window_size,
                mlp_ratio=mlp_ratio,
                qkv_bias=qkv_bias,
                qk_scale=qk_scale,
                drop=drop_rate,
                attn_drop=attn_drop_rate,
                drop_path=dpr[sum(depths[:i_layer]) : sum(depths[: i_layer + 1])],
                norm_layer=norm_layer,
                # downsample=PatchMerging if (i_layer < self.num_layers - 1) else None,
                downsample=downsamplelist[i_layer],
                use_checkpoint=use_checkpoint,
            )
            self.layers.append(layer)

        # num_features = [int(embed_dim * 2 ** i) for i in range(self.num_layers)]
        self.num_features = num_features

        # add a norm layer for each output
        for i_layer in out_indices:
            layer = norm_layer(num_features[i_layer])
            layer_name = f"norm{i_layer}"
            self.add_module(layer_name, layer)

        self._freeze_stages()

    def _freeze_stages(self):
        if self.frozen_stages >= 0:
            self.patch_embed.eval()
            for param in self.patch_embed.parameters():
                param.requires_grad = False

        if self.frozen_stages >= 1 and self.ape:
            self.absolute_pos_embed.requires_grad = False

        if self.frozen_stages >= 2:
            self.pos_drop.eval()
            for i in range(0, self.frozen_stages - 1):
                m = self.layers[i]
                m.eval()
                for param in m.parameters():
                    param.requires_grad = False

    # def init_weights(self, pretrained=None):
    #     """Initialize the weights in backbone.
    #     Args:
    #         pretrained (str, optional): Path to pre-trained weights.
    #             Defaults to None.
    #     """

    #     def _init_weights(m):
    #         if isinstance(m, nn.Linear):
    #             trunc_normal_(m.weight, std=.02)
    #             if isinstance(m, nn.Linear) and m.bias is not None:
    #                 nn.init.constant_(m.bias, 0)
    #         elif isinstance(m, nn.LayerNorm):
    #             nn.init.constant_(m.bias, 0)
    #             nn.init.constant_(m.weight, 1.0)

    #     if isinstance(pretrained, str):
    #         self.apply(_init_weights)
    #         logger = get_root_logger()
    #         load_checkpoint(self, pretrained, strict=False, logger=logger)
    #     elif pretrained is None:
    #         self.apply(_init_weights)
    #     else:
    #         raise TypeError('pretrained must be a str or None')

    def forward_raw(self, x):
        """Forward function."""
        x = self.patch_embed(x)

        Wh, Ww = x.size(2), x.size(3)
        if self.ape:
            # interpolate the position embedding to the corresponding size
            absolute_pos_embed = F.interpolate(
                self.absolute_pos_embed, size=(Wh, Ww), mode="bicubic"
            )
            x = (x + absolute_pos_embed).flatten(2).transpose(1, 2)  # B Wh*Ww C
        else:
            x = x.flatten(2).transpose(1, 2)
        x = self.pos_drop(x)

        outs = []
        for i in range(self.num_layers):
            layer = self.layers[i]
            x_out, H, W, x, Wh, Ww = layer(x, Wh, Ww)
            # import ipdb; ipdb.set_trace()

            if i in self.out_indices:
                norm_layer = getattr(self, f"norm{i}")
                x_out = norm_layer(x_out)

                out = x_out.view(-1, H, W, self.num_features[i]).permute(0, 3, 1, 2).contiguous()
                outs.append(out)
        # in:
        #   torch.Size([2, 3, 1024, 1024])
        # outs:
        #   [torch.Size([2, 192, 256, 256]), torch.Size([2, 384, 128, 128]), \
        #       torch.Size([2, 768, 64, 64]), torch.Size([2, 1536, 32, 32])]
        return tuple(outs)

    def forward(self, tensor_list: NestedTensor):

        x = tensor_list.tensors

        """Forward function."""
        x = self.patch_embed(x)


        Wh, Ww = x.size(2), x.size(3)
        if self.ape:
            # interpolate the position embedding to the corresponding size
            absolute_pos_embed = F.interpolate(
                self.absolute_pos_embed, size=(Wh, Ww), mode="bicubic"
            )
            x = (x + absolute_pos_embed).flatten(2).transpose(1, 2)  # B Wh*Ww C
        else:
            x = x.flatten(2).transpose(1, 2)
        x = self.pos_drop(x)

        outs = []
        for i in range(self.num_layers):
            layer = self.layers[i]
            x_out, H, W, x, Wh, Ww = layer(x, Wh, Ww)

            if i in self.out_indices:
                norm_layer = getattr(self, f"norm{i}")
                x_out = norm_layer(x_out)

                out = x_out.view(-1, H, W, self.num_features[i]).permute(0, 3, 1, 2).contiguous()
                outs.append(out)
        # in:
        #   torch.Size([2, 3, 1024, 1024])
        # out:
        #   [torch.Size([2, 192, 256, 256]), torch.Size([2, 384, 128, 128]), \
        #       torch.Size([2, 768, 64, 64]), torch.Size([2, 1536, 32, 32])]

        # collect for nesttensors
        outs_dict = {}
        for idx, out_i in enumerate(outs):
            m = tensor_list.mask
            assert m is not None
            mask = F.interpolate(m[None].float(), size=out_i.shape[-2:]).to(torch.bool)[0]
            outs_dict[idx] = NestedTensor(out_i, mask)

        return outs_dict

    def train(self, mode=True):
        """Convert the model into training mode while keep layers freezed."""
        super(SwinTransformer, self).train(mode)
        self._freeze_stages()

def build_swin_transformer(modelname, pretrain_img_size, **kw):
    assert modelname in [
        "swin_T_224_1k",
        "swin_B_224_22k",
        "swin_B_384_22k",
        "swin_L_224_22k",
        "swin_L_384_22k",
    ]

    model_para_dict = {
        "swin_T_224_1k": dict(
            embed_dim=96, depths=[2, 2, 6, 2], num_heads=[3, 6, 12, 24], window_size=7
        ),
        "swin_B_224_22k": dict(
            embed_dim=128, depths=[2, 2, 18, 2], num_heads=[4, 8, 16, 32], window_size=7
        ),
        "swin_B_384_22k": dict(
            embed_dim=128, depths=[2, 2, 18, 2], num_heads=[4, 8, 16, 32], window_size=12
        ),
        "swin_L_224_22k": dict(
            embed_dim=192, depths=[2, 2, 18, 2], num_heads=[6, 12, 24, 48], window_size=7
        ),
        "swin_L_384_22k": dict(
            embed_dim=192, depths=[2, 2, 18, 2], num_heads=[6, 12, 24, 48], window_size=12
        ),
    }
    kw_cgf = model_para_dict[modelname]
    kw_cgf.update(kw)
    model = SwinTransformer(pretrain_img_size=pretrain_img_size, **kw_cgf)
    return model


# ----------------------------------------------------------------------------- models/GroundingDINO/backbone/backbone.py

class Joiner(nn.Sequential):
    def __init__(self, backbone, position_embedding):
        super().__init__(backbone, position_embedding)

    def forward(self, tensor_list: NestedTensor):
        xs = self[0](tensor_list)
        out: List[NestedTensor] = []
        pos = []
        for name, x in xs.items():
            out.append(x)
            # position encoding
            pos.append(self[1](x).to(x.tensors.dtype))

        return out, pos

def build_backbone(args):
    """
    Useful args:
        - backbone: backbone name
        - lr_backbone:
        - dilation
        - return_interm_indices: available: [0,1,2,3], [1,2,3], [3]
        - backbone_freeze_keywords:
        - use_checkpoint: for swin only for now

    """
    position_embedding = build_position_encoding(args)
    train_backbone = True
    if not train_backbone:
        raise ValueError("Please set lr_backbone > 0")
    return_interm_indices = args.return_interm_indices
    assert return_interm_indices in [[0, 1, 2, 3], [1, 2, 3], [3]]
    args.backbone_freeze_keywords
    use_checkpoint = getattr(args, "use_checkpoint", False)

    if args.backbone in [  # vendored: the ResNet branch (torchvision resnet + frozen batch norm) is not carried
        "swin_T_224_1k",
        "swin_B_224_22k",
        "swin_B_384_22k",
        "swin_L_224_22k",
        "swin_L_384_22k",
    ]:
        pretrain_img_size = int(args.backbone.split("_")[-2])
        backbone = build_swin_transformer(
            args.backbone,
            pretrain_img_size=pretrain_img_size,
            out_indices=tuple(return_interm_indices),
            dilation=False,
            use_checkpoint=use_checkpoint,
        )

        bb_num_channels = backbone.num_features[4 - len(return_interm_indices) :]
    else:
        raise NotImplementedError("Unknown backbone {}".format(args.backbone))

    assert len(bb_num_channels) == len(
        return_interm_indices
    ), f"len(bb_num_channels) {len(bb_num_channels)} != len(return_interm_indices) {len(return_interm_indices)}"

    model = Joiner(backbone, position_embedding)
    model.num_channels = bb_num_channels
    assert isinstance(
        bb_num_channels, List
    ), "bb_num_channels is expected to be a List but {}".format(type(bb_num_channels))
    # import ipdb; ipdb.set_trace()
    return model


# ----------------------------------------------------------------------------- models/GroundingDINO/ms_deform_attn.py

def _is_power_of_2(n):
    if (not isinstance(n, int)) or (n < 0):
        raise ValueError("invalid input for _is_power_of_2: {} (type: {})".format(n, type(n)))
    return (n & (n - 1) == 0) and n != 0

def multi_scale_deformable_attn_pytorch(
    value: torch.Tensor,
    value_spatial_shapes: torch.Tensor,
    sampling_locations: torch.Tensor,
    attention_weights: torch.Tensor,
) -> torch.Tensor:

    bs, _, num_heads, embed_dims = value.shape
    _, num_queries, num_heads, num_levels, num_points, _ = sampling_locations.shape
    value_list = value.split([H_ * W_ for H_, W_ in value_spatial_shapes], dim=1)
    sampling_grids = 2 * sampling_locations - 1
    sampling_value_list = []
    for level, (H_, W_) in enumerate(value_spatial_shapes):
        # bs, H_*W_, num_heads, embed_dims ->
        # bs, H_*W_, num_heads*embed_dims ->
        # bs, num_heads*embed_dims, H_*W_ ->
        # bs*num_heads, embed_dims, H_, W_
        value_l_ = (
            value_list[level].flatten(2).transpose(1, 2).reshape(bs * num_heads, embed_dims, H_, W_)
        )
        # bs, num_queries, num_heads, num_points, 2 ->
        # bs, num_heads, num_queries, num_points, 2 ->
        # bs*num_heads, num_queries, num_points, 2
        sampling_grid_l_ = sampling_grids[:, :, :, level].transpose(1, 2).flatten(0, 1)
        # bs*num_heads, embed_dims, num_queries, num_points
        sampling_value_l_ = F.grid_sample(
            value_l_, sampling_grid_l_, mode="bilinear", padding_mode="zeros", align_corners=False
        )
        sampling_value_list.append(sampling_value_l_)
    # (bs, num_queries, num_heads, num_levels, num_points) ->
    # (bs, num_heads, num_queries, num_levels, num_points) ->
    # (bs, num_heads, 1, num_queries, num_levels*num_points)
    attention_weights = attention_weights.transpose(1, 2).reshape(
        bs * num_heads, 1, num_queries, num_levels * num_points
    )
    output = (
        (torch.stack(sampling_value_list, dim=-2).flatten(-2) * attention_weights)
        .sum(-1)
        .view(bs, num_heads * embed_dims, num_queries)
    )
    return output.transpose(1, 2).contiguous()

class MultiScaleDeformableAttention(nn.Module):
    """Multi-Scale Deformable Attention Module used in Deformable-DETR

    `Deformable DETR: Deformable Transformers for End-to-End Object Detection.
    <https://arxiv.org/pdf/2010.04159.pdf>`_.

    Args:
        embed_dim (int): The embedding dimension of Attention. Default: 256.
        num_heads (int): The number of attention heads. Default: 8.
        num_levels (int): The number of feature map used in Attention. Default: 4.
        num_points (int): The number of sampling points for each query
            in each head. Default: 4.
        img2col_steps (int): The step used in image_to_column. Defualt: 64.
            dropout (float): Dropout layer used in output. Default: 0.1.
        batch_first (bool): if ``True``, then the input and output tensor will be
            provided as `(bs, n, embed_dim)`. Default: False. `(n, bs, embed_dim)`
    """

    def __init__(
        self,
        embed_dim: int = 256,
        num_heads: int = 8,
        num_levels: int = 4,
        num_points: int = 4,
        img2col_step: int = 64,
        batch_first: bool = False,
    ):
        super().__init__()
        if embed_dim % num_heads != 0:
            raise ValueError(
                "embed_dim must be divisible by num_heads, but got {} and {}".format(
                    embed_dim, num_heads
                )
            )
        head_dim = embed_dim // num_heads

        self.batch_first = batch_first

        if not _is_power_of_2(head_dim):
            warnings.warn(
                """
                You'd better set d_model in MSDeformAttn to make sure that
                each dim of the attention head a power of 2, which is more efficient.
                """
            )

        self.im2col_step = img2col_step
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_levels = num_levels
        self.num_points = num_points
        self.sampling_offsets = nn.Linear(embed_dim, num_heads * num_levels * num_points * 2)
        self.attention_weights = nn.Linear(embed_dim, num_heads * num_levels * num_points)
        self.value_proj = nn.Linear(embed_dim, embed_dim)
        self.output_proj = nn.Linear(embed_dim, embed_dim)

        self.init_weights()

    def _reset_parameters(self):
        return self.init_weights()

    def init_weights(self):
        """
        Default initialization for Parameters of Module.
        """
        constant_(self.sampling_offsets.weight.data, 0.0)
        thetas = torch.arange(self.num_heads, dtype=torch.float32) * (
            2.0 * math.pi / self.num_heads
        )
        grid_init = torch.stack([thetas.cos(), thetas.sin()], -1)
        grid_init = (
            (grid_init / grid_init.abs().max(-1, keepdim=True)[0])
            .view(self.num_heads, 1, 1, 2)
            .repeat(1, self.num_levels, self.num_points, 1)
        )
        for i in range(self.num_points):
            grid_init[:, :, i, :] *= i + 1
        with torch.no_grad():
            self.sampling_offsets.bias = nn.Parameter(grid_init.view(-1))
        constant_(self.attention_weights.weight.data, 0.0)
        constant_(self.attention_weights.bias.data, 0.0)
        xavier_uniform_(self.value_proj.weight.data)
        constant_(self.value_proj.bias.data, 0.0)
        xavier_uniform_(self.output_proj.weight.data)
        constant_(self.output_proj.bias.data, 0.0)

    def freeze_sampling_offsets(self):
        print("Freeze sampling offsets")
        self.sampling_offsets.weight.requires_grad = False
        self.sampling_offsets.bias.requires_grad = False

    def freeze_attention_weights(self):
        print("Freeze attention weights")
        self.attention_weights.weight.requires_grad = False
        self.attention_weights.bias.requires_grad = False

    def forward(
        self,
        query: torch.Tensor,
        key: Optional[torch.Tensor] = None,
        value: Optional[torch.Tensor] = None,
        query_pos: Optional[torch.Tensor] = None,
        key_padding_mask: Optional[torch.Tensor] = None,
        reference_points: Optional[torch.Tensor] = None,
        spatial_shapes: Optional[torch.Tensor] = None,
        level_start_index: Optional[torch.Tensor] = None,
        **kwargs
    ) -> torch.Tensor:

        """Forward Function of MultiScaleDeformableAttention

        Args:
            query (torch.Tensor): Query embeddings with shape
                `(num_query, bs, embed_dim)`
            key (torch.Tensor): Key embeddings with shape
                `(num_key, bs, embed_dim)`
            value (torch.Tensor): Value embeddings with shape
                `(num_key, bs, embed_dim)`
            query_pos (torch.Tensor): The position embedding for `query`. Default: None.
            key_padding_mask (torch.Tensor): ByteTensor for `query`, with shape `(bs, num_key)`,
                indicating which elements within `key` to be ignored in attention.
            reference_points (torch.Tensor): The normalized reference points
                with shape `(bs, num_query, num_levels, 2)`,
                all elements is range in [0, 1], top-left (0, 0),
                bottom-right (1, 1), including padding are.
                or `(N, Length_{query}, num_levels, 4)`, add additional
                two dimensions `(h, w)` to form reference boxes.
            spatial_shapes (torch.Tensor): Spatial shape of features in different levels.
                With shape `(num_levels, 2)`, last dimension represents `(h, w)`.
            level_start_index (torch.Tensor): The start index of each level. A tensor with
                shape `(num_levels, )` which can be represented as
                `[0, h_0 * w_0, h_0 * w_0 + h_1 * w_1, ...]`.

        Returns:
            torch.Tensor: forward results with shape `(num_query, bs, embed_dim)`
        """

        if value is None:
            value = query

        if query_pos is not None:
            query = query + query_pos

        if not self.batch_first:
            # change to (bs, num_query ,embed_dims)
            query = query.permute(1, 0, 2)
            value = value.permute(1, 0, 2)

        bs, num_query, _ = query.shape
        bs, num_value, _ = value.shape

        assert (spatial_shapes[:, 0] * spatial_shapes[:, 1]).sum() == num_value

        value = self.value_proj(value)
        if key_padding_mask is not None:
            value = value.masked_fill(key_padding_mask[..., None], float(0))
        value = value.view(bs, num_value, self.num_heads, -1)
        sampling_offsets = self.sampling_offsets(query).view(
            bs, num_query, self.num_heads, self.num_levels, self.num_points, 2
        )
        attention_weights = self.attention_weights(query).view(
            bs, num_query, self.num_heads, self.num_levels * self.num_points
        )
        attention_weights = attention_weights.softmax(-1)
        attention_weights = attention_weights.view(
            bs,
            num_query,
            self.num_heads,
            self.num_levels,
            self.num_points,
        )

        # bs, num_query, num_heads, num_levels, num_points, 2
        if reference_points.shape[-1] == 2:
            offset_normalizer = torch.stack([spatial_shapes[..., 1], spatial_shapes[..., 0]], -1)
            sampling_locations = (
                reference_points[:, :, None, :, None, :]
                + sampling_offsets / offset_normalizer[None, None, None, :, None, :]
            )
        elif reference_points.shape[-1] == 4:
            sampling_locations = (
                reference_points[:, :, None, :, None, :2]
                + sampling_offsets
                / self.num_points
                * reference_points[:, :, None, :, None, 2:]
                * 0.5
            )
        else:
            raise ValueError(
                "Last dim of reference_points must be 2 or 4, but get {} instead.".format(
                    reference_points.shape[-1]
                )
            )
    
        output = multi_scale_deformable_attn_pytorch(  # vendored: no compiled MultiScaleDeformableAttention op; the pure-PyTorch path (grid_sample) runs on CPU and CUDA alike
            value, spatial_shapes, sampling_locations, attention_weights
        )

        output = self.output_proj(output)

        if not self.batch_first:
            output = output.permute(1, 0, 2)

        return output


MSDeformAttn = MultiScaleDeformableAttention  # vendored: transformer.py imports the module under this name

# ----------------------------------------------------------------------------- models/GroundingDINO/utils.py

def _get_clones(module, N, layer_share=False):
    # import ipdb; ipdb.set_trace()
    if layer_share:
        return nn.ModuleList([module for i in range(N)])
    else:
        return nn.ModuleList([copy.deepcopy(module) for i in range(N)])

def get_sine_pos_embed(
    pos_tensor: torch.Tensor,
    num_pos_feats: int = 128,
    temperature: int = 10000,
    exchange_xy: bool = True,
):
    """generate sine position embedding from a position tensor
    Args:
        pos_tensor (torch.Tensor): shape: [..., n].
        num_pos_feats (int): projected shape for each float in the tensor.
        temperature (int): temperature in the sine/cosine function.
        exchange_xy (bool, optional): exchange pos x and pos y. \
            For example, input tensor is [x,y], the results will be [pos(y), pos(x)]. Defaults to True.
    Returns:
        pos_embed (torch.Tensor): shape: [..., n*num_pos_feats].
    """
    scale = 2 * math.pi
    dim_t = torch.arange(num_pos_feats, dtype=torch.float32, device=pos_tensor.device)
    dim_t = temperature ** (2 * torch.div(dim_t, 2, rounding_mode="floor") / num_pos_feats)

    def sine_func(x: torch.Tensor):
        sin_x = x * scale / dim_t
        sin_x = torch.stack((sin_x[..., 0::2].sin(), sin_x[..., 1::2].cos()), dim=3).flatten(2)
        return sin_x

    pos_res = [sine_func(x) for x in pos_tensor.split([1] * pos_tensor.shape[-1], dim=-1)]
    if exchange_xy:
        pos_res[0], pos_res[1] = pos_res[1], pos_res[0]
    pos_res = torch.cat(pos_res, dim=-1)
    return pos_res

# vendored: the docstring below is r-prefixed (upstream's `\sum` is an invalid escape since Python 3.12)
def gen_encoder_output_proposals(
    memory: Tensor, memory_padding_mask: Tensor, spatial_shapes: Tensor, learnedwh=None
):
    r"""
    Input:
        - memory: bs, \sum{hw}, d_model
        - memory_padding_mask: bs, \sum{hw}
        - spatial_shapes: nlevel, 2
        - learnedwh: 2
    Output:
        - output_memory: bs, \sum{hw}, d_model
        - output_proposals: bs, \sum{hw}, 4
    """
    N_, S_, C_ = memory.shape
    proposals = []
    _cur = 0
    for lvl, (H_, W_) in enumerate(spatial_shapes):
        mask_flatten_ = memory_padding_mask[:, _cur : (_cur + H_ * W_)].view(N_, H_, W_, 1)
        valid_H = torch.sum(~mask_flatten_[:, :, 0, 0], 1)
        valid_W = torch.sum(~mask_flatten_[:, 0, :, 0], 1)

        # import ipdb; ipdb.set_trace()

        grid_y, grid_x = torch.meshgrid(
            torch.linspace(0, H_ - 1, H_, dtype=torch.float32, device=memory.device),
            torch.linspace(0, W_ - 1, W_, dtype=torch.float32, device=memory.device),
        )
        grid = torch.cat([grid_x.unsqueeze(-1), grid_y.unsqueeze(-1)], -1)  # H_, W_, 2

        scale = torch.cat([valid_W.unsqueeze(-1), valid_H.unsqueeze(-1)], 1).view(N_, 1, 1, 2)
        grid = (grid.unsqueeze(0).expand(N_, -1, -1, -1) + 0.5) / scale

        if learnedwh is not None:
            # import ipdb; ipdb.set_trace()
            wh = torch.ones_like(grid) * learnedwh.sigmoid() * (2.0**lvl)
        else:
            wh = torch.ones_like(grid) * 0.05 * (2.0**lvl)

        # scale = torch.cat([W_[None].unsqueeze(-1), H_[None].unsqueeze(-1)], 1).view(1, 1, 1, 2).repeat(N_, 1, 1, 1)
        # grid = (grid.unsqueeze(0).expand(N_, -1, -1, -1) + 0.5) / scale
        # wh = torch.ones_like(grid) / scale
        proposal = torch.cat((grid, wh), -1).view(N_, -1, 4)
        proposals.append(proposal)
        _cur += H_ * W_
    # import ipdb; ipdb.set_trace()
    output_proposals = torch.cat(proposals, 1)
    output_proposals_valid = ((output_proposals > 0.01) & (output_proposals < 0.99)).all(
        -1, keepdim=True
    )
    output_proposals = torch.log(output_proposals / (1 - output_proposals))  # unsigmoid
    output_proposals = output_proposals.masked_fill(memory_padding_mask.unsqueeze(-1), float("inf"))
    output_proposals = output_proposals.masked_fill(~output_proposals_valid, float("inf"))

    output_memory = memory
    output_memory = output_memory.masked_fill(memory_padding_mask.unsqueeze(-1), float(0))
    output_memory = output_memory.masked_fill(~output_proposals_valid, float(0))

    # output_memory = output_memory.masked_fill(memory_padding_mask.unsqueeze(-1), float('inf'))
    # output_memory = output_memory.masked_fill(~output_proposals_valid, float('inf'))

    return output_memory, output_proposals

def sigmoid_focal_loss(
    inputs, targets, num_boxes, alpha: float = 0.25, gamma: float = 2, no_reduction=False
):
    """
    Loss used in RetinaNet for dense detection: https://arxiv.org/abs/1708.02002.
    Args:
        inputs: A float tensor of arbitrary shape.
                The predictions for each example.
        targets: A float tensor with the same shape as inputs. Stores the binary
                 classification label for each element in inputs
                (0 for the negative class and 1 for the positive class).
        alpha: (optional) Weighting factor in range (0,1) to balance
                positive vs negative examples. Default = -1 (no weighting).
        gamma: Exponent of the modulating factor (1 - p_t) to
               balance easy vs hard examples.
    Returns:
        Loss tensor
    """
    prob = inputs.sigmoid()
    ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction="none")
    p_t = prob * targets + (1 - prob) * (1 - targets)
    loss = ce_loss * ((1 - p_t) ** gamma)

    if alpha >= 0:
        alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
        loss = alpha_t * loss

    if no_reduction:
        return loss

    return loss.mean(1).sum() / num_boxes

class MLP(nn.Module):
    """Very simple multi-layer perceptron (also called FFN)"""

    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        self.num_layers = num_layers
        h = [hidden_dim] * (num_layers - 1)
        self.layers = nn.ModuleList(
            nn.Linear(n, k) for n, k in zip([input_dim] + h, h + [output_dim])
        )

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = F.relu(layer(x)) if i < self.num_layers - 1 else layer(x)
        return x

def _get_activation_fn(activation, d_model=256, batch_dim=0):
    """Return an activation function given a string"""
    if activation == "relu":
        return F.relu
    if activation == "gelu":
        return F.gelu
    if activation == "glu":
        return F.glu
    if activation == "prelu":
        return nn.PReLU()
    if activation == "selu":
        return F.selu

    raise RuntimeError(f"activation should be relu/gelu, not {activation}.")

def gen_sineembed_for_position(pos_tensor):
    # n_query, bs, _ = pos_tensor.size()
    # sineembed_tensor = torch.zeros(n_query, bs, 256)
    scale = 2 * math.pi
    dim_t = torch.arange(128, dtype=torch.float32, device=pos_tensor.device)
    dim_t = 10000 ** (2 * (torch.div(dim_t, 2, rounding_mode='floor')) / 128)
    x_embed = pos_tensor[:, :, 0] * scale
    y_embed = pos_tensor[:, :, 1] * scale
    pos_x = x_embed[:, :, None] / dim_t
    pos_y = y_embed[:, :, None] / dim_t
    pos_x = torch.stack((pos_x[:, :, 0::2].sin(), pos_x[:, :, 1::2].cos()), dim=3).flatten(2)
    pos_y = torch.stack((pos_y[:, :, 0::2].sin(), pos_y[:, :, 1::2].cos()), dim=3).flatten(2)
    if pos_tensor.size(-1) == 2:
        pos = torch.cat((pos_y, pos_x), dim=2)
    elif pos_tensor.size(-1) == 4:
        w_embed = pos_tensor[:, :, 2] * scale
        pos_w = w_embed[:, :, None] / dim_t
        pos_w = torch.stack((pos_w[:, :, 0::2].sin(), pos_w[:, :, 1::2].cos()), dim=3).flatten(2)

        h_embed = pos_tensor[:, :, 3] * scale
        pos_h = h_embed[:, :, None] / dim_t
        pos_h = torch.stack((pos_h[:, :, 0::2].sin(), pos_h[:, :, 1::2].cos()), dim=3).flatten(2)

        pos = torch.cat((pos_y, pos_x, pos_w, pos_h), dim=2)
    else:
        raise ValueError("Unknown pos_tensor shape(-1):{}".format(pos_tensor.size(-1)))
    return pos

class ContrastiveEmbed(nn.Module):
    def __init__(self, max_text_len=256):
        """
        Args:
            max_text_len: max length of text.
        """
        super().__init__()
        self.max_text_len = max_text_len

    def forward(self, x, text_dict):
        """_summary_

        Args:
            x (_type_): _description_
            text_dict (_type_): _description_
            {
                'encoded_text': encoded_text, # bs, 195, d_model
                'text_token_mask': text_token_mask, # bs, 195
                        # True for used tokens. False for padding tokens
            }
        Returns:
            _type_: _description_
        """
        assert isinstance(text_dict, dict)
        # print(x)  #torch.Size([2, 16320, 256])
        # print(text_dict)

        # import pdb;pdb.set_trace()
        y = text_dict["encoded_text"]  #torch.Size([2, 195, 256])
        text_token_mask = text_dict["text_token_mask"]

        res = x @ y.transpose(-1, -2)
        res.masked_fill_(~text_token_mask[:, None, :], float("-inf"))
        # 接着，对res进行掩码操作，将未使用的文本token（即padding的token）对应的得分置为负无穷float("-inf")。这是为了在计算相似度时，排除padding部分的影响。


        # padding to max_text_len
        new_res = torch.full((*res.shape[:-1], self.max_text_len), float("-inf"), device=res.device)
        new_res[..., : res.shape[-1]] = res  #torch.Size([2, 16320, 195])

        return new_res


# ----------------------------------------------------------------------------- models/GroundingDINO/transformer_vanilla.py

class TransformerEncoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        nhead,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
    ):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        # Implementation of Feedforward model
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before
        self.nhead = nhead

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        return tensor if pos is None else tensor + pos

    def forward(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        # repeat attn mask
        if src_mask.dim() == 3 and src_mask.shape[0] == src.shape[1]:
            # bs, num_q, num_k
            src_mask = src_mask.repeat(self.nhead, 1, 1)

        q = k = self.with_pos_embed(src, pos)

        src2 = self.self_attn(q, k, value=src, attn_mask=src_mask)[0]

        # src2 = self.self_attn(q, k, value=src, attn_mask=src_mask, key_padding_mask=src_key_padding_mask)[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src


# ----------------------------------------------------------------------------- models/GroundingDINO/fuse_modules.py

class BiMultiHeadAttention(nn.Module):
    def __init__(self, v_dim, l_dim, embed_dim, num_heads, dropout=0.1, cfg=None):
        super(BiMultiHeadAttention, self).__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.v_dim = v_dim
        self.l_dim = l_dim

        assert (
            self.head_dim * self.num_heads == self.embed_dim
        ), f"embed_dim must be divisible by num_heads (got `embed_dim`: {self.embed_dim} and `num_heads`: {self.num_heads})."
        self.scale = self.head_dim ** (-0.5)
        self.dropout = dropout

        self.v_proj = nn.Linear(self.v_dim, self.embed_dim)
        self.l_proj = nn.Linear(self.l_dim, self.embed_dim)
        self.values_v_proj = nn.Linear(self.v_dim, self.embed_dim)
        self.values_l_proj = nn.Linear(self.l_dim, self.embed_dim)

        self.out_v_proj = nn.Linear(self.embed_dim, self.v_dim)
        self.out_l_proj = nn.Linear(self.embed_dim, self.l_dim)

        self.stable_softmax_2d = True
        self.clamp_min_for_underflow = True
        self.clamp_max_for_overflow = True

        self._reset_parameters()

    def _shape(self, tensor: torch.Tensor, seq_len: int, bsz: int):
        return tensor.view(bsz, seq_len, self.num_heads, self.head_dim).transpose(1, 2).contiguous()

    def _reset_parameters(self):
        nn.init.xavier_uniform_(self.v_proj.weight)
        self.v_proj.bias.data.fill_(0)
        nn.init.xavier_uniform_(self.l_proj.weight)
        self.l_proj.bias.data.fill_(0)
        nn.init.xavier_uniform_(self.values_v_proj.weight)
        self.values_v_proj.bias.data.fill_(0)
        nn.init.xavier_uniform_(self.values_l_proj.weight)
        self.values_l_proj.bias.data.fill_(0)
        nn.init.xavier_uniform_(self.out_v_proj.weight)
        self.out_v_proj.bias.data.fill_(0)
        nn.init.xavier_uniform_(self.out_l_proj.weight)
        self.out_l_proj.bias.data.fill_(0)

    def forward(self, v, l, attention_mask_v=None, attention_mask_l=None):
        """_summary_

        Args:
            v (_type_): bs, n_img, dim
            l (_type_): bs, n_text, dim
            attention_mask_v (_type_, optional): _description_. bs, n_img
            attention_mask_l (_type_, optional): _description_. bs, n_text

        Returns:
            _type_: _description_
        """
        # if os.environ.get('IPDB_SHILONG_DEBUG', None) == 'INFO':
        #     import ipdb; ipdb.set_trace()
        bsz, tgt_len, _ = v.size()

        query_states = self.v_proj(v) * self.scale
        key_states = self._shape(self.l_proj(l), -1, bsz)
        value_v_states = self._shape(self.values_v_proj(v), -1, bsz)
        value_l_states = self._shape(self.values_l_proj(l), -1, bsz)

        proj_shape = (bsz * self.num_heads, -1, self.head_dim)
        query_states = self._shape(query_states, tgt_len, bsz).view(*proj_shape)
        key_states = key_states.view(*proj_shape)
        value_v_states = value_v_states.view(*proj_shape)
        value_l_states = value_l_states.view(*proj_shape)

        src_len = key_states.size(1)
        attn_weights = torch.bmm(query_states, key_states.transpose(1, 2))  # bs*nhead, nimg, ntxt

        if attn_weights.size() != (bsz * self.num_heads, tgt_len, src_len):
            raise ValueError(
                f"Attention weights should be of size {(bsz * self.num_heads, tgt_len, src_len)}, but is {attn_weights.size()}"
            )

        if self.stable_softmax_2d:
            attn_weights = attn_weights - attn_weights.max()

        if self.clamp_min_for_underflow:
            attn_weights = torch.clamp(
                attn_weights, min=-50000
            )  # Do not increase -50000, data type half has quite limited range
        if self.clamp_max_for_overflow:
            attn_weights = torch.clamp(
                attn_weights, max=50000
            )  # Do not increase 50000, data type half has quite limited range

        attn_weights_T = attn_weights.transpose(1, 2)
        attn_weights_l = attn_weights_T - torch.max(attn_weights_T, dim=-1, keepdim=True)[0]
        if self.clamp_min_for_underflow:
            attn_weights_l = torch.clamp(
                attn_weights_l, min=-50000
            )  # Do not increase -50000, data type half has quite limited range
        if self.clamp_max_for_overflow:
            attn_weights_l = torch.clamp(
                attn_weights_l, max=50000
            )  # Do not increase 50000, data type half has quite limited range

        # mask vison for language
        if attention_mask_v is not None:
            attention_mask_v = (
                attention_mask_v[:, None, None, :].repeat(1, self.num_heads, 1, 1).flatten(0, 1)
            )
            attn_weights_l.masked_fill_(attention_mask_v, float("-inf"))

        attn_weights_l = attn_weights_l.softmax(dim=-1)

        # mask language for vision
        if attention_mask_l is not None:
            attention_mask_l = (
                attention_mask_l[:, None, None, :].repeat(1, self.num_heads, 1, 1).flatten(0, 1)
            )
            attn_weights.masked_fill_(attention_mask_l, float("-inf"))
        attn_weights_v = attn_weights.softmax(dim=-1)

        attn_probs_v = F.dropout(attn_weights_v, p=self.dropout, training=self.training)
        attn_probs_l = F.dropout(attn_weights_l, p=self.dropout, training=self.training)

        attn_output_v = torch.bmm(attn_probs_v, value_l_states)
        attn_output_l = torch.bmm(attn_probs_l, value_v_states)

        if attn_output_v.size() != (bsz * self.num_heads, tgt_len, self.head_dim):
            raise ValueError(
                f"`attn_output_v` should be of size {(bsz, self.num_heads, tgt_len, self.head_dim)}, but is {attn_output_v.size()}"
            )

        if attn_output_l.size() != (bsz * self.num_heads, src_len, self.head_dim):
            raise ValueError(
                f"`attn_output_l` should be of size {(bsz, self.num_heads, src_len, self.head_dim)}, but is {attn_output_l.size()}"
            )

        attn_output_v = attn_output_v.view(bsz, self.num_heads, tgt_len, self.head_dim)
        attn_output_v = attn_output_v.transpose(1, 2)
        attn_output_v = attn_output_v.reshape(bsz, tgt_len, self.embed_dim)

        attn_output_l = attn_output_l.view(bsz, self.num_heads, src_len, self.head_dim)
        attn_output_l = attn_output_l.transpose(1, 2)
        attn_output_l = attn_output_l.reshape(bsz, src_len, self.embed_dim)

        attn_output_v = self.out_v_proj(attn_output_v)
        attn_output_l = self.out_l_proj(attn_output_l)

        return attn_output_v, attn_output_l

class BiAttentionBlock(nn.Module):
    def __init__(
        self,
        v_dim,
        l_dim,
        embed_dim,
        num_heads,
        dropout=0.1,
        drop_path=0.0,
        init_values=1e-4,
        cfg=None,
    ):
        """
        Inputs:
            embed_dim - Dimensionality of input and attention feature vectors
            hidden_dim - Dimensionality of hidden layer in feed-forward network
                         (usually 2-4x larger than embed_dim)
            num_heads - Number of heads to use in the Multi-Head Attention block
            dropout - Amount of dropout to apply in the feed-forward network
        """
        super(BiAttentionBlock, self).__init__()

        # pre layer norm
        self.layer_norm_v = nn.LayerNorm(v_dim)
        self.layer_norm_l = nn.LayerNorm(l_dim)
        self.attn = BiMultiHeadAttention(
            v_dim=v_dim, l_dim=l_dim, embed_dim=embed_dim, num_heads=num_heads, dropout=dropout
        )

        # add layer scale for training stability
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()
        self.gamma_v = nn.Parameter(init_values * torch.ones((v_dim)), requires_grad=True)
        self.gamma_l = nn.Parameter(init_values * torch.ones((l_dim)), requires_grad=True)

    def forward(self, v, l, attention_mask_v=None, attention_mask_l=None):
        v = self.layer_norm_v(v)
        l = self.layer_norm_l(l)
        delta_v, delta_l = self.attn(
            v, l, attention_mask_v=attention_mask_v, attention_mask_l=attention_mask_l
        )
        # v, l = v + delta_v, l + delta_l
        v = v + self.drop_path(self.gamma_v * delta_v)
        l = l + self.drop_path(self.gamma_l * delta_l)
        return v, l


# ----------------------------------------------------------------------------- models/GroundingDINO/bertwarper.py

class BertModelWarper(nn.Module):
    def __init__(self, bert_model):
        super().__init__()
        # self.bert = bert_modelc

        self.config = bert_model.config
        self.embeddings = bert_model.embeddings
        self.encoder = bert_model.encoder
        self.pooler = bert_model.pooler

        self.get_extended_attention_mask = bert_model.get_extended_attention_mask
        self.invert_attention_mask = bert_model.invert_attention_mask
        self.get_head_mask = bert_model.get_head_mask

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        position_ids=None,
        head_mask=None,
        inputs_embeds=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
        past_key_values=None,
        use_cache=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        r"""
        encoder_hidden_states  (:obj:`torch.FloatTensor` of shape :obj:`(batch_size, sequence_length, hidden_size)`, `optional`):
            Sequence of hidden-states at the output of the last layer of the encoder. Used in the cross-attention if
            the model is configured as a decoder.
        encoder_attention_mask (:obj:`torch.FloatTensor` of shape :obj:`(batch_size, sequence_length)`, `optional`):
            Mask to avoid performing attention on the padding token indices of the encoder input. This mask is used in
            the cross-attention if the model is configured as a decoder. Mask values selected in ``[0, 1]``:

            - 1 for tokens that are **not masked**,
            - 0 for tokens that are **masked**.
        past_key_values (:obj:`tuple(tuple(torch.FloatTensor))` of length :obj:`config.n_layers` with each tuple having 4 tensors of shape :obj:`(batch_size, num_heads, sequence_length - 1, embed_size_per_head)`):
            Contains precomputed key and value hidden states of the attention blocks. Can be used to speed up decoding.

            If :obj:`past_key_values` are used, the user can optionally input only the last :obj:`decoder_input_ids`
            (those that don't have their past key value states given to this model) of shape :obj:`(batch_size, 1)`
            instead of all :obj:`decoder_input_ids` of shape :obj:`(batch_size, sequence_length)`.
        use_cache (:obj:`bool`, `optional`):
            If set to :obj:`True`, :obj:`past_key_values` key value states are returned and can be used to speed up
            decoding (see :obj:`past_key_values`).
        """
        output_attentions = (
            output_attentions if output_attentions is not None else self.config.output_attentions
        )
        output_hidden_states = (
            output_hidden_states
            if output_hidden_states is not None
            else self.config.output_hidden_states
        )
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        if self.config.is_decoder:
            use_cache = use_cache if use_cache is not None else self.config.use_cache
        else:
            use_cache = False

        if input_ids is not None and inputs_embeds is not None:
            raise ValueError("You cannot specify both input_ids and inputs_embeds at the same time")
        elif input_ids is not None:
            input_shape = input_ids.size()
            batch_size, seq_length = input_shape
        elif inputs_embeds is not None:
            input_shape = inputs_embeds.size()[:-1]
            batch_size, seq_length = input_shape
        else:
            raise ValueError("You have to specify either input_ids or inputs_embeds")

        device = input_ids.device if input_ids is not None else inputs_embeds.device

        # past_key_values_length
        past_key_values_length = (
            past_key_values[0][0].shape[2] if past_key_values is not None else 0
        )

        if attention_mask is None:
            attention_mask = torch.ones(
                ((batch_size, seq_length + past_key_values_length)), device=device
            )
        if token_type_ids is None:
            token_type_ids = torch.zeros(input_shape, dtype=torch.long, device=device)

        # We can provide a self-attention mask of dimensions [batch_size, from_seq_length, to_seq_length]
        # ourselves in which case we just need to make it broadcastable to all heads.
        extended_attention_mask: torch.Tensor = self.get_extended_attention_mask(
            attention_mask, input_shape, device
        )

        # If a 2D or 3D attention mask is provided for the cross-attention
        # we need to make broadcastable to [batch_size, num_heads, seq_length, seq_length]
        if self.config.is_decoder and encoder_hidden_states is not None:
            encoder_batch_size, encoder_sequence_length, _ = encoder_hidden_states.size()
            encoder_hidden_shape = (encoder_batch_size, encoder_sequence_length)
            if encoder_attention_mask is None:
                encoder_attention_mask = torch.ones(encoder_hidden_shape, device=device)
            encoder_extended_attention_mask = self.invert_attention_mask(encoder_attention_mask)
        else:
            encoder_extended_attention_mask = None
        # if os.environ.get('IPDB_SHILONG_DEBUG', None) == 'INFO':
        #     import ipdb; ipdb.set_trace()

        # Prepare head mask if needed
        # 1.0 in head_mask indicate we keep the head
        # attention_probs has shape bsz x n_heads x N x N
        # input head_mask has shape [num_heads] or [num_hidden_layers x num_heads]
        # and head_mask is converted to shape [num_hidden_layers x batch x num_heads x seq_length x seq_length]
        head_mask = self.get_head_mask(head_mask, self.config.num_hidden_layers)

        embedding_output = self.embeddings(
            input_ids=input_ids,
            position_ids=position_ids,
            token_type_ids=token_type_ids,
            inputs_embeds=inputs_embeds,
            past_key_values_length=past_key_values_length,
        )

        encoder_outputs = self.encoder(
            embedding_output,
            attention_mask=extended_attention_mask,
            head_mask=head_mask,
            encoder_hidden_states=encoder_hidden_states,
            encoder_attention_mask=encoder_extended_attention_mask,
            past_key_values=past_key_values,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        sequence_output = encoder_outputs[0]
        pooled_output = self.pooler(sequence_output) if self.pooler is not None else None

        if not return_dict:
            return (sequence_output, pooled_output) + encoder_outputs[1:]

        return BaseModelOutputWithPoolingAndCrossAttentions(
            last_hidden_state=sequence_output,
            pooler_output=pooled_output,
            past_key_values=encoder_outputs.past_key_values,
            hidden_states=encoder_outputs.hidden_states,
            attentions=encoder_outputs.attentions,
            cross_attentions=encoder_outputs.cross_attentions,
        )

def generate_masks_with_special_tokens_and_transfer_map(tokenized, special_tokens_list, tokenizer):
    """Generate attention mask between each pair of special tokens
    Args:
        input_ids (torch.Tensor): input ids. Shape: [bs, num_token]
        special_tokens_mask (list): special tokens mask.
    Returns:
        torch.Tensor: attention mask between each special tokens.
    """
    input_ids = tokenized["input_ids"]
    bs, num_token = input_ids.shape
    # special_tokens_mask: bs, num_token. 1 for special tokens. 0 for normal tokens
    special_tokens_mask = torch.zeros((bs, num_token), device=input_ids.device).bool()
    for special_token in special_tokens_list:
        special_tokens_mask |= input_ids == special_token

    # idxs: each row is a list of indices of special tokens
    idxs = torch.nonzero(special_tokens_mask)

    # generate attention mask and positional ids
    attention_mask = (
        torch.eye(num_token, device=input_ids.device).bool().unsqueeze(0).repeat(bs, 1, 1)
    )
    position_ids = torch.zeros((bs, num_token), device=input_ids.device)
    cate_to_token_mask_list = [[] for _ in range(bs)]
    previous_col = 0
    for i in range(idxs.shape[0]):
        row, col = idxs[i]
        if (col == 0) or (col == num_token - 1):
            attention_mask[row, col, col] = True
            position_ids[row, col] = 0
        else:
            attention_mask[row, previous_col + 1 : col + 1, previous_col + 1 : col + 1] = True
            position_ids[row, previous_col + 1 : col + 1] = torch.arange(
                0, col - previous_col, device=input_ids.device
            )
            c2t_maski = torch.zeros((num_token), device=input_ids.device).bool()
            c2t_maski[previous_col + 1 : col] = True
            cate_to_token_mask_list[row].append(c2t_maski)
        previous_col = col

    cate_to_token_mask_list = [
        torch.stack(cate_to_token_mask_listi, dim=0)
        for cate_to_token_mask_listi in cate_to_token_mask_list
    ]

    # # padding mask
    # padding_mask = tokenized['attention_mask']
    # attention_mask = attention_mask & padding_mask.unsqueeze(1).bool() & padding_mask.unsqueeze(2).bool()

    return attention_mask, position_ids.to(torch.long), cate_to_token_mask_list


# ----------------------------------------------------------------------------- models/GroundingDINO/transformer.py

class Transformer(nn.Module):
    def __init__(
        self,
        d_model=256,
        nhead=8,
        num_queries=300,
        num_encoder_layers=6,
        num_unicoder_layers=0,
        num_decoder_layers=6,
        dim_feedforward=2048,
        dropout=0.0,
        activation="relu",
        normalize_before=False,
        return_intermediate_dec=False,
        query_dim=4,
        num_patterns=0,
        # for deformable encoder
        num_feature_levels=1,
        enc_n_points=4,
        dec_n_points=4,
        # init query
        learnable_tgt_init=False,
        # two stage
        two_stage_type="no",  # ['no', 'standard', 'early', 'combine', 'enceachlayer', 'enclayer1']
        embed_init_tgt=False,
        # for text
        use_text_enhancer=False,
        use_fusion_layer=False,
        use_checkpoint=False,
        use_transformer_ckpt=False,
        use_text_cross_attention=False,
        text_dropout=0.1,
        fusion_dropout=0.1,
        fusion_droppath=0.0,
    ):
        super().__init__()
        self.num_feature_levels = num_feature_levels
        self.num_encoder_layers = num_encoder_layers
        self.num_unicoder_layers = num_unicoder_layers
        self.num_decoder_layers = num_decoder_layers
        self.num_queries = num_queries
        assert query_dim == 4

        # choose encoder layer type
        encoder_layer = DeformableTransformerEncoderLayer(
            d_model, dim_feedforward, dropout, activation, num_feature_levels, nhead, enc_n_points
        )

        if use_text_enhancer:
            text_enhance_layer = TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead // 2,
                dim_feedforward=dim_feedforward // 2,
                dropout=text_dropout,
            )
        else:
            text_enhance_layer = None

        if use_fusion_layer:
            feature_fusion_layer = BiAttentionBlock(
                v_dim=d_model,
                l_dim=d_model,
                embed_dim=dim_feedforward // 2,
                num_heads=nhead // 2,
                dropout=fusion_dropout,
                drop_path=fusion_droppath,
            )
        else:
            feature_fusion_layer = None

        encoder_norm = nn.LayerNorm(d_model) if normalize_before else None
        assert encoder_norm is None
        self.encoder = TransformerEncoder(
            encoder_layer,
            num_encoder_layers,
            d_model=d_model,
            num_queries=num_queries,
            text_enhance_layer=text_enhance_layer,
            feature_fusion_layer=feature_fusion_layer,
            use_checkpoint=use_checkpoint,
            use_transformer_ckpt=use_transformer_ckpt,
        )

        # choose decoder layer type
        decoder_layer = DeformableTransformerDecoderLayer(
            d_model,
            dim_feedforward,
            dropout,
            activation,
            num_feature_levels,
            nhead,
            dec_n_points,
            use_text_cross_attention=use_text_cross_attention,
        )

        decoder_norm = nn.LayerNorm(d_model)
        self.decoder = TransformerDecoder(
            decoder_layer,
            num_decoder_layers,
            decoder_norm,
            return_intermediate=return_intermediate_dec,
            d_model=d_model,
            query_dim=query_dim,
            num_feature_levels=num_feature_levels,
        )

        self.d_model = d_model
        self.nhead = nhead
        self.dec_layers = num_decoder_layers
        self.num_queries = num_queries  # useful for single stage model only
        self.num_patterns = num_patterns
        if not isinstance(num_patterns, int):
            Warning("num_patterns should be int but {}".format(type(num_patterns)))
            self.num_patterns = 0

        if num_feature_levels > 1:
            if self.num_encoder_layers > 0:
                self.level_embed = nn.Parameter(torch.Tensor(num_feature_levels, d_model))
            else:
                self.level_embed = None

        self.learnable_tgt_init = learnable_tgt_init
        assert learnable_tgt_init, "why not learnable_tgt_init"
        self.embed_init_tgt = embed_init_tgt
        if (two_stage_type != "no" and embed_init_tgt) or (two_stage_type == "no"):
            self.tgt_embed = nn.Embedding(self.num_queries, d_model)
            nn.init.normal_(self.tgt_embed.weight.data)
        else:
            self.tgt_embed = None

        # for two stage
        self.two_stage_type = two_stage_type
        assert two_stage_type in ["no", "standard"], "unknown param {} of two_stage_type".format(
            two_stage_type
        )
        if two_stage_type == "standard":
            # anchor selection at the output of encoder
            self.enc_output = nn.Linear(d_model, d_model)
            self.enc_output_norm = nn.LayerNorm(d_model)
            self.two_stage_wh_embedding = None

        if two_stage_type == "no":
            self.init_ref_points(num_queries)  # init self.refpoint_embed

        self.enc_out_class_embed = None
        self.enc_out_bbox_embed = None

        self._reset_parameters()

    def _reset_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        for m in self.modules():
            if isinstance(m, MSDeformAttn):
                m._reset_parameters()
        if self.num_feature_levels > 1 and self.level_embed is not None:
            nn.init.normal_(self.level_embed)

    def get_valid_ratio(self, mask):
        _, H, W = mask.shape
        valid_H = torch.sum(~mask[:, :, 0], 1)
        valid_W = torch.sum(~mask[:, 0, :], 1)
        valid_ratio_h = valid_H.float() / H
        valid_ratio_w = valid_W.float() / W
        valid_ratio = torch.stack([valid_ratio_w, valid_ratio_h], -1)
        return valid_ratio

    def init_ref_points(self, use_num_queries):
        self.refpoint_embed = nn.Embedding(use_num_queries, 4)

    def forward(self, srcs, masks, refpoint_embed, pos_embeds, tgt, attn_mask=None, text_dict=None):
        """
        Input:
            - srcs: List of multi features [bs, ci, hi, wi]
            - masks: List of multi masks [bs, hi, wi]
            - refpoint_embed: [bs, num_dn, 4]. None in infer
            - pos_embeds: List of multi pos embeds [bs, ci, hi, wi]
            - tgt: [bs, num_dn, d_model]. None in infer

        """
        # prepare input for encoder
        src_flatten = []
        mask_flatten = []
        lvl_pos_embed_flatten = []
        spatial_shapes = []
        for lvl, (src, mask, pos_embed) in enumerate(zip(srcs, masks, pos_embeds)):
            bs, c, h, w = src.shape
            spatial_shape = (h, w)
            spatial_shapes.append(spatial_shape)

            src = src.flatten(2).transpose(1, 2)  # bs, hw, c
            mask = mask.flatten(1)  # bs, hw
            pos_embed = pos_embed.flatten(2).transpose(1, 2)  # bs, hw, c
            if self.num_feature_levels > 1 and self.level_embed is not None:
                lvl_pos_embed = pos_embed + self.level_embed[lvl].view(1, 1, -1)
            else:
                lvl_pos_embed = pos_embed
            lvl_pos_embed_flatten.append(lvl_pos_embed)
            src_flatten.append(src)
            mask_flatten.append(mask)
        src_flatten = torch.cat(src_flatten, 1)  # bs, \sum{hxw}, c
        mask_flatten = torch.cat(mask_flatten, 1)  # bs, \sum{hxw}
        lvl_pos_embed_flatten = torch.cat(lvl_pos_embed_flatten, 1)  # bs, \sum{hxw}, c
        spatial_shapes = torch.as_tensor(
            spatial_shapes, dtype=torch.long, device=src_flatten.device
        )
        level_start_index = torch.cat(
            (spatial_shapes.new_zeros((1,)), spatial_shapes.prod(1).cumsum(0)[:-1])
        )
        valid_ratios = torch.stack([self.get_valid_ratio(m) for m in masks], 1)

        # two stage
        enc_topk_proposals = enc_refpoint_embed = None

        #########################################################
        # Begin Encoder
        #########################################################
        
        memory, memory_text = self.encoder(
            src_flatten,
            pos=lvl_pos_embed_flatten,
            level_start_index=level_start_index,
            spatial_shapes=spatial_shapes,
            valid_ratios=valid_ratios,
            key_padding_mask=mask_flatten,
            memory_text=text_dict["encoded_text"],
            text_attention_mask=~text_dict["text_token_mask"],
            # we ~ the mask . False means use the token; True means pad the token
            position_ids=text_dict["position_ids"],
            text_self_attention_masks=text_dict["text_self_attention_masks"],
        )
        
        #########################################################
        # End Encoder
        # - memory: bs, \sum{hw}, c
        # - mask_flatten: bs, \sum{hw}
        # - lvl_pos_embed_flatten: bs, \sum{hw}, c
        # - enc_intermediate_output: None or (nenc+1, bs, nq, c) or (nenc, bs, nq, c)
        # - enc_intermediate_refpoints: None or (nenc+1, bs, nq, c) or (nenc, bs, nq, c)
        #########################################################
        text_dict["encoded_text"] = memory_text
        # if os.environ.get("SHILONG_AMP_INFNAN_DEBUG") == '1':
        #     if memory.isnan().any() | memory.isinf().any():
        #         import ipdb; ipdb.set_trace()


        if self.two_stage_type == "standard":  #把encoder的输出作为proposal
            output_memory, output_proposals = gen_encoder_output_proposals(
                memory, mask_flatten, spatial_shapes
            )
            output_memory = self.enc_output_norm(self.enc_output(output_memory))

            if text_dict is not None:
                enc_outputs_class_unselected = self.enc_out_class_embed(output_memory, text_dict)
            else:
                enc_outputs_class_unselected = self.enc_out_class_embed(output_memory)

            topk_logits = enc_outputs_class_unselected.max(-1)[0]
            enc_outputs_coord_unselected = (
                self.enc_out_bbox_embed(output_memory) + output_proposals
            )  # (bs, \sum{hw}, 4) unsigmoid
            topk = self.num_queries

            topk_proposals = torch.topk(topk_logits, topk, dim=1)[1]  # bs, nq

            # gather boxes
            refpoint_embed_undetach = torch.gather(
                enc_outputs_coord_unselected, 1, topk_proposals.unsqueeze(-1).repeat(1, 1, 4)
            )  # unsigmoid
            refpoint_embed_ = refpoint_embed_undetach.detach()
            init_box_proposal = torch.gather(
                output_proposals, 1, topk_proposals.unsqueeze(-1).repeat(1, 1, 4)
            ).sigmoid()  # sigmoid

            # gather tgt
            tgt_undetach = torch.gather(
                output_memory, 1, topk_proposals.unsqueeze(-1).repeat(1, 1, self.d_model)
            )
            if self.embed_init_tgt:
                tgt_ = (
                    self.tgt_embed.weight[:, None, :].repeat(1, bs, 1).transpose(0, 1)
                )  # nq, bs, d_model
            else:
                tgt_ = tgt_undetach.detach()

            if refpoint_embed is not None:
                refpoint_embed = torch.cat([refpoint_embed, refpoint_embed_], dim=1)
                tgt = torch.cat([tgt, tgt_], dim=1)
            else:
                refpoint_embed, tgt = refpoint_embed_, tgt_

        elif self.two_stage_type == "no":
            tgt_ = (
                self.tgt_embed.weight[:, None, :].repeat(1, bs, 1).transpose(0, 1)
            )  # nq, bs, d_model
            refpoint_embed_ = (
                self.refpoint_embed.weight[:, None, :].repeat(1, bs, 1).transpose(0, 1)
            )  # nq, bs, 4

            if refpoint_embed is not None:
                refpoint_embed = torch.cat([refpoint_embed, refpoint_embed_], dim=1)
                tgt = torch.cat([tgt, tgt_], dim=1)
            else:
                refpoint_embed, tgt = refpoint_embed_, tgt_

            if self.num_patterns > 0:
                tgt_embed = tgt.repeat(1, self.num_patterns, 1)
                refpoint_embed = refpoint_embed.repeat(1, self.num_patterns, 1)
                tgt_pat = self.patterns.weight[None, :, :].repeat_interleave(
                    self.num_queries, 1
                )  # 1, n_q*n_pat, d_model
                tgt = tgt_embed + tgt_pat

            init_box_proposal = refpoint_embed_.sigmoid()

        else:
            raise NotImplementedError("unknown two_stage_type {}".format(self.two_stage_type))
        #########################################################
        # End preparing tgt
        # - tgt: bs, NQ, d_model
        # - refpoint_embed(unsigmoid): bs, NQ, d_model
        #########################################################

        #########################################################
        # Begin Decoder
        #########################################################

        #memory  torch.Size([2, 16320, 256])

        # import pdb;pdb.set_trace()
        hs, references = self.decoder(
            tgt=tgt.transpose(0, 1),
            memory=memory.transpose(0, 1),
            memory_key_padding_mask=mask_flatten,
            pos=lvl_pos_embed_flatten.transpose(0, 1),
            refpoints_unsigmoid=refpoint_embed.transpose(0, 1),
            level_start_index=level_start_index,
            spatial_shapes=spatial_shapes,
            valid_ratios=valid_ratios,
            tgt_mask=attn_mask,
            memory_text=text_dict["encoded_text"],
            text_attention_mask=~text_dict["text_token_mask"],
            # we ~ the mask . False means use the token; True means pad the token
        )
        #########################################################
        # End Decoder
        # hs: n_dec, bs, nq, d_model
        # references: n_dec+1, bs, nq, query_dim
        #########################################################

        #########################################################
        # Begin postprocess
        #########################################################
        if self.two_stage_type == "standard":
            hs_enc = tgt_undetach.unsqueeze(0)
            ref_enc = refpoint_embed_undetach.sigmoid().unsqueeze(0)
        else:
            hs_enc = ref_enc = None
        #########################################################
        # End postprocess
        # hs_enc: (n_enc+1, bs, nq, d_model) or (1, bs, nq, d_model) or (n_enc, bs, nq, d_model) or None
        # ref_enc: (n_enc+1, bs, nq, query_dim) or (1, bs, nq, query_dim) or (n_enc, bs, nq, d_model) or None
        #########################################################

        return hs, references, hs_enc, ref_enc, init_box_proposal

class TransformerEncoder(nn.Module):
    def __init__(
        self,
        encoder_layer,
        num_layers,
        d_model=256,
        num_queries=300,
        enc_layer_share=False,
        text_enhance_layer=None,
        feature_fusion_layer=None,
        use_checkpoint=False,
        use_transformer_ckpt=False,
    ):
        """_summary_

        Args:
            encoder_layer (_type_): _description_
            num_layers (_type_): _description_
            norm (_type_, optional): _description_. Defaults to None.
            d_model (int, optional): _description_. Defaults to 256.
            num_queries (int, optional): _description_. Defaults to 300.
            enc_layer_share (bool, optional): _description_. Defaults to False.

        """
        super().__init__()
        # prepare layers
        self.layers = []
        self.text_layers = []
        self.fusion_layers = []
        if num_layers > 0:
            self.layers = _get_clones(encoder_layer, num_layers, layer_share=enc_layer_share)

            if text_enhance_layer is not None:
                self.text_layers = _get_clones(
                    text_enhance_layer, num_layers, layer_share=enc_layer_share
                )
            if feature_fusion_layer is not None:
                self.fusion_layers = _get_clones(
                    feature_fusion_layer, num_layers, layer_share=enc_layer_share
                )
        else:
            self.layers = []
            del encoder_layer

            if text_enhance_layer is not None:
                self.text_layers = []
                del text_enhance_layer
            if feature_fusion_layer is not None:
                self.fusion_layers = []
                del feature_fusion_layer

        self.query_scale = None
        self.num_queries = num_queries
        self.num_layers = num_layers
        self.d_model = d_model

        self.use_checkpoint = use_checkpoint
        self.use_transformer_ckpt = use_transformer_ckpt

    @staticmethod
    def get_reference_points(spatial_shapes, valid_ratios, device):
        reference_points_list = []
        for lvl, (H_, W_) in enumerate(spatial_shapes):

            ref_y, ref_x = torch.meshgrid(
                torch.linspace(0.5, H_ - 0.5, H_, dtype=torch.float32, device=device),
                torch.linspace(0.5, W_ - 0.5, W_, dtype=torch.float32, device=device),
            )
            ref_y = ref_y.reshape(-1)[None] / (valid_ratios[:, None, lvl, 1] * H_)
            ref_x = ref_x.reshape(-1)[None] / (valid_ratios[:, None, lvl, 0] * W_)
            ref = torch.stack((ref_x, ref_y), -1)
            reference_points_list.append(ref)
        reference_points = torch.cat(reference_points_list, 1)
        reference_points = reference_points[:, :, None] * valid_ratios[:, None]
        return reference_points

    def forward(
        self,
        # for images
        src: Tensor,
        pos: Tensor,
        spatial_shapes: Tensor,
        level_start_index: Tensor,
        valid_ratios: Tensor,
        key_padding_mask: Tensor,
        # for texts
        memory_text: Tensor = None,
        text_attention_mask: Tensor = None,
        pos_text: Tensor = None,
        text_self_attention_masks: Tensor = None,
        position_ids: Tensor = None,
    ):
        """
        Input:
            - src: [bs, sum(hi*wi), 256]
            - pos: pos embed for src. [bs, sum(hi*wi), 256]
            - spatial_shapes: h,w of each level [num_level, 2]
            - level_start_index: [num_level] start point of level in sum(hi*wi).
            - valid_ratios: [bs, num_level, 2]
            - key_padding_mask: [bs, sum(hi*wi)]

            - memory_text: bs, n_text, 256
            - text_attention_mask: bs, n_text
                False for no padding; True for padding
            - pos_text: bs, n_text, 256

            - position_ids: bs, n_text
        Intermedia:
            - reference_points: [bs, sum(hi*wi), num_level, 2]
        Outpus:
            - output: [bs, sum(hi*wi), 256]
        """

        output = src

        # preparation and reshape
        if self.num_layers > 0:
            reference_points = self.get_reference_points(
                spatial_shapes, valid_ratios, device=src.device
            )

        if self.text_layers:
            # generate pos_text
            bs, n_text, text_dim = memory_text.shape
            if pos_text is None and position_ids is None:
                pos_text = (
                    torch.arange(n_text, device=memory_text.device)
                    .float()
                    .unsqueeze(0)
                    .unsqueeze(-1)
                    .repeat(bs, 1, 1)
                )
                pos_text = get_sine_pos_embed(pos_text, num_pos_feats=256, exchange_xy=False)
            if position_ids is not None:
                pos_text = get_sine_pos_embed(
                    position_ids[..., None], num_pos_feats=256, exchange_xy=False
                )

        # main process
        for layer_id, layer in enumerate(self.layers):
            # if output.isnan().any() or memory_text.isnan().any():
            #     if os.environ.get('IPDB_SHILONG_DEBUG', None) == 'INFO':
            #         import ipdb; ipdb.set_trace()
            if self.fusion_layers:
                if self.use_checkpoint:
                    output, memory_text = checkpoint.checkpoint(
                        self.fusion_layers[layer_id],
                        output,
                        memory_text,
                        key_padding_mask,
                        text_attention_mask,
                    )
                else:
                    output, memory_text = self.fusion_layers[layer_id](
                        v=output,
                        l=memory_text,
                        attention_mask_v=key_padding_mask,
                        attention_mask_l=text_attention_mask,
                    )

            if self.text_layers:
                memory_text = self.text_layers[layer_id](
                    src=memory_text.transpose(0, 1),
                    src_mask=~text_self_attention_masks,  # note we use ~ for mask here
                    src_key_padding_mask=text_attention_mask,
                    pos=(pos_text.transpose(0, 1) if pos_text is not None else None),
                ).transpose(0, 1)

            # main process
            if self.use_transformer_ckpt:
                output = checkpoint.checkpoint(
                    layer,
                    output,
                    pos,
                    reference_points,
                    spatial_shapes,
                    level_start_index,
                    key_padding_mask,
                )
            else:
                output = layer(
                    src=output,
                    pos=pos,
                    reference_points=reference_points,
                    spatial_shapes=spatial_shapes,
                    level_start_index=level_start_index,
                    key_padding_mask=key_padding_mask,
                )

        return output, memory_text

class TransformerDecoder(nn.Module):
    def __init__(
        self,
        decoder_layer,
        num_layers,
        norm=None,
        return_intermediate=False,
        d_model=256,
        query_dim=4,
        num_feature_levels=1,
    ):
        super().__init__()
        if num_layers > 0:
            self.layers = _get_clones(decoder_layer, num_layers)
        else:
            self.layers = []
        self.num_layers = num_layers
        self.norm = norm
        self.return_intermediate = return_intermediate
        assert return_intermediate, "support return_intermediate only"
        self.query_dim = query_dim
        assert query_dim in [2, 4], "query_dim should be 2/4 but {}".format(query_dim)
        self.num_feature_levels = num_feature_levels

        self.ref_point_head = MLP(query_dim // 2 * d_model, d_model, d_model, 2)
        self.query_pos_sine_scale = None

        self.query_scale = None
        self.bbox_embed = None
        self.class_embed = None

        self.d_model = d_model

        self.ref_anchor_head = None

    def forward(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        refpoints_unsigmoid: Optional[Tensor] = None,  # num_queries, bs, 2
        # for memory
        level_start_index: Optional[Tensor] = None,  # num_levels
        spatial_shapes: Optional[Tensor] = None,  # bs, num_levels, 2
        valid_ratios: Optional[Tensor] = None,
        # for text
        memory_text: Optional[Tensor] = None,
        text_attention_mask: Optional[Tensor] = None,
    ):
        """
        Input:
            - tgt: nq, bs, d_model
            - memory: hw, bs, d_model
            - pos: hw, bs, d_model
            - refpoints_unsigmoid: nq, bs, 2/4
            - valid_ratios/spatial_shapes: bs, nlevel, 2
        """
        output = tgt

        intermediate = []
        reference_points = refpoints_unsigmoid.sigmoid()
        ref_points = [reference_points]

        

        for layer_id, layer in enumerate(self.layers):

            if reference_points.shape[-1] == 4:
                reference_points_input = (
                    reference_points[:, :, None]
                    * torch.cat([valid_ratios, valid_ratios], -1)[None, :]
                )  # nq, bs, nlevel, 4
            else:
                assert reference_points.shape[-1] == 2
                reference_points_input = reference_points[:, :, None] * valid_ratios[None, :]
            query_sine_embed = gen_sineembed_for_position(
                reference_points_input[:, :, 0, :]
            )  # nq, bs, 256*2

            # conditional query
            raw_query_pos = self.ref_point_head(query_sine_embed)  # nq, bs, 256
            pos_scale = self.query_scale(output) if self.query_scale is not None else 1
            query_pos = pos_scale * raw_query_pos
            # if os.environ.get("SHILONG_AMP_INFNAN_DEBUG") == '1':
            #     if query_pos.isnan().any() | query_pos.isinf().any():
            #         import ipdb; ipdb.set_trace()

            # main process
            output = layer(
                tgt=output,
                tgt_query_pos=query_pos,
                tgt_query_sine_embed=query_sine_embed,
                tgt_key_padding_mask=tgt_key_padding_mask,
                tgt_reference_points=reference_points_input,
                memory_text=memory_text,
                text_attention_mask=text_attention_mask,
                memory=memory,
                memory_key_padding_mask=memory_key_padding_mask,
                memory_level_start_index=level_start_index,
                memory_spatial_shapes=spatial_shapes,
                memory_pos=pos,
                self_attn_mask=tgt_mask,
                cross_attn_mask=memory_mask,
            )
            if output.isnan().any() | output.isinf().any():
                print(f"output layer_id {layer_id} is nan")
                try:
                    num_nan = output.isnan().sum().item()
                    num_inf = output.isinf().sum().item()
                    print(f"num_nan {num_nan}, num_inf {num_inf}")
                except Exception as e:
                    print(e)
                    # if os.environ.get("SHILONG_AMP_INFNAN_DEBUG") == '1':
                    #     import ipdb; ipdb.set_trace()

            # iter update
            if self.bbox_embed is not None:
                # box_holder = self.bbox_embed(output)
                # box_holder[..., :self.query_dim] += inverse_sigmoid(reference_points)
                # new_reference_points = box_holder[..., :self.query_dim].sigmoid()

                reference_before_sigmoid = inverse_sigmoid(reference_points)
                delta_unsig = self.bbox_embed[layer_id](output)
                outputs_unsig = delta_unsig + reference_before_sigmoid
                new_reference_points = outputs_unsig.sigmoid()

                reference_points = new_reference_points.detach()
                # if layer_id != self.num_layers - 1:
                ref_points.append(new_reference_points)

            intermediate.append(self.norm(output))

        # import pdb;pdb.set_trace()

        return [
            [itm_out.transpose(0, 1) for itm_out in intermediate],
            [itm_refpoint.transpose(0, 1) for itm_refpoint in ref_points],
        ]

class DeformableTransformerEncoderLayer(nn.Module):
    def __init__(
        self,
        d_model=256,
        d_ffn=1024,
        dropout=0.1,
        activation="relu",
        n_levels=4,
        n_heads=8,
        n_points=4,
    ):
        super().__init__()

        # self attention
        self.self_attn = MSDeformAttn(
            embed_dim=d_model,
            num_levels=n_levels,
            num_heads=n_heads,
            num_points=n_points,
            batch_first=True,
        )
        self.dropout1 = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)

        # ffn
        self.linear1 = nn.Linear(d_model, d_ffn)
        self.activation = _get_activation_fn(activation, d_model=d_ffn)
        self.dropout2 = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ffn, d_model)
        self.dropout3 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(d_model)

    @staticmethod
    def with_pos_embed(tensor, pos):
        return tensor if pos is None else tensor + pos

    def forward_ffn(self, src):
        src2 = self.linear2(self.dropout2(self.activation(self.linear1(src))))
        src = src + self.dropout3(src2)
        src = self.norm2(src)
        return src

    def forward(
        self, src, pos, reference_points, spatial_shapes, level_start_index, key_padding_mask=None
    ):
        # self attention
        # import ipdb; ipdb.set_trace()
        src2 = self.self_attn(
            query=self.with_pos_embed(src, pos),
            reference_points=reference_points,
            value=src,
            spatial_shapes=spatial_shapes,
            level_start_index=level_start_index,
            key_padding_mask=key_padding_mask,
        )
        src = src + self.dropout1(src2)
        src = self.norm1(src)

        # ffn
        src = self.forward_ffn(src)

        return src

class DeformableTransformerDecoderLayer(nn.Module):
    def __init__(
        self,
        d_model=256,
        d_ffn=1024,
        dropout=0.1,
        activation="relu",
        n_levels=4,
        n_heads=8,
        n_points=4,
        use_text_feat_guide=False,
        use_text_cross_attention=False,
    ):
        super().__init__()

        # cross attention
        self.cross_attn = MSDeformAttn(
            embed_dim=d_model,
            num_levels=n_levels,
            num_heads=n_heads,
            num_points=n_points,
            batch_first=True,
        )
        self.dropout1 = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.norm1 = nn.LayerNorm(d_model)

        # cross attention text
        if use_text_cross_attention:
            self.ca_text = nn.MultiheadAttention(d_model, n_heads, dropout=dropout)
            self.catext_dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
            self.catext_norm = nn.LayerNorm(d_model)

        # self attention
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout)
        self.dropout2 = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.norm2 = nn.LayerNorm(d_model)

        # ffn
        self.linear1 = nn.Linear(d_model, d_ffn)
        self.activation = _get_activation_fn(activation, d_model=d_ffn, batch_dim=1)
        self.dropout3 = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.linear2 = nn.Linear(d_ffn, d_model)
        self.dropout4 = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.norm3 = nn.LayerNorm(d_model)

        self.key_aware_proj = None
        self.use_text_feat_guide = use_text_feat_guide
        assert not use_text_feat_guide
        self.use_text_cross_attention = use_text_cross_attention

    def rm_self_attn_modules(self):
        self.self_attn = None
        self.dropout2 = None
        self.norm2 = None

    @staticmethod
    def with_pos_embed(tensor, pos):
        return tensor if pos is None else tensor + pos

    def forward_ffn(self, tgt):
        with torch.cuda.amp.autocast(enabled=False):
            tgt2 = self.linear2(self.dropout3(self.activation(self.linear1(tgt))))
        tgt = tgt + self.dropout4(tgt2)
        tgt = self.norm3(tgt)
        return tgt

    def forward(
        self,
        # for tgt
        tgt: Optional[Tensor],  # nq, bs, d_model
        tgt_query_pos: Optional[Tensor] = None,  # pos for query. MLP(Sine(pos))
        tgt_query_sine_embed: Optional[Tensor] = None,  # pos for query. Sine(pos)
        tgt_key_padding_mask: Optional[Tensor] = None,
        tgt_reference_points: Optional[Tensor] = None,  # nq, bs, 4
        memory_text: Optional[Tensor] = None,  # bs, num_token, d_model
        text_attention_mask: Optional[Tensor] = None,  # bs, num_token
        # for memory
        memory: Optional[Tensor] = None,  # hw, bs, d_model
        memory_key_padding_mask: Optional[Tensor] = None,
        memory_level_start_index: Optional[Tensor] = None,  # num_levels
        memory_spatial_shapes: Optional[Tensor] = None,  # bs, num_levels, 2
        memory_pos: Optional[Tensor] = None,  # pos for memory
        # sa
        self_attn_mask: Optional[Tensor] = None,  # mask used for self-attention
        cross_attn_mask: Optional[Tensor] = None,  # mask used for cross-attention
    ):
        """
        Input:
            - tgt/tgt_query_pos: nq, bs, d_model
            -
        """
        assert cross_attn_mask is None

        # self attention
        if self.self_attn is not None:
            # import ipdb; ipdb.set_trace()
            q = k = self.with_pos_embed(tgt, tgt_query_pos)
            tgt2 = self.self_attn(q, k, tgt, attn_mask=self_attn_mask)[0]
            tgt = tgt + self.dropout2(tgt2)
            tgt = self.norm2(tgt)

        if self.use_text_cross_attention:
            tgt2 = self.ca_text(
                self.with_pos_embed(tgt, tgt_query_pos),
                memory_text.transpose(0, 1),
                memory_text.transpose(0, 1),
                key_padding_mask=text_attention_mask,
            )[0]
            tgt = tgt + self.catext_dropout(tgt2)
            tgt = self.catext_norm(tgt)

        tgt2 = self.cross_attn(
            query=self.with_pos_embed(tgt, tgt_query_pos).transpose(0, 1),
            reference_points=tgt_reference_points.transpose(0, 1).contiguous(),
            value=memory.transpose(0, 1),
            spatial_shapes=memory_spatial_shapes,
            level_start_index=memory_level_start_index,
            key_padding_mask=memory_key_padding_mask,
        ).transpose(0, 1)
        tgt = tgt + self.dropout1(tgt2)
        tgt = self.norm1(tgt)

        # ffn
        tgt = self.forward_ffn(tgt)

        return tgt

def build_transformer(args):
    return Transformer(
        d_model=args.hidden_dim,
        dropout=args.dropout,
        nhead=args.nheads,
        num_queries=args.num_queries,
        dim_feedforward=args.dim_feedforward,
        num_encoder_layers=args.enc_layers,
        num_decoder_layers=args.dec_layers,
        normalize_before=args.pre_norm,
        return_intermediate_dec=True,
        query_dim=args.query_dim,
        activation=args.transformer_activation,
        num_patterns=args.num_patterns,
        num_feature_levels=args.num_feature_levels,
        enc_n_points=args.enc_n_points,
        dec_n_points=args.dec_n_points,
        learnable_tgt_init=True,
        # two stage
        two_stage_type=args.two_stage_type,  # ['no', 'standard', 'early']
        embed_init_tgt=args.embed_init_tgt,
        use_text_enhancer=args.use_text_enhancer,
        use_fusion_layer=args.use_fusion_layer,
        use_checkpoint=args.use_checkpoint,
        use_transformer_ckpt=args.use_transformer_ckpt,
        use_text_cross_attention=args.use_text_cross_attention,
        text_dropout=args.text_dropout,
        fusion_dropout=args.fusion_dropout,
        fusion_droppath=args.fusion_droppath,
    )


# ----------------------------------------------------------------------------- models/GroundingDINO/matcher.py

class HungarianMatcher(nn.Module):
    """This class computes an assignment between the targets and the predictions of the network
    For efficiency reasons, the targets don't include the no_object. Because of this, in general,
    there are more predictions than targets. In this case, we do a 1-to-1 matching of the best predictions,
    while the others are un-matched (and thus treated as non-objects).
    """

    def __init__(self, cost_class: float = 1, cost_bbox: float = 1, cost_giou: float = 1, focal_alpha = 0.25):
        """Creates the matcher
        Params:
            cost_class: This is the relative weight of the classification error in the matching cost
            cost_bbox: This is the relative weight of the L1 error of the bounding box coordinates in the matching cost
            cost_giou: This is the relative weight of the giou loss of the bounding box in the matching cost
        """
        super().__init__()
        self.cost_class = cost_class
        self.cost_bbox = cost_bbox
        self.cost_giou = cost_giou
        assert cost_class != 0 or cost_bbox != 0 or cost_giou != 0, "all costs cant be 0"

        self.focal_alpha = focal_alpha

    @torch.no_grad()
    def forward(self, outputs, targets, label_map):
        """ Performs the matching
        Params:
            outputs: This is a dict that contains at least these entries:
                 "pred_logits": Tensor of dim [batch_size, num_queries, num_classes] with the classification logits
                 "pred_boxes": Tensor of dim [batch_size, num_queries, 4] with the predicted box coordinates
            targets: This is a list of targets (len(targets) = batch_size), where each target is a dict containing:
                 "labels": Tensor of dim [num_target_boxes] (where num_target_boxes is the number of ground-truth
                           objects in the target) containing the class labels
                 "boxes": Tensor of dim [num_target_boxes, 4] containing the target box coordinates
        Returns:
            A list of size batch_size, containing tuples of (index_i, index_j) where:
                - index_i is the indices of the selected predictions (in order)
                - index_j is the indices of the corresponding selected targets (in order)
            For each batch element, it holds:
                len(index_i) = len(index_j) = min(num_queries, num_target_boxes)
        """

        bs, num_queries = outputs["pred_logits"].shape[:2]

        # We flatten to compute the cost matrices in a batch
        out_prob = outputs["pred_logits"].flatten(0, 1).sigmoid()  # [batch_size * num_queries, num_classes]
        out_bbox = outputs["pred_boxes"].flatten(0, 1)  # [batch_size * num_queries, 4]

        # Also concat the target labels and boxes
        tgt_ids = torch.cat([v["labels"] for v in targets])
        tgt_bbox = torch.cat([v["boxes"] for v in targets])

        # Compute the classification cost.
        alpha = self.focal_alpha
        gamma = 2.0

        new_label_map=label_map[tgt_ids.cpu()]

        neg_cost_class = (1 - alpha) * (out_prob ** gamma) * (-(1 - out_prob + 1e-8).log())
        pos_cost_class = alpha * ((1 - out_prob) ** gamma) * (-(out_prob + 1e-8).log())
        new_label_map=new_label_map.to(pos_cost_class.device) 
        cost_bbox = torch.cdist(out_bbox[:, :2], tgt_bbox[:, :2], p=1)

        # cost_class=(pos_cost_class @ new_label_map.T - neg_cost_class@ new_label_map.T)
        cost_class=[]
        for idx_map in new_label_map:       
            idx_map = idx_map / idx_map.sum()
            cost_class.append(pos_cost_class @ idx_map - neg_cost_class@ idx_map)
        if cost_class:
            cost_class=torch.stack(cost_class,dim=0).T
        else:
            cost_class=torch.zeros_like(cost_bbox)
        # Compute the L1 cost between boxes
        

        # Compute the giou cost betwen boxes
        cost_giou = -generalized_box_iou(box_cxcywh_to_xyxy(out_bbox), box_cxcywh_to_xyxy(tgt_bbox))
        # import pdb;pdb.set_trace()
        # Final cost matrix
        C = self.cost_bbox * cost_bbox + self.cost_class * cost_class + self.cost_giou * cost_giou
        C = C.view(bs, num_queries, -1).cpu()
        C[torch.isnan(C)] = 0.0
        C[torch.isinf(C)] = 0.0

        sizes = [len(v["boxes"]) for v in targets]
        try:
            indices = [linear_sum_assignment(c[i]) for i, c in enumerate(C.split(sizes, -1))]
        except:
            print("warning: use SimpleMinsumMatcher")
            indices = []
            device = C.device
            for i, (c, _size) in enumerate(zip(C.split(sizes, -1), sizes)):
                weight_mat = c[i]
                idx_i = weight_mat.min(0)[1]
                idx_j = torch.arange(_size).to(device)
                indices.append((idx_i, idx_j))
        return [(torch.as_tensor(i, dtype=torch.int64), torch.as_tensor(j, dtype=torch.int64)) for i, j in indices]

def build_matcher(args):
    assert args.matcher_type in ['HungarianMatcher', 'SimpleMinsumMatcher'], "Unknown args.matcher_type: {}".format(args.matcher_type)
    if args.matcher_type == 'HungarianMatcher':
        return HungarianMatcher(
            cost_class=args.set_cost_class, cost_bbox=args.set_cost_bbox, cost_giou=args.set_cost_giou,
            focal_alpha=args.focal_alpha
        )
    else:  # vendored: SimpleMinsumMatcher is not carried (the checkpoint's config uses HungarianMatcher)
        raise NotImplementedError("Unknown args.matcher_type: {}".format(args.matcher_type))


# ----------------------------------------------------------------------------- models/GroundingDINO/groundingdino.py

class GroundingDINO(nn.Module):
    """This is the Cross-Attention Detector module that performs object detection"""

    def __init__(
        self,
        backbone,
        transformer,
        num_queries,
        aux_loss=False,
        iter_update=False,
        query_dim=2,
        num_feature_levels=1,
        nheads=8,
        # two stage
        two_stage_type="no",  # ['no', 'standard']
        dec_pred_bbox_embed_share=True,
        two_stage_class_embed_share=True,
        two_stage_bbox_embed_share=True,
        num_patterns=0,
        dn_number=100,
        dn_box_noise_scale=0.4,
        dn_label_noise_ratio=0.5,
        dn_labelbook_size=100,
        text_encoder_type="bert-base-uncased",
        sub_sentence_present=True,
        max_text_len=256,
        tokenizer=None,  # vendored: injected text tokenizer
        bert=None,  # vendored: injected BertModel
    ):
        """Initializes the model.
        Parameters:
            backbone: torch module of the backbone to be used. See backbone.py
            transformer: torch module of the transformer architecture. See transformer.py
            num_queries: number of object queries, ie detection slot. This is the maximal number of objects
                         Conditional DETR can detect in a single image. For COCO, we recommend 100 queries.
            aux_loss: True if auxiliary decoding losses (loss at each decoder layer) are to be used.
        """
        super().__init__()
        self.num_queries = num_queries
        self.transformer = transformer
        self.hidden_dim = hidden_dim = transformer.d_model
        self.num_feature_levels = num_feature_levels
        self.nheads = nheads
        self.max_text_len = 256
        self.sub_sentence_present = sub_sentence_present

        # setting query dim
        self.query_dim = query_dim
        assert query_dim == 4

        # visual exemplar cropping
        self.feature_map_proj = nn.Conv2d(
            (256 + 512 + 1024), hidden_dim, kernel_size=1
        )
        # vendored: upstream builds a 3-layer `feature_map_encoder` and a `feature_map_pos_embed` here that its
        # `combine_features` never calls (the lines are commented out upstream) and the checkpoint carries no
        # weights for; they are not constructed so the checkpoint loads strict=True

        # for dn training
        self.num_patterns = num_patterns
        self.dn_number = dn_number
        self.dn_box_noise_scale = dn_box_noise_scale
        self.dn_label_noise_ratio = dn_label_noise_ratio
        self.dn_labelbook_size = dn_labelbook_size

        # bert
        self.tokenizer = tokenizer  # vendored: the tokenizer and the BERT module are injected (built from the staged, digest-verified snapshot); upstream fetched them by name
        self.bert = bert
        self.bert.pooler.dense.weight.requires_grad_(False)
        self.bert.pooler.dense.bias.requires_grad_(False)
        self.bert = BertModelWarper(bert_model=self.bert)

        self.feat_map = nn.Linear(self.bert.config.hidden_size, self.hidden_dim, bias=True)
        nn.init.constant_(self.feat_map.bias.data, 0)
        nn.init.xavier_uniform_(self.feat_map.weight.data)
        # freeze

        # special tokens
        self.specical_tokens = self.tokenizer.convert_tokens_to_ids(["[CLS]", "[SEP]", ".", "?"])

        # prepare input projection layers
        if num_feature_levels > 1:
            num_backbone_outs = len(backbone.num_channels)
            input_proj_list = []
            for _ in range(num_backbone_outs):
                in_channels = backbone.num_channels[_]
                input_proj_list.append(
                    nn.Sequential(
                        nn.Conv2d(in_channels, hidden_dim, kernel_size=1),
                        nn.GroupNorm(32, hidden_dim),
                    )
                )
            for _ in range(num_feature_levels - num_backbone_outs):
                input_proj_list.append(
                    nn.Sequential(
                        nn.Conv2d(in_channels, hidden_dim, kernel_size=3, stride=2, padding=1),
                        nn.GroupNorm(32, hidden_dim),
                    )
                )
                in_channels = hidden_dim
            self.input_proj = nn.ModuleList(input_proj_list)
        else:
            assert two_stage_type == "no", "two_stage_type should be no if num_feature_levels=1 !!!"
            self.input_proj = nn.ModuleList(
                [
                    nn.Sequential(
                        nn.Conv2d(backbone.num_channels[-1], hidden_dim, kernel_size=1),
                        nn.GroupNorm(32, hidden_dim),
                    )
                ]
            )

        self.backbone = backbone
        self.aux_loss = aux_loss
        self.box_pred_damping = box_pred_damping = None

        self.iter_update = iter_update
        assert iter_update, "Why not iter_update?"

        # prepare pred layers
        self.dec_pred_bbox_embed_share = dec_pred_bbox_embed_share
        # prepare class & box embed
        _class_embed = ContrastiveEmbed()

        _bbox_embed = MLP(hidden_dim, hidden_dim, 4, 3)
        nn.init.constant_(_bbox_embed.layers[-1].weight.data, 0)
        nn.init.constant_(_bbox_embed.layers[-1].bias.data, 0)

        if dec_pred_bbox_embed_share:
            box_embed_layerlist = [_bbox_embed for i in range(transformer.num_decoder_layers)]
        else:
            box_embed_layerlist = [
                copy.deepcopy(_bbox_embed) for i in range(transformer.num_decoder_layers)
            ]
        class_embed_layerlist = [_class_embed for i in range(transformer.num_decoder_layers)]
        self.bbox_embed = nn.ModuleList(box_embed_layerlist)
        self.class_embed = nn.ModuleList(class_embed_layerlist)
        self.transformer.decoder.bbox_embed = self.bbox_embed
        self.transformer.decoder.class_embed = self.class_embed

        # two stage
        self.two_stage_type = two_stage_type
        assert two_stage_type in ["no", "standard"], "unknown param {} of two_stage_type".format(
            two_stage_type
        )
        if two_stage_type != "no":
            if two_stage_bbox_embed_share:
                assert dec_pred_bbox_embed_share
                self.transformer.enc_out_bbox_embed = _bbox_embed
            else:
                self.transformer.enc_out_bbox_embed = copy.deepcopy(_bbox_embed)

            if two_stage_class_embed_share:
                assert dec_pred_bbox_embed_share
                self.transformer.enc_out_class_embed = _class_embed
            else:
                self.transformer.enc_out_class_embed = copy.deepcopy(_class_embed)

            self.refpoint_embed = None

        self._reset_parameters()

    def _reset_parameters(self):
        # init input_proj
        for proj in self.input_proj:
            nn.init.xavier_uniform_(proj[0].weight, gain=1)
            nn.init.constant_(proj[0].bias, 0)

    def init_ref_points(self, use_num_queries):
        self.refpoint_embed = nn.Embedding(use_num_queries, self.query_dim)

    def add_exemplar_tokens(self, tokenized, text_dict, exemplar_tokens, labels):
        input_ids = tokenized["input_ids"]
        
        device = input_ids.device
        new_input_ids = []
        encoded_text = text_dict["encoded_text"]
        new_encoded_text = []
        text_token_mask = text_dict["text_token_mask"]
        new_text_token_mask = []
        position_ids = text_dict["position_ids"]
        text_self_attention_masks = text_dict["text_self_attention_masks"]
        
        
        for sample_ind in range(len(labels)):
            label = labels[sample_ind][0]
            exemplars = exemplar_tokens[sample_ind]
            label_count = -1
            assert len(input_ids[sample_ind]) == len(position_ids[sample_ind])
            for token_ind in range(len(input_ids[sample_ind])):
                input_id = input_ids[sample_ind][token_ind]
                if (input_id not in self.specical_tokens) and (token_ind == 0 or (input_ids[sample_ind][token_ind - 1] in self.specical_tokens)):
                    label_count += 1
                if label_count == label:
                    # Get the index where to insert the exemplar tokens.
                    ind_to_insert_exemplar = token_ind
                    while input_ids[sample_ind][ind_to_insert_exemplar] not in self.specical_tokens:
                        ind_to_insert_exemplar += 1
                    break
            
            # * token indicates exemplar.
            new_input_ids.append(torch.cat([input_ids[sample_ind][:ind_to_insert_exemplar], torch.tensor([1008] * exemplars.shape[0]).to(device), input_ids[sample_ind][ind_to_insert_exemplar:]]))
            new_encoded_text.append(torch.cat([encoded_text[sample_ind][:ind_to_insert_exemplar, :], exemplars, encoded_text[sample_ind][ind_to_insert_exemplar:, :]]))
            new_text_token_mask.append(torch.full((len(new_input_ids[sample_ind]),), True).to(device)) 

        tokenized['input_ids'] = torch.stack(new_input_ids)
        
        text_self_attention_masks, position_ids, _ = generate_masks_with_special_tokens_and_transfer_map(tokenized, self.specical_tokens, None)


        return {"encoded_text": torch.stack(new_encoded_text), 
                "text_token_mask": torch.stack(new_text_token_mask), 
                "position_ids": position_ids, 
                "text_self_attention_masks": text_self_attention_masks}

                

            

    def combine_features(self, features):
        
        (bs, c, h, w) = (features[0].decompose()[0].shape[-4], features[0].decompose()[0].shape[-3], features[0].decompose()[0].shape[-2], features[0].decompose()[0].shape[-1])
        
        x = torch.cat([
            F.interpolate(feat.decompose()[0], size=(h, w), mode='bilinear', align_corners=True)
            for feat in features
        ], dim=1)
        
        x = self.feature_map_proj(x)
        
        #pos_emb = self.feature_map_pos_embed(bs, h, w, x.device)
        
        #pos_emb = pos_emb.flatten(2).permute(2, 0, 1)
        
        #x = x.flatten(2).permute(2, 0, 1)
        
        #x = self.feature_map_encoder(x, pos_emb, src_key_padding_mask=None, src_mask=None)
        
        #x = x.permute(1, 2, 0).reshape(-1, self.hidden_dim, h, w)
        
        return x


    def forward(self, samples: NestedTensor, exemplars: List, labels, targets: List = None, **kw):
        """The forward expects a NestedTensor, which consists of:
           - samples.tensor: batched images, of shape [batch_size x 3 x H x W]
           - samples.mask: a binary mask of shape [batch_size x H x W], containing 1 on padded pixels

        It returns a dict with the following elements:
           - "pred_logits": the classification logits (including no-object) for all queries.
                            Shape= [batch_size x num_queries x num_classes]
           - "pred_boxes": The normalized boxes coordinates for all queries, represented as
                           (center_x, center_y, width, height). These values are normalized in [0, 1],
                           relative to the size of each individual image (disregarding possible padding).
                           See PostProcess for information on how to retrieve the unnormalized bounding box.
           - "aux_outputs": Optional, only returned when auxilary losses are activated. It is a list of
                            dictionnaries containing the two above keys for each decoder layer.
        """
        
        if targets is None:
            captions = kw["captions"]
        else:
            captions = [t["caption"] for t in targets]
        
        # encoder texts

        tokenized = self.tokenizer(captions, padding="longest", return_tensors="pt").to(
            samples.device
        )

        one_hot_token = tokenized

        (
            text_self_attention_masks,
            position_ids,
            cate_to_token_mask_list,
        ) = generate_masks_with_special_tokens_and_transfer_map(
            tokenized, self.specical_tokens, self.tokenizer
        )

        if text_self_attention_masks.shape[1] > self.max_text_len:
            text_self_attention_masks = text_self_attention_masks[
                :, : self.max_text_len, : self.max_text_len
            ]
            position_ids = position_ids[:, : self.max_text_len]
            tokenized["input_ids"] = tokenized["input_ids"][:, : self.max_text_len]
            tokenized["attention_mask"] = tokenized["attention_mask"][:, : self.max_text_len]
            tokenized["token_type_ids"] = tokenized["token_type_ids"][:, : self.max_text_len]

        # extract text embeddings
        if self.sub_sentence_present:
            tokenized_for_encoder = {k: v for k, v in tokenized.items() if k != "attention_mask"}
            tokenized_for_encoder["attention_mask"] = text_self_attention_masks
            tokenized_for_encoder["position_ids"] = position_ids
        else:
            tokenized_for_encoder = tokenized

        bert_output = self.bert(**tokenized_for_encoder)  # bs, 195, 768

        encoded_text = self.feat_map(bert_output["last_hidden_state"])  # bs, 195, d_model
        text_token_mask = tokenized.attention_mask.bool()  # bs, 195
        # text_token_mask: True for nomask, False for mask
        # text_self_attention_masks: True for nomask, False for mask

        if encoded_text.shape[1] > self.max_text_len:
            encoded_text = encoded_text[:, : self.max_text_len, :]
            text_token_mask = text_token_mask[:, : self.max_text_len]
            position_ids = position_ids[:, : self.max_text_len]
            text_self_attention_masks = text_self_attention_masks[
                :, : self.max_text_len, : self.max_text_len
            ]
        

        text_dict = {
            "encoded_text": encoded_text,  # bs, 195, d_model
            "text_token_mask": text_token_mask,  # bs, 195
            "position_ids": position_ids,  # bs, 195
            "text_self_attention_masks": text_self_attention_masks,  # bs, 195,195
        }


        


        if isinstance(samples, (list, torch.Tensor)):
            samples = nested_tensor_from_tensor_list(samples)
        
        features, poss = self.backbone(samples)
        combined_features = self.combine_features(features)
        
        # Get visual exemplar tokens.
        bs = len(exemplars)
        num_exemplars = exemplars[0].shape[0]
        if num_exemplars > 0:
            exemplar_tokens = roi_align(combined_features, boxes=exemplars, output_size=(1, 1), spatial_scale=(1 / 8), aligned=True).squeeze(-1).squeeze(-1).reshape(bs, num_exemplars, -1)
        else:
            exemplar_tokens = None

        if exemplar_tokens is not None:
            text_dict = self.add_exemplar_tokens(tokenized, text_dict, exemplar_tokens, labels)
        
        srcs = []
        masks = []
        for l, feat in enumerate(features):
            src, mask = feat.decompose()
            srcs.append(self.input_proj[l](src))
            masks.append(mask)
            assert mask is not None
        if self.num_feature_levels > len(srcs):
            _len_srcs = len(srcs)
            for l in range(_len_srcs, self.num_feature_levels):
                if l == _len_srcs:
                    src = self.input_proj[l](features[-1].tensors)
                else:
                    src = self.input_proj[l](srcs[-1])
                m = samples.mask
                mask = F.interpolate(m[None].float(), size=src.shape[-2:]).to(torch.bool)[0]
                pos_l = self.backbone[1](NestedTensor(src, mask)).to(src.dtype)
                srcs.append(src)
                masks.append(mask)
                poss.append(pos_l)
        
        input_query_bbox = input_query_label = attn_mask = dn_meta = None
        hs, reference, hs_enc, ref_enc, init_box_proposal = self.transformer(
            srcs, masks, input_query_bbox, poss, input_query_label, attn_mask, text_dict
        )

        
        # deformable-detr-like anchor update
        outputs_coord_list = []
        for dec_lid, (layer_ref_sig, layer_bbox_embed, layer_hs) in enumerate(
            zip(reference[:-1], self.bbox_embed, hs)
        ):
            layer_delta_unsig = layer_bbox_embed(layer_hs)
            layer_outputs_unsig = layer_delta_unsig + inverse_sigmoid(layer_ref_sig)
            layer_outputs_unsig = layer_outputs_unsig.sigmoid()
            outputs_coord_list.append(layer_outputs_unsig)
        outputs_coord_list = torch.stack(outputs_coord_list)


        outputs_class = torch.stack(
            [
                layer_cls_embed(layer_hs, text_dict)
                for layer_cls_embed, layer_hs in zip(self.class_embed, hs)
            ]
        )

        out = {"pred_logits": outputs_class[-1], "pred_boxes": outputs_coord_list[-1]}
        

        # Used to calculate losses
        bs, len_td = text_dict['text_token_mask'].shape
        out['text_mask']=torch.zeros(bs, self.max_text_len, dtype=torch.bool).to(
            samples.device
        )
        for b in range(bs):
            for j in range(len_td):
                if text_dict['text_token_mask'][b][j] == True:
                    out['text_mask'][b][j] = True

        # for intermediate outputs
        if self.aux_loss:
            out['aux_outputs'] = self._set_aux_loss(outputs_class, outputs_coord_list)
        out['token']=one_hot_token
        # # for encoder output
        if hs_enc is not None:
            # prepare intermediate outputs
            interm_coord = ref_enc[-1]
            interm_class = self.transformer.enc_out_class_embed(hs_enc[-1], text_dict)
            out['interm_outputs'] = {'pred_logits': interm_class, 'pred_boxes': interm_coord}
            out['interm_outputs_for_matching_pre'] = {'pred_logits': interm_class, 'pred_boxes': init_box_proposal}

        # outputs['pred_logits'].shape
        # torch.Size([4, 900, 256])

        # outputs['pred_boxes'].shape
        # torch.Size([4, 900, 4])

        # outputs['text_mask'].shape
        # torch.Size([256])

        # outputs['text_mask']

        # outputs['aux_outputs'][0].keys()
        # dict_keys(['pred_logits', 'pred_boxes', 'one_hot', 'text_mask'])

        # outputs['aux_outputs'][img_idx]

        # outputs['token']
        # <class 'transformers.tokenization_utils_base.BatchEncoding'>

        # outputs['interm_outputs'].keys()
        # dict_keys(['pred_logits', 'pred_boxes', 'one_hot', 'text_mask'])


        # outputs['interm_outputs_for_matching_pre'].keys()
        # dict_keys(['pred_logits', 'pred_boxes'])

        # outputs['one_hot'].shape
        # torch.Size([4, 900, 256])

        return out

    @torch.jit.unused
    def _set_aux_loss(self, outputs_class, outputs_coord):
        # this is a workaround to make torchscript happy, as torchscript
        # doesn't support dictionary with non-homogeneous values, such
        # as a dict having both a Tensor and a list.
        return [
            {"pred_logits": a, "pred_boxes": b}
            for a, b in zip(outputs_class[:-1], outputs_coord[:-1])
        ]

class SetCriterion(nn.Module):
    def __init__(self, matcher, weight_dict, focal_alpha,focal_gamma, losses):
        """ Create the criterion.
        Parameters:
            matcher: module able to compute a matching between targets and proposals
            weight_dict: dict containing as key the names of the losses and as values their relative weight.
            losses: list of all the losses to be applied. See get_loss for list of available losses.
            focal_alpha: alpha in Focal Loss
        """
        super().__init__()
        self.matcher = matcher
        self.weight_dict = weight_dict
        self.losses = losses
        self.focal_alpha = focal_alpha
        self.focal_gamma= focal_gamma

    @torch.no_grad()
    def loss_cardinality(self, outputs, targets, indices, num_boxes):
        """ Compute the cardinality error, ie the absolute error in the number of predicted non-empty boxes
        This is not really a loss, it is intended for logging purposes only. It doesn't propagate gradients
        """

        pred_logits = outputs['pred_logits']
        device = pred_logits.device
        tgt_lengths = torch.as_tensor([len(v["labels"]) for v in targets], device=device)
        # Count the number of predictions that are NOT "no-object" (which is the last class)
        card_pred = (pred_logits.argmax(-1) != pred_logits.shape[-1] - 1).sum(1)
        card_err = F.l1_loss(card_pred.float(), tgt_lengths.float())
        losses = {'cardinality_error': card_err}
        return losses

    def loss_boxes(self, outputs, targets, indices, num_boxes):
        """Compute the losses related to the bounding boxes, the L1 regression loss and the GIoU loss
           targets dicts must contain the key "boxes" containing a tensor of dim [nb_target_boxes, 4]
           The target boxes are expected in format (center_x, center_y, w, h), normalized by the image size.
        """
        assert 'pred_boxes' in outputs
        idx = self._get_src_permutation_idx(indices)
        src_boxes = outputs['pred_boxes'][idx]
        target_boxes = torch.cat([t['boxes'][i] for t, (_, i) in zip(targets, indices)], dim=0)

        loss_bbox = F.l1_loss(src_boxes[:, :2], target_boxes[:, :2], reduction='none')

        losses = {}
        losses['loss_bbox'] = loss_bbox.sum() / num_boxes

        loss_giou = 1 - torch.diag(generalized_box_iou(  # vendored: box_ops helpers are module-level here
            box_cxcywh_to_xyxy(src_boxes),
            box_cxcywh_to_xyxy(target_boxes)))
        losses['loss_giou'] = loss_giou.sum() / num_boxes

        # calculate the x,y and h,w loss
        with torch.no_grad():
            losses['loss_xy'] = loss_bbox[..., :2].sum() / num_boxes
            losses['loss_hw'] = loss_bbox[..., 2:].sum() / num_boxes


        return losses


    def token_sigmoid_binary_focal_loss(self, outputs, targets, indices, num_boxes):
        pred_logits=outputs['pred_logits']
        new_targets=outputs['one_hot'].to(pred_logits.device)
        text_mask=outputs['text_mask']

        assert (new_targets.dim() == 3)
        assert (pred_logits.dim() == 3)  # batch x from x to
        
        bs, n, _ = pred_logits.shape
        alpha=self.focal_alpha
        gamma=self.focal_gamma
        if text_mask is not None:
            # ODVG: each sample has different mask 
            text_mask = text_mask.repeat(1, pred_logits.size(1)).view(outputs['text_mask'].shape[0],-1,outputs['text_mask'].shape[1])
            pred_logits = torch.masked_select(pred_logits, text_mask)
            new_targets = torch.masked_select(new_targets, text_mask)

        new_targets=new_targets.float()
        p = torch.sigmoid(pred_logits)
        ce_loss = F.binary_cross_entropy_with_logits(pred_logits, new_targets, reduction="none")
        p_t = p * new_targets + (1 - p) * (1 - new_targets)
        loss = ce_loss * ((1 - p_t) ** gamma)

        if alpha >= 0:
            alpha_t = alpha * new_targets + (1 - alpha) * (1 - new_targets)
            loss = alpha_t * loss

        total_num_pos=0
        for batch_indices in indices:
            total_num_pos += len(batch_indices[0])
        num_pos_avg_per_gpu = max(total_num_pos , 1.0)
        loss=loss.sum()/num_pos_avg_per_gpu
        
        losses = {'loss_ce': loss}
        return losses


    def _get_src_permutation_idx(self, indices):
        # permute predictions following indices
        batch_idx = torch.cat([torch.full_like(src, i) for i, (src, _) in enumerate(indices)])
        src_idx = torch.cat([src for (src, _) in indices])
        return batch_idx, src_idx

    def _get_tgt_permutation_idx(self, indices):
        # permute targets following indices
        batch_idx = torch.cat([torch.full_like(tgt, i) for i, (_, tgt) in enumerate(indices)])
        tgt_idx = torch.cat([tgt for (_, tgt) in indices])
        return batch_idx, tgt_idx

    def get_loss(self, loss, outputs, targets, indices, num_boxes, **kwargs):
        loss_map = {
            'labels': self.token_sigmoid_binary_focal_loss,
            'cardinality': self.loss_cardinality,
            'boxes': self.loss_boxes,
        }
        assert loss in loss_map, f'do you really want to compute {loss} loss?'
        return loss_map[loss](outputs, targets, indices, num_boxes, **kwargs)

    def forward(self, outputs, targets, cat_list, caption, return_indices=False):
        """ This performs the loss computation.
        Parameters:
             outputs: dict of tensors, see the output specification of the model for the format
             targets: list of dicts, such that len(targets) == batch_size.
                      The expected keys in each dict depends on the losses applied, see each loss' doc
            
             return_indices: used for vis. if True, the layer0-5 indices will be returned as well.
        """
        device=next(iter(outputs.values())).device
        one_hot = torch.zeros(outputs['pred_logits'].size(),dtype=torch.int64) # torch.Size([bs, 900, 256])
        token = outputs['token'] 
        
        label_map_list = []
        indices = []
        for j in range(len(cat_list)): # bs
            label_map=[]
            for i in range(len(cat_list[j])):
                label_id=torch.tensor([i])
                per_label = create_positive_map_exemplar(token['input_ids'][j], label_id, [101, 102, 1012, 1029])
                label_map.append(per_label)
            label_map=torch.stack(label_map,dim=0).squeeze(1)
            
            label_map_list.append(label_map)
        for j in range(len(cat_list)): # bs
            for_match = {
                "pred_logits" : outputs['pred_logits'][j].unsqueeze(0),
                "pred_boxes" : outputs['pred_boxes'][j].unsqueeze(0)
            }
            
            inds = self.matcher(for_match, [targets[j]], label_map_list[j])
            indices.extend(inds)
        # indices : A list of size batch_size, containing tuples of (index_i, index_j) where:
        # - index_i is the indices of the selected predictions (in order)
        # - index_j is the indices of the corresponding selected targets (in order)

        # import pdb; pdb.set_trace()
        tgt_ids = [v["labels"].cpu() for v in targets]
        # len(tgt_ids) == bs
        for i in range(len(indices)):
            tgt_ids[i]=tgt_ids[i][indices[i][1]]
            one_hot[i,indices[i][0]] = label_map_list[i][tgt_ids[i]].to(torch.long)
        outputs['one_hot'] = one_hot
        if return_indices:
            indices0_copy = indices
            indices_list = []

        # Compute the average number of target boxes accross all nodes, for normalization purposes
        num_boxes_list = [len(t["labels"]) for t in targets]
        num_boxes = sum(num_boxes_list)
        num_boxes = torch.as_tensor([num_boxes], dtype=torch.float, device=device)
        num_boxes = torch.clamp(num_boxes, min=1).item()  # vendored: single process, no distributed reduction

        # Compute all the requested losses
        losses = {}
        for loss in self.losses:
            losses.update(self.get_loss(loss, outputs, targets, indices, num_boxes))

        # In case of auxiliary losses, we repeat this process with the output of each intermediate layer.
        if 'aux_outputs' in outputs:
            for idx, aux_outputs in enumerate(outputs['aux_outputs']):
                indices = []
                for j in range(len(cat_list)): # bs
                    aux_output_single = {
                        'pred_logits' : aux_outputs['pred_logits'][j].unsqueeze(0),
                        'pred_boxes': aux_outputs['pred_boxes'][j].unsqueeze(0)
                    }
                    inds = self.matcher(aux_output_single, [targets[j]], label_map_list[j])
                    indices.extend(inds)
                one_hot_aux = torch.zeros(outputs['pred_logits'].size(),dtype=torch.int64)
                tgt_ids = [v["labels"].cpu() for v in targets]
                for i in range(len(indices)):
                    tgt_ids[i]=tgt_ids[i][indices[i][1]]
                    one_hot_aux[i,indices[i][0]] = label_map_list[i][tgt_ids[i]].to(torch.long)
                aux_outputs['one_hot'] = one_hot_aux
                aux_outputs['text_mask'] = outputs['text_mask']
                if return_indices:
                    indices_list.append(indices)
                for loss in self.losses:
                    kwargs = {}
                    l_dict = self.get_loss(loss, aux_outputs, targets, indices, num_boxes, **kwargs)                
                    l_dict = {k + f'_{idx}': v for k, v in l_dict.items()}
                    losses.update(l_dict)

        # interm_outputs loss
        if 'interm_outputs' in outputs:
            interm_outputs = outputs['interm_outputs']
            indices = []
            for j in range(len(cat_list)): # bs
                interm_output_single = {
                    'pred_logits' : interm_outputs['pred_logits'][j].unsqueeze(0),
                    'pred_boxes': interm_outputs['pred_boxes'][j].unsqueeze(0)
                }
                inds = self.matcher(interm_output_single, [targets[j]], label_map_list[j])
                indices.extend(inds)
            one_hot_aux = torch.zeros(outputs['pred_logits'].size(),dtype=torch.int64)
            tgt_ids = [v["labels"].cpu() for v in targets]
            for i in range(len(indices)):
                tgt_ids[i]=tgt_ids[i][indices[i][1]]
                one_hot_aux[i,indices[i][0]] = label_map_list[i][tgt_ids[i]].to(torch.long)
            interm_outputs['one_hot'] = one_hot_aux
            interm_outputs['text_mask'] = outputs['text_mask']
            if return_indices:
                indices_list.append(indices)
            for loss in self.losses:
                kwargs = {}
                l_dict = self.get_loss(loss, interm_outputs, targets, indices, num_boxes, **kwargs)
                l_dict = {k + f'_interm': v for k, v in l_dict.items()}
                losses.update(l_dict)

        if return_indices:
            indices_list.append(indices0_copy)
            return losses, indices_list


        return losses

def build_groundingdino(args, tokenizer, bert):  # vendored: no build registry; the tokenizer and BERT are passed in
    device = torch.device(args.device)
    backbone = build_backbone(args)
    transformer = build_transformer(args)

    dn_labelbook_size = args.dn_labelbook_size
    dec_pred_bbox_embed_share = args.dec_pred_bbox_embed_share
    sub_sentence_present = args.sub_sentence_present

    model = GroundingDINO(
        backbone,
        transformer,
        num_queries=args.num_queries,
        aux_loss=args.aux_loss,
        iter_update=True,
        query_dim=4,
        num_feature_levels=args.num_feature_levels,
        nheads=args.nheads,
        dec_pred_bbox_embed_share=dec_pred_bbox_embed_share,
        two_stage_type=args.two_stage_type,
        two_stage_bbox_embed_share=args.two_stage_bbox_embed_share,
        two_stage_class_embed_share=args.two_stage_class_embed_share,
        num_patterns=args.num_patterns,
        dn_number=0,
        dn_box_noise_scale=args.dn_box_noise_scale,
        dn_label_noise_ratio=args.dn_label_noise_ratio,
        dn_labelbook_size=dn_labelbook_size,
        text_encoder_type=args.text_encoder_type,
        sub_sentence_present=sub_sentence_present,
        max_text_len=args.max_text_len,
        tokenizer=tokenizer,  # vendored
        bert=bert,  # vendored
    )



    matcher = build_matcher(args)

    # prepare weight dict
    weight_dict = {'loss_ce': args.cls_loss_coef, 'loss_bbox': args.bbox_loss_coef}
    weight_dict['loss_giou'] = args.giou_loss_coef
    clean_weight_dict_wo_dn = copy.deepcopy(weight_dict)

    

    clean_weight_dict = copy.deepcopy(weight_dict)

    # upstream to-do: this is a hack  # vendored: comment reworded (placeholder tokens are refused in the tutorial)
    if args.aux_loss:
        aux_weight_dict = {}
        for i in range(args.dec_layers - 1):
            aux_weight_dict.update({k + f'_{i}': v for k, v in clean_weight_dict.items()})
        weight_dict.update(aux_weight_dict)

    if args.two_stage_type != 'no':
        interm_weight_dict = {}
        try:
            no_interm_box_loss = args.no_interm_box_loss
        except:
            no_interm_box_loss = False
        _coeff_weight_dict = {
            'loss_ce': 1.0,
            'loss_bbox': 1.0 if not no_interm_box_loss else 0.0,
            'loss_giou': 1.0 if not no_interm_box_loss else 0.0,
        }
        try:
            interm_loss_coef = args.interm_loss_coef
        except:
            interm_loss_coef = 1.0
        interm_weight_dict.update({k + f'_interm': v * interm_loss_coef * _coeff_weight_dict[k] for k, v in clean_weight_dict_wo_dn.items()})
        weight_dict.update(interm_weight_dict)

    # losses = ['labels', 'boxes', 'cardinality']
    losses = ['labels', 'boxes']

    criterion = SetCriterion(matcher=matcher, weight_dict=weight_dict,
                             focal_alpha=args.focal_alpha, focal_gamma=args.focal_gamma,losses=losses
                             )
    criterion.to(device)
    postprocessors = None  # vendored: PostProcess (COCO-style top-k boxes) is not carried; the pipeline thresholds the sigmoid scores as upstream's inference scripts do

    return model, criterion, postprocessors

def create_positive_map_exemplar(input_ids, label, special_tokens):
    tokens_positive = torch.zeros(256, dtype=torch.float)
    count = -1
    for token_ind in range(len(input_ids)):
        input_id = input_ids[token_ind]
        if (input_id not in special_tokens) and (token_ind == 0 or (input_ids[token_ind - 1] in special_tokens)):
            count += 1
        if count == label:
            ind_to_insert_ones = token_ind
            
            while input_ids[ind_to_insert_ones] not in special_tokens:
                tokens_positive[ind_to_insert_ones] = 1
                ind_to_insert_ones += 1
            break
    return tokens_positive

**Module 4/8:** `src/countgd_pipeline/synthetic.py` (carried verbatim; see the note above)

In [ ]:
"""Synthetic counting scenes with known object boxes: the tutorial's first sample and the box-IoU benchmark.

FSC-147 annotates one point per object and three exemplar boxes per image, so it can score a count and the
placement of the predicted points but not the extent of the predicted boxes. These scenes supply what it lacks:
every counted object is drawn inside a known box. A scene is a flat background with clutter specks, 6 to 40
**targets** of one colour and shape (the category the prompt names, e.g. ``"red circle"``) and 4 to 20
**distractors** of a different colour *and* shape, which a counter must leave out. Everything is drawn with
integer rasterisation from a seeded `random.Random`, so a seed always yields the same pixels; the splits use
disjoint seed ranges, so no scene appears in two splits.
"""
# ruff: noqa: E501

from __future__ import annotations

import random
from collections.abc import Mapping
from typing import Any

from PIL import Image, ImageDraw

SYNTHETIC_SIZE = (512, 384)  # width, height; the counter resizes to a shortest side of 800 px like any image
SYNTHETIC_SHAPES = ("circle", "square", "triangle")
SYNTHETIC_COLOURS: dict[str, tuple[int, int, int]] = {
    "red": (200, 40, 40),
    "blue": (40, 70, 200),
    "green": (40, 150, 60),
    "yellow": (220, 190, 30),
}
SYNTHETIC_TARGETS = (6, 40)  # inclusive range of target objects per scene
SYNTHETIC_DISTRACTORS = (4, 20)  # inclusive range of distractor objects per scene
SYNTHETIC_RADIUS = (7, 14)  # half-side of an object's box in pixels (each object varies by +-2 around the scene's radius)
SYNTHETIC_SPECKS = 300
SYNTHETIC_SEED_BASE = {"train": 1_000, "validation": 2_000, "test": 3_000}
SYNTHETIC_SPLIT = {"train": 24, "validation": 8, "test": 12}
DEMO_SEED = 1  # the tutorial's first scene: 35 blue circles among 16 green triangles (text and text+exemplars count 35 on the CPU; exemplars alone count every shape, 51)


def _draw(draw: ImageDraw.ImageDraw, shape: str, box: list[int], fill: tuple[int, int, int]) -> None:
    x0, y0, x1, y1 = box
    if shape == "circle":
        draw.ellipse(box, fill=fill)
    elif shape == "square":
        draw.rectangle(box, fill=fill)
    else:
        draw.polygon([((x0 + x1) // 2, y0), (x1, y1), (x0, y1)], fill=fill)


def synthetic_scene(seed: int, *, n_targets: int | None = None) -> dict[str, Any]:
    """One scene as a counting record: ``{id, image, label, count, boxes, points, exemplars, distractors}``.

    `boxes` are the targets' full extents `[x0, y0, x1, y1]` (inclusive pixel bounds of the drawn shape plus one,
    so a box's width is the shape's width), `points` their centres, `exemplars` three of the target boxes chosen
    by the same seed. `n_targets` overrides the drawn target count (clipped to what fits)."""
    if isinstance(seed, bool) or not isinstance(seed, int) or seed < 0:
        raise ValueError("seed must be a non-negative int")
    rng = random.Random(seed)
    width, height = SYNTHETIC_SIZE
    image = Image.new("RGB", SYNTHETIC_SIZE, tuple(rng.randint(150, 225) for _ in range(3)))
    draw = ImageDraw.Draw(image)
    for _ in range(SYNTHETIC_SPECKS):
        draw.point((rng.randrange(width), rng.randrange(height)), fill=tuple(rng.randint(90, 250) for _ in range(3)))
    target_shape = rng.choice(SYNTHETIC_SHAPES)
    target_colour = rng.choice(sorted(SYNTHETIC_COLOURS))
    distractor_shape = rng.choice([s for s in SYNTHETIC_SHAPES if s != target_shape])
    distractor_colour = rng.choice([c for c in sorted(SYNTHETIC_COLOURS) if c != target_colour])
    wanted_targets = rng.randint(*SYNTHETIC_TARGETS) if n_targets is None else int(n_targets)
    wanted_distractors = rng.randint(*SYNTHETIC_DISTRACTORS)
    radius = rng.randint(*SYNTHETIC_RADIUS)
    placed: list[tuple[int, int, int]] = []
    targets: list[list[float]] = []
    distractors = 0
    for kind, wanted in (("target", wanted_targets), ("distractor", wanted_distractors)):
        tries = 0
        while wanted > 0 and tries < 5_000:
            tries += 1
            size = radius + rng.randint(-2, 2)
            cx, cy = rng.randint(size + 2, width - size - 3), rng.randint(size + 2, height - size - 3)
            if any((cx - px) ** 2 + (cy - py) ** 2 <= (size + ps + 3) ** 2 for px, py, ps in placed):
                continue
            placed.append((cx, cy, size))
            box = [cx - size, cy - size, cx + size, cy + size]
            if kind == "target":
                _draw(draw, target_shape, box, SYNTHETIC_COLOURS[target_colour])
                targets.append([float(box[0]), float(box[1]), float(box[2] + 1), float(box[3] + 1)])
            else:
                _draw(draw, distractor_shape, box, SYNTHETIC_COLOURS[distractor_colour])
                distractors += 1
            wanted -= 1
    exemplars = [list(b) for b in rng.sample(targets, k=min(3, len(targets)))]
    return {
        "id": f"synth-{seed:05d}",
        "image": image,
        "label": f"{target_colour} {target_shape}",
        "count": len(targets),
        "boxes": targets,
        "points": [[(b[0] + b[2]) / 2, (b[1] + b[3]) / 2] for b in targets],
        "exemplars": exemplars,
        "distractors": {"count": distractors, "label": f"{distractor_colour} {distractor_shape}"},
    }


def build_synthetic_dataset(sizes: Mapping[str, int] | None = None) -> dict[str, list[dict[str, Any]]]:
    """The seeded synthetic splits: scene `i` of split `s` is ``synthetic_scene(SYNTHETIC_SEED_BASE[s] + i)``."""
    sizes = dict(sizes or SYNTHETIC_SPLIT)
    out: dict[str, list[dict[str, Any]]] = {}
    for name, n in sizes.items():
        if name not in SYNTHETIC_SEED_BASE:
            raise ValueError(f"unknown split {name!r}; expected one of {sorted(SYNTHETIC_SEED_BASE)}")
        if isinstance(n, bool) or not isinstance(n, int) or not 0 <= n < 1_000:
            raise ValueError(f"{name}: size must be an int in 0..999")
        out[name] = [synthetic_scene(SYNTHETIC_SEED_BASE[name] + i) for i in range(n)]
    return out

**Module 5/8:** `src/countgd_pipeline/model.py` (carried verbatim; see the note above)

In [ ]:
# ruff: noqa: E501
from __future__ import annotations

import argparse
import hashlib
import json
import os
import pickletools
import zipfile
from collections.abc import Callable
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import torch
from huggingface_hub import snapshot_download
from safetensors import safe_open
from transformers import AutoTokenizer, BertConfig, BertModel

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .modeling import build_groundingdino` removed — names are kernel globals defined by the carried modules

MANIFEST_NAME = "dimer-base-manifest.json"
#: Fleet snapshot scheme (DIMER NOTEBOOK_SPEC 1.1 MOD13): the pinned files live in repository-local
#: snapshot directories named by their keys and described by committed manifests; a standalone notebook
#: carries the manifests inline and stages/verifies working-directory copies. Two snapshots here: the
#: CountGD checkpoint (the authors' Space) and the BERT tokenizer (vocabulary + configurations, no weights).
_WEIGHTS_ROOT = Path.cwd() / "weights"  # standalone rewrite (build_notebook.py): working-directory snapshots, no repository checkout
DEFAULT_WEIGHTS_DIR = _WEIGHTS_ROOT / DEFAULT_MODEL_KEY
TOKENIZER_WEIGHTS_DIR = _WEIGHTS_ROOT / TOKENIZER_KEY
DEFAULT_CACHE_DIR = Path.home() / ".cache" / "countgd-object-counting-pipeline"
_PICKLE_STRING_OPS = ("SHORT_BINUNICODE", "BINUNICODE", "UNICODE", "BINUNICODE8")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _check_manifest_files(root: Path, manifest: dict[str, Any]) -> None:
    files = manifest.get("files") or []
    if not files:
        raise RuntimeError(f"Manifest {MANIFEST_NAME} at {root} contains no files")
    for entry in files:
        rel_path = entry.get("path")
        if not rel_path:
            continue
        target = root / rel_path
        if not target.is_file():
            raise RuntimeError(f"Manifest file missing: {rel_path}")
        exp_bytes = entry.get("bytes")
        if exp_bytes is not None and target.stat().st_size != exp_bytes:
            raise RuntimeError(f"Size mismatch for {rel_path}: {target.stat().st_size} != {exp_bytes}")
        exp_sha = entry.get("sha256")
        if exp_sha is not None and _sha256(target) != exp_sha:
            raise RuntimeError(f"SHA-256 mismatch for {rel_path}")


def _refuse_unsafe(root: Path, *, allowed: tuple[str, ...] = ()) -> None:
    unsafe = sorted(p.name for p in root.iterdir() if p.is_file() and p.suffix.lower() in UNSAFE_WEIGHT_EXTENSIONS and p.name not in allowed)
    if unsafe:
        raise RuntimeError(f"Refusing unsafe weight files: {unsafe}")


# ------------------------------------------------------------------ the source checkpoint: pin, audit, convert


def verify_source(snapshot_path: str | Path) -> dict[str, Any]:
    """Assert the pinned byte count and SHA-256 of the source checkpoint `checkpoint_best_regular.pth`."""
    path = Path(snapshot_path) / SOURCE_CKPT_NAME
    if not path.is_file():
        raise RuntimeError(f"source checkpoint missing: {path}")
    size = path.stat().st_size
    if size != SOURCE_CKPT_SIZE_BYTES:
        raise RuntimeError(f"Unexpected {SOURCE_CKPT_NAME} size: {size}; expected {SOURCE_CKPT_SIZE_BYTES}")
    digest = _sha256(path)
    if digest != SOURCE_CKPT_SHA256:
        raise RuntimeError(f"Unexpected {SOURCE_CKPT_NAME} SHA-256: {digest}; expected {SOURCE_CKPT_SHA256}")
    return {"path": str(path), "bytes": size, "sha256": digest}


def audit_pickle(path: str | Path) -> dict[str, Any]:
    """Statically list every global a torch zip checkpoint's pickles would import, executing nothing, and refuse
    any name outside `PICKLE_ALLOWED_GLOBALS`. A file that is not a torch zip archive is refused outright."""
    path = Path(path)
    if not zipfile.is_zipfile(path):
        raise ValueError(f"{path.name} is not a torch zip archive; refusing to audit a bare pickle")
    found: set[str] = set()
    members: list[str] = []
    with zipfile.ZipFile(path) as archive:
        for name in archive.namelist():
            if not name.endswith(".pkl"):
                continue
            members.append(name)
            strings: list[str] = []
            for op, arg, _ in pickletools.genops(archive.read(name)):
                if op.name in _PICKLE_STRING_OPS:
                    strings.append(arg)
                elif op.name == "GLOBAL":
                    found.add(str(arg).replace(" ", "."))
                elif op.name == "STACK_GLOBAL":
                    if len(strings) < 2:
                        raise ValueError(f"{name}: STACK_GLOBAL without a preceding module and name")
                    found.add(f"{strings[-2]}.{strings[-1]}")
    if not members:
        raise ValueError(f"{path.name} holds no pickle member")
    names = sorted(found)
    violations = sorted(set(names) - set(PICKLE_ALLOWED_GLOBALS))
    if violations:
        raise ValueError(f"{path.name}: pickle names globals outside the allow-list: {violations}")
    return {"members": members, "globals": names, "violations": violations, "audit_sha256": hashlib.sha256("\n".join(names).encode()).hexdigest()}


def verify_converted(snapshot_path: str | Path) -> dict[str, Any]:
    """Assert the pinned byte count and SHA-256 of the converted `countgd.safetensors`."""
    path = Path(snapshot_path) / MODEL_FILENAME
    if not path.is_file():
        raise RuntimeError(f"Pinned checkpoint is missing {MODEL_FILENAME}")
    size = path.stat().st_size
    if size != MODEL_SIZE_BYTES:
        raise RuntimeError(f"Unexpected {MODEL_FILENAME} size: {size}; expected {MODEL_SIZE_BYTES}")
    digest = _sha256(path)
    if digest != MODEL_SHA256:
        raise RuntimeError(f"Unexpected {MODEL_FILENAME} SHA-256: {digest}; expected {MODEL_SHA256}")
    return {"path": str(path), "bytes": size, "sha256": digest}


def _tied_aliases(state: dict[str, torch.Tensor]) -> dict[str, str]:
    """Group tensors that share storage, offset, shape and stride; the lexicographically first name of each group
    is stored and the others are recorded as aliases of it."""
    groups: dict[tuple[Any, ...], list[str]] = {}
    for name, tensor in state.items():
        key = (tensor.untyped_storage().data_ptr(), tensor.storage_offset(), tuple(tensor.shape), tuple(tensor.stride()))
        groups.setdefault(key, []).append(name)
    aliases: dict[str, str] = {}
    for names in groups.values():
        canonical = min(names)
        aliases.update({name: canonical for name in names if name != canonical})
    return aliases


def convert_checkpoint(snapshot_path: str | Path) -> dict[str, Any]:
    """One-time conversion of the pinned pickle into the served safetensors file (fleet asset spec §11.2).

    Size + digest check → static pickle audit and audit-digest check → `torch.load(weights_only=True)` with only
    `argparse.Namespace` added to torch's restricted set → the `model` state dict (1,146 tensors) without the 38
    `feature_map_encoder.*` tensors → tied box-head tensors stored once, their aliases in one metadata entry →
    `countgd.safetensors` → pinned size + digest check. Deterministic: the same source bytes give the same file."""
    from safetensors.torch import save_file

    root = Path(snapshot_path)
    source = verify_source(root)
    audit = audit_pickle(root / SOURCE_CKPT_NAME)
    if audit["audit_sha256"] != PICKLE_AUDIT_SHA256:
        raise ValueError(f"pickle audit digest {audit['audit_sha256']} != pinned {PICKLE_AUDIT_SHA256}")
    with torch.serialization.safe_globals([argparse.Namespace]):
        payload = torch.load(root / SOURCE_CKPT_NAME, map_location="cpu", weights_only=True)
    if not isinstance(payload, dict) or not isinstance(payload.get(SOURCE_STATE_KEY), dict):
        raise ValueError(f"{SOURCE_CKPT_NAME} did not unpickle to a checkpoint with a '{SOURCE_STATE_KEY}' state dict")
    state = payload[SOURCE_STATE_KEY]
    if any(not isinstance(v, torch.Tensor) for v in state.values()) or len(state) != SOURCE_STATE_TENSORS:
        raise ValueError(f"{SOURCE_CKPT_NAME}: '{SOURCE_STATE_KEY}' is not {SOURCE_STATE_TENSORS} tensors")
    kept = {k: v for k, v in state.items() if not k.startswith(DROPPED_PREFIX)}
    dropped = len(state) - len(kept)
    if len(kept) != STATE_TENSORS:
        raise ValueError(f"converted state has {len(kept)} entries; expected {STATE_TENSORS}")
    aliases = _tied_aliases(kept)
    tensors = {k: v.contiguous() for k, v in kept.items() if k not in aliases}
    target = root / MODEL_FILENAME
    save_file(tensors, str(target), metadata={ALIAS_METADATA_KEY: json.dumps(aliases, sort_keys=True, separators=(",", ":"))})
    converted = verify_converted(root)
    return {
        "source": source,
        "audit": audit,
        "checkpoint": {"epoch": payload.get("epoch"), "unread_entries": sorted(k for k in payload if k != SOURCE_STATE_KEY)},
        "tensors_stored": len(tensors),
        "aliases": len(aliases),
        "dropped": dropped,
        "converted": converted,
    }


def ensure_converted(snapshot_path: str | Path) -> dict[str, Any]:
    """Return the verified converted file, converting from the pinned source first when it is absent. A converted
    file that exists but fails its digest is refused, not silently regenerated."""
    root = Path(snapshot_path)
    if (root / MODEL_FILENAME).is_file():
        return {"converted_this_run": False, **verify_converted(root)}
    report = convert_checkpoint(root)
    return {"converted_this_run": True, **report["converted"], "report": report}


# ------------------------------------------------------------------ snapshot verification


def verify_checkpoint(
    snapshot_path: str | Path,
    *,
    require_source: bool = False,
    return_manifest_verified: bool = False,
) -> Path | tuple[Path, bool]:
    """Verify a CountGD weights directory for loading: no unsafe formats other than the audited source checkpoint,
    the manifest's sizes and digests when a manifest is present (always when `require_source`), and the pinned
    byte count and SHA-256 of `countgd.safetensors` always."""
    root = Path(snapshot_path)
    if not root.is_dir():
        raise RuntimeError(f"Checkpoint directory does not exist: {root}")
    _refuse_unsafe(root, allowed=(SOURCE_CKPT_NAME,))
    manifest_path = root / MANIFEST_NAME
    manifest_verified = False
    if manifest_path.is_file() and (require_source or (root / SOURCE_CKPT_NAME).is_file()):
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        except Exception as exc:
            raise RuntimeError(f"Corrupt manifest {MANIFEST_NAME}: {exc}") from exc
        _check_manifest_files(root, manifest)
        manifest_verified = True
    elif require_source:
        raise RuntimeError(f"Missing {MANIFEST_NAME} at {root}")
    if require_source:
        verify_source(root)
    verify_converted(root)
    if return_manifest_verified:
        return root, manifest_verified
    return root


def _read_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    """Load and identity-check ``<root>/dimer-base-manifest.json``."""
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except ValueError as exc:
        raise RuntimeError(f"Corrupt manifest {MANIFEST_NAME}: {exc}") from exc
    if manifest.get("modelId") != model_id or manifest.get("revision") != revision:
        raise ValueError(f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, package pins {model_id}@{revision}; refusing")
    if not manifest.get("files"):
        raise RuntimeError(f"Manifest {MANIFEST_NAME} contains no files")
    return manifest


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Manifest-driven verification of the CountGD source snapshot; raise on the first mismatch.

    The manifest's identity must be the pinned Space revision; every entry is size- and SHA-256-checked, the
    source checkpoint is held to the package's own pins as well (the constants win over an edited manifest), and
    no unsafe file other than that checkpoint may sit in the directory. The converted file is checked by
    :func:`ensure_converted` / :func:`verify_converted`. Returns ``{"path": ..., **manifest}``."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root, MODEL_ID, MODEL_REVISION)
    _refuse_unsafe(root, allowed=(SOURCE_CKPT_NAME,))
    _check_manifest_files(root, manifest)
    verify_source(root)
    return {"path": str(root), **manifest}


def verify_tokenizer_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Manifest-driven verification of the BERT tokenizer snapshot (vocabulary and configurations; no weights)."""
    root = Path(path) if path is not None else TOKENIZER_WEIGHTS_DIR
    manifest = _read_manifest(root, TOKENIZER_MODEL_ID, TOKENIZER_REVISION)
    unsafe = sorted(p.name for p in root.iterdir() if p.is_file() and p.suffix.lower() in UNSAFE_WEIGHT_EXTENSIONS)
    if unsafe:
        raise RuntimeError(f"Refusing unsafe weight files in the tokenizer snapshot: {unsafe}")
    _check_manifest_files(root, manifest)
    for name in ("config.json", "vocab.txt", "tokenizer_config.json"):
        if not (root / name).is_file():
            raise RuntimeError(f"tokenizer snapshot is missing {name}")
    return {"path": str(root), **manifest}


def _hub_download(model_id: str, revision: str, repo_type: str = "model") -> Callable[[str, Path], None]:
    def fetch(relative_path: str, root: Path) -> None:
        from huggingface_hub import hf_hub_download

        hf_hub_download(model_id, relative_path, revision=revision, repo_type=repo_type, local_dir=str(root))

    return fetch


def _stage_missing(root: Path, model_id: str, revision: str, *, allow_download: bool, downloader: Callable[[str, Path], None] | None, repo_type: str = "model") -> list[str]:
    manifest = _read_manifest(root, model_id, revision)
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them at {revision}")
    fetch = downloader or _hub_download(model_id, revision, repo_type)
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch CountGD manifest-listed files that are absent locally from the authors' Space at the pinned revision
    (a fresh clone commits the manifest and the Space README but git-ignores the checkpoint). Returns the relative
    paths fetched; :func:`verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _stage_missing(root, MODEL_ID, MODEL_REVISION, allow_download=allow_download, downloader=downloader, repo_type=MODEL_REPO_TYPE)


def stage_missing_tokenizer_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """The same for the BERT tokenizer snapshot (all of its files are committed; a standalone notebook fetches
    them at the pinned revision)."""
    root = Path(path) if path is not None else TOKENIZER_WEIGHTS_DIR
    return _stage_missing(root, TOKENIZER_MODEL_ID, TOKENIZER_REVISION, allow_download=allow_download, downloader=downloader)


def resolve_weights_path(
    weights_path: str | Path | None = None,
    cache_dir: str | Path | None = None,
) -> tuple[Path, str]:
    """Resolve the CountGD weights directory with precedence:

    1. Explicit argument `weights_path` -> 'explicit_path'
    2. Environment variable `COUNTGD_WEIGHTS_DIR` -> 'env_var'
    3. Source checkout convention `weights/countgd` -> 'repo_offline'
       (only if pyproject.toml exists at repo root and the directory holds the converted file or the source)
    4. The pinned source checkpoint downloaded from the authors' Space into `<cache_dir>/countgd` -> 'hf_hub'
       (the caller converts it; nothing but the manifest-listed checkpoint is fetched)
    """
    if weights_path is not None:
        return Path(weights_path), "explicit_path"
    env_dir = os.environ.get("COUNTGD_WEIGHTS_DIR")
    if env_dir:
        return Path(env_dir), "env_var"
    repo_root = Path.cwd()  # standalone rewrite (build_notebook.py): no repository checkout to resolve
    if (repo_root / "pyproject.toml").is_file():
        repo_weights = repo_root / "weights" / DEFAULT_MODEL_KEY
        if (repo_weights / MODEL_FILENAME).is_file() or (repo_weights / SOURCE_CKPT_NAME).is_file():
            return repo_weights, "repo_offline"
    target = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    target = target / DEFAULT_MODEL_KEY
    target.mkdir(parents=True, exist_ok=True)
    if not (target / MODEL_FILENAME).is_file() and not (target / SOURCE_CKPT_NAME).is_file():
        _hub_download(MODEL_ID, MODEL_REVISION, MODEL_REPO_TYPE)(SOURCE_CKPT_NAME, target)
    return target, "hf_hub"


def resolve_tokenizer_path(tokenizer_path: str | Path | None = None, cache_dir: str | Path | None = None) -> tuple[Path, str]:
    """The tokenizer snapshot: explicit path, `COUNTGD_TOKENIZER_DIR`, the checkout's `weights/bert-base-uncased`, or the Hub."""
    if tokenizer_path is not None:
        return Path(tokenizer_path), "explicit_path"
    env_dir = os.environ.get("COUNTGD_TOKENIZER_DIR")
    if env_dir:
        return Path(env_dir), "env_var"
    if (TOKENIZER_WEIGHTS_DIR / "vocab.txt").is_file():
        return TOKENIZER_WEIGHTS_DIR, "repo_offline"
    hub_path = Path(
        snapshot_download(
            repo_id=TOKENIZER_MODEL_ID,
            revision=TOKENIZER_REVISION,
            allow_patterns=list(TOKENIZER_FILES),
            cache_dir=str(cache_dir) if cache_dir is not None else None,
        )
    )
    return hub_path, "hf_hub"


_resolve_weights_path = resolve_weights_path


def _resolve_device(device: str | torch.device | None) -> torch.device:
    if device is not None:
        return torch.device(device)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def model_args() -> SimpleNamespace:
    """The vendored builders read an attribute namespace of the upstream config."""
    return SimpleNamespace(device="cpu", **MODEL_CONFIG)


def load_state_dict(weight_path: str | Path) -> dict[str, torch.Tensor]:
    """Read the converted safetensors file and restore the tied box-head aliases recorded in its one metadata
    entry (safetensors stores a shared tensor once; `dec_pred_bbox_embed_share=True` ties the six decoder box
    heads and the encoder's)."""
    tensors: dict[str, torch.Tensor] = {}
    with safe_open(str(weight_path), "pt") as handle:
        metadata = handle.metadata() or {}
        for key in handle.keys():
            tensors[key] = handle.get_tensor(key)
    if set(metadata) != {ALIAS_METADATA_KEY}:
        raise RuntimeError(f"{Path(weight_path).name}: expected exactly the metadata entry {ALIAS_METADATA_KEY!r}, found {sorted(metadata)}")
    aliases = json.loads(metadata[ALIAS_METADATA_KEY])
    for alias, canonical in aliases.items():
        if canonical not in tensors:
            raise RuntimeError(f"alias {alias} names a missing tensor {canonical}")
        tensors[alias] = tensors[canonical]
    return tensors


def build_model(tokenizer_dir: str | Path) -> tuple[Any, Any, Any]:
    """The GroundingDINO/CountGD network, its criterion and the tokenizer, from the vendored code and the
    tokenizer snapshot's configuration (BERT at random init: its weights come from the checkpoint)."""
    tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_dir), local_files_only=True)
    bert = BertModel(BertConfig.from_pretrained(str(tokenizer_dir), local_files_only=True))
    model, criterion, _ = build_groundingdino(model_args(), tokenizer, bert)
    return model, criterion, tokenizer


def load_components(
    *,
    device: str | torch.device | None = None,
    cache_dir: str | Path | None = None,
    weights_path: str | Path | None = None,
    tokenizer_path: str | Path | None = None,
    manifest_verified: bool = False,
    return_metadata: bool = False,
) -> tuple[Any, Any, Any, torch.device, Path] | tuple[Any, Any, Any, torch.device, Path, dict[str, Any]]:
    """Acquire, convert when needed, verify, and load the one supported CountGD checkpoint into the vendored
    network, strict=True. Each file is hashed once: the served file here (`ensure_converted`), the source by the
    caller's :func:`verify_snapshot` when it passes `manifest_verified=True` (or by the conversion itself)."""
    candidate_path, source = resolve_weights_path(weights_path=weights_path, cache_dir=cache_dir)
    verified = Path(candidate_path)
    if not verified.is_dir():
        raise RuntimeError(f"Checkpoint directory does not exist: {verified}")
    _refuse_unsafe(verified, allowed=(SOURCE_CKPT_NAME,))
    conversion = ensure_converted(verified)
    tokenizer_dir, tokenizer_source = resolve_tokenizer_path(tokenizer_path=tokenizer_path, cache_dir=cache_dir)
    for name in ("config.json", "vocab.txt", "tokenizer_config.json"):
        if not (tokenizer_dir / name).is_file():
            raise RuntimeError(f"tokenizer snapshot at {tokenizer_dir} is missing {name}")
    target_device = _resolve_device(device)
    model, criterion, tokenizer = build_model(tokenizer_dir)
    weight_file = verified / MODEL_FILENAME
    model.load_state_dict(load_state_dict(weight_file), strict=True)
    model = model.eval().to(target_device)
    criterion = criterion.to(target_device)
    metadata: dict[str, Any] = {
        "checkpoint_path": verified,
        "checkpoint_source": source,
        "tokenizer_path": tokenizer_dir,
        "tokenizer_source": tokenizer_source,
        "manifest_verified": manifest_verified,
        "converted_this_run": conversion["converted_this_run"],
        "weight_sha256": conversion["sha256"],
        "weight_size_bytes": conversion["bytes"],
        "device": str(target_device),
    }
    if return_metadata:
        return model, criterion, tokenizer, target_device, verified, metadata
    return model, criterion, tokenizer, target_device, verified

**Module 6/8:** `src/countgd_pipeline/provenance.py` (carried verbatim; see the note above)

In [ ]:
# ruff: noqa: E501
from __future__ import annotations

import json
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules

_RUNTIME_PACKAGES = ("torch", "torchvision", "transformers", "safetensors", "scipy", "numpy", "pillow", "huggingface-hub")


def _package_version(name: str) -> str | None:
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def build_provenance(
    *,
    pipeline: Any | None = None,
    checkpoint_path: str | Path | None = None,
    include_runtime: bool = True,
) -> dict[str, Any]:
    checkpoint_source = None
    manifest_verified = False
    weight_sha256 = MODEL_SHA256
    weight_size = MODEL_SIZE_BYTES
    device_str = None
    resolved_checkpoint_path = None
    adapter: dict[str, Any] | None = None

    if pipeline is not None:
        if getattr(pipeline, "checkpoint_path", None) is not None:
            resolved_checkpoint_path = str(pipeline.checkpoint_path)
        checkpoint_source = getattr(pipeline, "checkpoint_source", None)
        manifest_verified = bool(getattr(pipeline, "manifest_verified", False))
        if getattr(pipeline, "weight_sha256", None):
            weight_sha256 = pipeline.weight_sha256
        if getattr(pipeline, "weight_size_bytes", None):
            weight_size = pipeline.weight_size_bytes
        if getattr(pipeline, "device", None) is not None:
            device_str = str(pipeline.device)
        if getattr(pipeline, "adapter", None):
            skip = ("history", "trainable_names")
            adapter = {k: v for k, v in pipeline.adapter.items() if k not in skip}

    if checkpoint_path is not None:
        resolved_checkpoint_path = str(checkpoint_path)
        if checkpoint_source is None:
            checkpoint_source = "explicit_path"

    model_record: dict[str, Any] = {
        "id": MODEL_ID,
        "repo_type": MODEL_REPO_TYPE,
        "revision": MODEL_REVISION,
        "license": MODEL_LICENSE,
        "weight_file": MODEL_FILENAME,
        "weight_sha256": weight_sha256,
        "weight_size_bytes": weight_size,
        "weight_format": "safetensors (converted once from the audited pickle below)",
        "derived_from": {"file": SOURCE_CKPT_NAME, "sha256": SOURCE_CKPT_SHA256, "bytes": SOURCE_CKPT_SIZE_BYTES, "pickle_audit_sha256": PICKLE_AUDIT_SHA256},
    }
    if checkpoint_source is not None:
        model_record["checkpoint_source"] = checkpoint_source
    if resolved_checkpoint_path is not None:
        model_record["checkpoint_path"] = resolved_checkpoint_path
    if pipeline is not None or checkpoint_path is not None:
        model_record["manifest_verified"] = manifest_verified

    inference_record: dict[str, Any] = {
        "counting": (
            "open-world counting: a text prompt and/or up to three exemplar boxes describe the object; 900 decoder "
            "queries each predict a box and per-token similarity logits; the queries whose best token score "
            "exceeds the threshold are counted and their box centres returned as points"
        ),
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "score_semantics": "sigmoid token similarities, uncalibrated; the count is a threshold decision, not a probability",
        "training_box": f"upstream trained on FSC-147 with {TARGET_BOX_PX:g} x {TARGET_BOX_PX:g} px boxes centred on the points; the GroundingDINO initialisation still yields object-sized boxes, which are scored by IoU where gold object boxes exist and read as points otherwise",
        "max_exemplars": MAX_EXEMPLARS,
        "deformable_attention": "pure-PyTorch grid_sample path (no compiled op) on CPU and CUDA alike",
    }
    if device_str is not None:
        inference_record["device"] = device_str

    provenance: dict[str, Any] = {
        "schema_version": 1,
        "model": model_record,
        "processor": {
            "resize": f"shortest side {SHORT_SIDE} px, longest at most {MAX_SIDE} px (upstream's test transform)",
            "normalization": "ImageNet mean / std",
            "tokenizer": {"id": TOKENIZER_MODEL_ID, "revision": TOKENIZER_REVISION, "license": TOKENIZER_LICENSE, "weights": "inside the CountGD checkpoint"},
        },
        "code": {"repository": UPSTREAM_REPOSITORY, "commit": UPSTREAM_COMMIT, "license": UPSTREAM_LICENSE, "carried_as": "modeling.py (vendored)"},
        "inference": inference_record,
        "adapter": adapter,
    }
    if include_runtime:
        provenance["runtime"] = {
            "python": platform.python_version(),
            "implementation": platform.python_implementation(),
            "platform": sys.platform,
            "packages": {name: _package_version(name) for name in _RUNTIME_PACKAGES},
        }
    return provenance


def write_provenance(
    path: str | Path,
    *,
    pipeline: Any | None = None,
    checkpoint_path: str | Path | None = None,
    include_runtime: bool = True,
) -> Path:
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(
        json.dumps(
            build_provenance(pipeline=pipeline, checkpoint_path=checkpoint_path, include_runtime=include_runtime),
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )
    return target

**Module 7/8:** `src/countgd_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Counting dataset contract for the CountGD row: the pinned FSC-147 test-split subset, validation, seeded
splitting, BYOD loaders and CSV export.

The default dataset is **real** and is the model's own benchmark domain: 80 photographs of the FSC-147 counting
benchmark (Ranjan et al., CVPR 2021) drawn with a fixed seed from its **test split** — 8 categories, 10 images
each, gold counts between 8 and 104 — whose categories CountGD never saw during training (the FSC-147 splits are
category-disjoint). Each image is pinned by byte size and SHA-256 of the file served by the Hub mirror
`isentropic/FSC147` at an immutable revision, and carries FSC-147's annotation: one **point** per object (the
gold count is the number of points) and the **three exemplar boxes** an annotator drew, both in the frame of
the 384-pixel-high image the mirror serves (the frame FSC-147's own tooling and CountGD's training used). Every
file is fetched at run time and refused on any byte-size or SHA-256 mismatch; the repository redistributes none
of the images. FSC-147's images were collected from the web by its authors, who state no per-image licence;
the mirror is published under MIT — the dataset card records this.

A record is ``{id, image, label, count, points, exemplars}``: a PIL image (or a path to one), the singular
category name CountGD's captions use, the gold count, the gold points (optional for BYOD; the localisation
metric needs them) and 0..3 exemplar boxes `[x0, y0, x1, y1]` in image pixels (optional; text-only counting
needs none). `validate_dataset(..., require_annotations=False)` accepts ``{id, image, label}`` records for the
inference contract.
"""
# ruff: noqa: E501  -- fleet dataset module written at the 110-column fleet width; this repo lints at 100

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .config import MAX_EXEMPLARS, MAX_TEXT_CHARS, MODEL_ID` removed — names are kernel globals defined by the carried modules

MAX_IMAGE_SIDE = 4096  # pixels; larger images are rejected before any decode-to-tensor work
MIN_IMAGE_SIDE = 32
MAX_COUNT = 2_000  # gold objects per image the contract accepts (FSC-147 goes to 3,701; a single 800 px pass is not the protocol for those)
MIN_EXEMPLAR_SIDE = 2.0  # pixels

CORPUS_NAME = "FSC-147 test-split subset (eight categories, ten images each)"
CORPUS_RELEASE = (
    "Hub mirror isentropic/FSC147 at revision 3e420cb6537e803dd6d4516623ce82a79c0317b8 (images_384_VarV2, "
    "annotation_FSC147_384.json, Train_Test_Val_FSC_147.json); images selected 2026-09-21 with seed 42 from the "
    "test split, gold count 8..120, categories with at least ten such images"
)
CORPUS_REPOSITORY = "isentropic/FSC147"
CORPUS_REVISION = "3e420cb6537e803dd6d4516623ce82a79c0317b8"
CORPUS_BASE_URL = f"https://huggingface.co/datasets/{CORPUS_REPOSITORY}/resolve/{CORPUS_REVISION}/images_384_VarV2/"
CORPUS_LICENSE = (
    "FSC-147 (Ranjan, Sharma, Nguyen and Hoai, CVPR 2021): images collected from the web by the authors, no "
    "per-image licence stated; the Hub mirror is published under MIT"
)
CORPUS_REFERENCE = "https://github.com/cvlab-stonybrook/LearningToCountEverything"
DEFAULT_CACHE_DIR = Path("weights") / "fsc147-subset"
# (id, category, file, bytes, sha256, orig_W, orig_H, ratio_w, ratio_h, count, points, exemplar_boxes)
SAMPLE_RECORDS = (
    ('apple_695', 'apple', '695.jpg', 22455, '256a39b73df01c4e24a66041950eacfea0792b141f49b5b6e90afad74c6d3c72', 800, 664, 0.57875, 0.578313, 11, [[255.344, 145.48], [231.471, 142.572], [212.257, 132.677], [190.715, 129.768], [181.982, 149.552], [173.243, 139.079], [154.029, 146.059], [142.384, 150.714], [124.339, 140.241], [134.235, 131.514], [147.628, 129.768]], [[144, 138, 160, 158], [175, 141, 190, 157], [245, 136, 265, 158]]),
    ('apple_2140', 'apple', '2140.jpg', 25577, 'b88e174b17f4b804686ee86c9ed8b67f4ffbf282ab50d9d21daf73745778ad8e', 450, 470, 0.906667, 0.817021, 21, [[79.152, 161.68], [145.012, 156.37], [142.066, 204.639], [188.269, 197.115], [213.339, 237.418], [162.221, 242.288], [103.732, 258.236], [176.474, 300.313], [280.178, 291.007], [269.362, 237.418], [239.877, 194.9], [280.668, 181.607], [334.243, 218.373], [347.027, 146.181], [289.517, 75.313], [261.99, 136.876], [206.457, 150.61], [231.028, 106.319], [168.114, 116.948], [196.13, 64.242], [112.572, 78.859]], [[135, 259, 229, 343], [120, 130, 173, 180], [292, 101, 389, 187]]),
    ('apple_2153', 'apple', '2153.jpg', 14598, 'b6d22826e71dc3edce5c57e74043c704d45ef6a14ac954ce3b7f1e35e1949c22', 450, 470, 0.906667, 0.817021, 12, [[138.094, 344.08], [21.878, 348.443], [71.373, 301.407], [34.254, 245.172], [51.471, 202.989], [119.798, 242.263], [133.253, 201.044], [204.816, 228.202], [189.212, 285.41], [262.38, 268.923], [255.925, 346.499], [6.809, 299.471]], [[8, 175, 90, 228], [22, 254, 130, 312], [95, 176, 174, 236]]),
    ('apple_2155', 'apple', '2155.jpg', 29370, 'fb8fc4cb863ff43032f32854b7e0ff8726326f245e0df202fa1975e019a71216', 1300, 1245, 0.308462, 0.308434, 18, [[2.036, 339.771], [9.069, 130.899], [63.765, 40.136], [179.555, 44.791], [268.001, 42.462], [369.827, 9.882], [399.504, 76.791], [340.153, 109.374], [222.614, 118.099], [285.456, 207.119], [382.628, 208.862], [174.901, 216.428], [64.345, 221.082], [88.201, 331.628], [201.666, 319.991], [314.551, 319.991], [397.175, 307.774], [106.866, 123.34]], [[288, 61, 394, 161], [56, 75, 160, 175], [148, 269, 254, 355]]),
    ('apple_2202', 'apple', '2202.jpg', 44204, 'cdd3523923442e0e49457ee17ada7bfc700f82fa82a78a8612c4e4c7085f4747', 450, 357, 1.075556, 1.07563, 99, [[39.699, 358.077], [18.059, 314.783], [80.193, 336.436], [138.833, 311.298], [150.008, 353.882], [216.326, 317.58], [265.2, 357.378], [291.035, 319.677], [337.8, 303.618], [381.79, 337.135], [342.694, 358.077], [441.128, 343.416], [434.847, 302.919], [464.167, 284.762], [480.924, 330.842], [480.924, 256.14], [477.428, 208.662], [420.886, 240.081], [350.373, 240.081], [350.373, 240.081], [388.071, 290.356], [295.918, 263.82], [261.005, 277.782], [199.569, 257.538], [168.851, 278.481], [103.931, 284.063], [59.951, 269.402], [16.66, 255.441], [52.971, 229.615], [97.65, 235.197], [142.328, 218.439], [182.124, 211.458], [225.404, 200.982], [252.637, 213.556], [309.878, 203.079], [376.896, 182.136], [422.274, 163.281], [466.963, 156.999], [13.875, 192.602], [61.339, 176.543], [5.496, 154.202], [28.534, 138.143], [20.156, 96.258], [32.718, 64.14], [22.942, 33.42], [19.457, 4.098], [62.737, 6.196], [76.698, 33.42], [113.009, 24.342], [106.727, 59.956], [77.397, 89.976], [131.153, 77.402], [145.813, 53.663], [149.309, 37.615], [138.134, 12.477], [182.823, 16.662], [214.928, 42.498], [191.89, 62.742], [215.627, 73.218], [173.046, 98.356], [203.764, 114.415], [168.851, 124.881], [168.851, 124.881], [108.814, 129.775], [184.21, 152.815], [125.571, 163.98], [256.821, 162.582], [256.821, 105.336], [254.724, 57.858], [247.044, 30.623], [244.958, 8.982], [271.481, 13.876], [311.976, 6.196], [347.577, 22.943], [362.236, 4.797], [382.489, 17.361], [418.789, 27.837], [457.886, 21.556], [420.187, 61.343], [465.565, 47.382], [385.275, 56.46], [373.411, 82.296], [373.411, 41.799], [357.353, 100.442], [347.577, 134.658], [306.394, 103.938], [302.898, 62.742], [310.577, 36.216], [285.442, 41.1], [391.556, 123.482], [446.011, 136.056], [436.245, 106.036], [480.225, 85.082], [478.827, 126.978], [446.011, 78.801], [390.169, 219.837], [99.736, 205.876], [76.01, 129.775], [106.028, 96.957]], [[282, 112, 333, 169], [234, 37, 274, 85], [190, 95, 238, 139]]),
    ('apple_2239', 'apple', '2239.jpg', 23195, '82c13799bbfaeaab0d496e8a51cdfb5d984fe51d4a0d5fcbfed231c763b51c93', 1300, 956, 0.401538, 0.401674, 17, [[150.473, 268.921], [89.981, 224.704], [112.085, 139.758], [172.573, 117.65], [131.279, 61.793], [199.91, 55.977], [251.672, 145.575], [265.634, 82.158], [324.375, 113.577], [354.04, 195.611], [320.303, 291.611], [267.959, 231.685], [298.203, 174.668], [228.407, 291.611], [195.256, 199.102], [79.513, 155.468], [91.824, 101.567]], [[146, 168, 231, 248], [131, 80, 211, 165], [305, 162, 389, 250]]),
    ('apple_2256', 'apple', '2256.jpg', 36703, '756afb9d7961c1ed8667c3b9edabeee740a719b428c21d0ef4c177cf8c18c192', 390, 280, 1.371795, 1.371429, 59, [[34.693, 35.493], [99.88, 15.717], [101.033, 82.039], [155.74, 45.97], [30.618, 102.405], [80.085, 149.541], [20.728, 176.297], [133.627, 195.497], [70.771, 230.414], [28.876, 220.512], [6.763, 243.785], [39.343, 306.048], [23.06, 261.833], [96.958, 328.731], [134.793, 272.297], [189.5, 310.121], [187.758, 245.541], [139.443, 323.493], [137.701, 346.766], [4.431, 346.19], [237.801, 37.824], [199.98, 15.717], [311.713, 26.194], [282.027, 92.517], [206.373, 95.424], [147.015, 128.009], [225.002, 169.893], [262.822, 142.56], [180.185, 180.96], [232.56, 210.048], [282.617, 197.829], [246.539, 257.76], [306.473, 288.014], [251.766, 322.341], [311.713, 346.19], [365.83, 320.009], [435.668, 326.99], [502.598, 339.209], [517.729, 250.19], [464.188, 262.409], [484.559, 192.59], [407.162, 204.233], [397.848, 264.741], [502.598, 292.663], [507.262, 153.024], [463.022, 123.936], [505.506, 80.297], [503.188, 20.955], [446.725, 52.375], [427.52, 6.405], [381.551, 33.175], [348.957, 90.185], [418.795, 94.258], [398.424, 134.414], [436.834, 166.985], [341.975, 162.912], [308.215, 126.843], [341.975, 230.99], [297.748, 243.785]], [[194, 133, 256, 195], [310, 128, 374, 195], [218, 224, 281, 296]]),
    ('apple_2271', 'apple', '2271.jpg', 29249, '731c241676a18ba98792e613e1df601c69d8f0c41aff7758356bcf7cae21d3a5', 460, 345, 1.113043, 1.113043, 42, [[308.269, 336.417], [258.003, 320.334], [186.613, 332.399], [292.185, 373.615], [220.806, 367.583], [159.477, 366.581], [75.019, 373.615], [80.05, 212.758], [53.905, 137.35], [56.921, 94.119], [99.15, 120.264], [124.282, 92.115], [194.66, 92.115], [171.531, 114.232], [149.415, 139.364], [113.23, 168.526], [161.48, 221.807], [198.678, 169.528], [216.776, 138.362], [230.856, 109.201], [246.94, 89.099], [293.187, 78.035], [295.201, 101.165], [279.118, 129.313], [272.072, 148.413], [259.005, 193.658], [322.349, 189.629], [335.416, 144.395], [343.452, 108.199], [343.452, 91.103], [392.715, 131.317], [392.715, 174.547], [455.057, 289.169], [476.171, 259.005], [503.307, 285.139], [505.322, 311.285], [505.322, 251.971], [500.302, 228.842], [476.171, 228.842], [494.269, 203.709], [507.336, 188.627], [504.32, 165.51]], [[125, 180, 204, 254], [293, 152, 365, 219], [187, 118, 239, 154]]),
    ('apple_6247', 'apple', '6247.jpg', 28054, 'af8d0402925bb0c23b18ece769dc9cbbc3bfea553c55826378a9212fe70f92c7', 240, 320, 1.7, 1.2, 15, [[213.928, 338.184], [209.729, 313.104], [208.08, 289.896], [213.928, 262.548], [207.332, 240.96], [204.782, 220.068], [198.271, 199.704], [191.675, 183.36], [186.728, 164.796], [181.781, 149.676], [187.476, 136.296], [190.774, 122.904], [186.728, 109.464], [198.271, 96.732], [198.271, 80.376]], [[191, 229, 222, 252], [179, 173, 207, 193], [193, 275, 228, 302]]),
    ('apple_6251', 'apple', '6251.jpg', 34199, '56f4241c8e598f1b8f16530c40195c37b6a4715468e3cc9423eb424f2211b56b', 1024, 683, 0.5625, 0.562225, 25, [[157.607, 290.474], [84.257, 270.11], [87.171, 220.072], [167.501, 218.852], [238.517, 299.143], [281.008, 329.453], [371.233, 271.212], [312.446, 268.89], [268.785, 221.18], [211.157, 170.034], [132.576, 161.308], [201.842, 125.18], [276.351, 135.125], [290.323, 91.435], [348.536, 156.597], [317.683, 192.725], [398.593, 212.51], [472.522, 236.944], [549.939, 199.708], [541.794, 166.548], [520.836, 183.943], [489.403, 120.58], [427.702, 126.4], [444.583, 167.65], [399.757, 117.612]], [[174, 99, 240, 163], [223, 177, 318, 279], [389, 114, 473, 172]]),
    ('cashew_nut_3157', 'cashew nut', '3157.jpg', 31968, '9f58380ff91bc7eafcebedc9de0f9298dfb0db72f81abab0244599f121d6d32f', 488, 800, 0.836066, 0.48, 42, [[85.429, 265.891], [152.515, 231.504], [158.802, 193.747], [241.088, 196.19], [208.456, 190.138], [186.877, 163.723], [122.525, 194.794], [254.465, 247.853], [251.831, 224.582], [232.175, 270.024], [176.034, 262.747], [141.78, 244.829], [170.858, 294.341], [286.695, 280.205], [313.55, 245.064], [280.107, 236.976], [316.894, 210.326], [192.546, 250.416], [113.906, 253.906], [113.095, 209.28], [238.053, 153.542], [261.663, 175.358], [139.247, 269.904], [316.593, 173.438], [350.337, 196.19], [336.048, 207.826], [362.192, 249.077], [308.483, 286.08], [225.387, 333.206], [194.578, 310.747], [334.326, 263.448], [285.583, 176.232], [115.528, 179.256], [212.921, 213.936], [187.078, 209.918], [111.473, 226.387], [135.493, 282.998], [280.818, 210.096], [88.974, 215.736], [105.294, 234.648], [212.511, 296.496], [197.512, 175.824]], [[216, 138, 248, 173], [134, 177, 182, 213], [154, 243, 212, 280]]),
    ('cashew_nut_3159', 'cashew nut', '3159.jpg', 37807, '593724148b2b6b1c455f2d7c4a4aef21160cd55852954220410c75544598ca1d', 800, 532, 0.72125, 0.721805, 58, [[222.794, 296.301], [134.564, 326.977], [176.432, 311.264], [112.876, 268.612], [194.377, 264.866], [159.98, 248.409], [127.834, 240.173], [154.751, 221.471], [109.14, 226.704], [94.181, 194.526], [104.653, 166.838], [158.487, 137.655], [176.432, 176.568], [191.391, 127.182], [231.766, 178.069], [226.53, 220.721], [209.336, 203.506], [257.933, 259.633], [252.704, 314.259], [305.045, 270.857], [378.324, 257.388], [337.199, 248.409], [331.963, 227.455], [299.809, 202.762], [276.628, 196.028], [342.428, 199.016], [406.735, 227.455], [385.797, 207.995], [426.179, 193.032], [465.055, 207.995], [459.826, 170.584], [433.652, 154.12], [367.852, 172.078], [340.935, 163.85], [319.997, 157.115], [284.108, 136.912], [231.016, 134.667], [264.663, 78.54], [220.551, 103.232], [294.573, 92.759], [328.969, 69.56], [366.359, 71.805], [412.714, 80.041], [435.895, 103.232], [391.783, 108.473], [348.414, 97.249], [339.442, 121.942], [325.233, 126.431], [371.595, 136.161], [400.005, 145.14], [423.186, 142.145], [461.319, 121.191], [487.493, 115.958], [503.937, 154.87], [477.77, 160.854], [506.931, 189.293], [521.139, 142.896], [191.391, 353.915]], [[123, 200, 191, 237], [192, 201, 281, 250], [238, 111, 307, 158]]),
    ('cashew_nut_3644', 'cashew nut', '3644.jpg', 15316, '8714b6e17e342a79af0adb67bfee08e6102cf56bd3713808d948e94de81051d8', 1000, 665, 0.577, 0.577444, 13, [[117.01, 229.338], [148.987, 140.896], [60.037, 139.736], [268.167, 169.41], [337.355, 155.442], [408.279, 135.081], [342.582, 87.373], [286.775, 107.15], [229.796, 117.042], [209.451, 59.442], [257.123, 268.898], [397.813, 225.844], [503.04, 263.078]], [[55, 200, 197, 275], [310, 120, 395, 181], [180, 38, 236, 86]]),
    ('cashew_nut_3646', 'cashew nut', '3646.jpg', 29785, 'adb56dbda4a0ccd6de83095e4d0d9c301fccb35128af2853511c6f8a417e5feb', 260, 350, 1.569231, 1.097143, 15, [[124.487, 253.67], [283.215, 189.992], [348.463, 173.162], [352.261, 213.021], [388.369, 208.15], [401.048, 176.267], [376.961, 138.174], [391.539, 231.19], [372.535, 264.85], [297.15, 225.43], [262.313, 255.985], [219.237, 243.588], [210.999, 279.014], [293.98, 292.751], [338.326, 321.09]], [[332, 180, 378, 236], [174, 263, 260, 307], [248, 161, 306, 204]]),
    ('cashew_nut_3649', 'cashew nut', '3649.jpg', 40942, 'df55d04ed72e58bf1e4623a767970a942a172404fb8c242f5b1eb858d84ed49d', 1200, 800, 0.48, 0.48, 35, [[250.656, 165.686], [336.499, 214.829], [305.395, 247.176], [248.794, 230.381], [197.165, 225.403], [185.966, 165.686], [144.912, 241.574], [176.016, 289.474], [136.205, 317.462], [90.797, 307.512], [116.299, 286.363], [100.128, 232.243], [95.15, 175.018], [131.227, 168.178], [143.045, 115.925], [184.723, 104.107], [210.85, 86.069], [204.629, 56.213], [260.611, 54.965], [308.506, 64.296], [338.986, 95.4], [329.035, 128.366], [382.526, 142.051], [414.874, 169.421], [419.851, 134.587], [408.653, 253.392], [361.378, 306.888], [321.571, 312.49], [297.931, 361.627], [238.219, 361.008], [202.138, 331.147], [246.302, 291.336], [172.906, 364.114], [224.534, 203.011], [378.797, 271.435]], [[305, 168, 378, 249], [289, 97, 374, 172], [209, 133, 290, 190]]),
    ('cashew_nut_3656', 'cashew nut', '3656.jpg', 14879, 'b30993bcc95b63ec2de8b5f48a43fe9e756049745d37db5853cae42bdd206c07', 800, 533, 0.72, 0.72045, 45, [[220.529, 256.913], [251.928, 277.863], [272.282, 269.131], [297.864, 279.023], [286.819, 298.223], [317.635, 306.955], [325.195, 262.15], [347.285, 272.042], [338.566, 237.136], [370.548, 247.604], [304.258, 237.136], [308.909, 211.531], [329.846, 216.769], [268.79, 216.186], [254.837, 235.969], [224.021, 232.482], [197.273, 245.277], [213.552, 214.442], [225.18, 200.48], [189.713, 191.171], [237.391, 180.696], [253.094, 166.15], [272.858, 181.28], [300.773, 192.915], [340.891, 202.223], [344.959, 176.626], [375.775, 180.113], [381.593, 217.353], [348.451, 219.096], [294.372, 162.08], [272.858, 148.117], [241.459, 142.296], [211.226, 157.426], [201.341, 132.988], [215.878, 112.628], [265.882, 122.513], [310.658, 123.096], [320.544, 160.336], [346.709, 149.861], [381.593, 142.88], [353.102, 119.609], [328.097, 113.204], [243.209, 105.64], [272.282, 82.953], [299.023, 89.934]], [[373, 124, 398, 168], [233, 220, 275, 261], [328, 127, 372, 165]]),
    ('cashew_nut_5141', 'cashew nut', '5141.jpg', 22411, '60c0e2098aead899d61c94a7bbe436a10551b4f4d37e282f5caa08928a5a18be', 910, 1024, 0.448352, 0.375, 30, [[205.009, 62.846], [147.969, 84.953], [90.231, 153.026], [33.191, 167.572], [141.011, 141.971], [225.185, 111.716], [253.704, 90.191], [223.095, 153.026], [285.703, 123.934], [178.578, 98.917], [296.831, 169.316], [332.309, 166.41], [270.401, 218.19], [223.79, 189.683], [164.666, 184.444], [103.448, 180.953], [111.102, 214.118], [46.409, 207.716], [63.8, 261.825], [104.143, 256.59], [182.053, 213.536], [147.275, 266.483], [210.577, 279.281], [232.833, 247.864], [288.487, 281.61], [314.918, 244.954], [378.915, 208.882], [128.493, 239.134], [191.791, 253.68], [337.873, 296.153]], [[122, 54, 181, 125], [244, 175, 289, 246], [187, 91, 259, 141]]),
    ('cashew_nut_5797', 'cashew nut', '5797.jpg', 25552, 'eefe516f70c048d6ad964441bc35c0ddc4ac91b02a37b96656d0b86ede629bd7', 470, 400, 0.959574, 0.96, 58, [[17.56, 170.477], [48.506, 183.619], [87.523, 176.179], [117.183, 171.754], [148.993, 166.339], [115.091, 129.168], [76.305, 113.05], [63.793, 289.747], [88.166, 252.979], [122.653, 251.059], [145.97, 263.683], [166.966, 242.035], [144.522, 210.499], [181.273, 190.666], [203.89, 232.723], [199.006, 273.514], [151.382, 303.014], [124.687, 310.694], [190.111, 314.707], [208.544, 364.397], [289.964, 348.221], [239.136, 310.166], [315.383, 325.814], [331.024, 231.792], [308.167, 269.558], [285.608, 237.907], [242.513, 257.05], [225.702, 265.488], [254.661, 222.374], [218.956, 203.866], [393.31, 323.376], [354.927, 313.488], [343.873, 282.067], [384.003, 226.79], [415.995, 262.867], [446.816, 202.358], [350.273, 199.45], [291.538, 188.39], [261.292, 172.099], [162.024, 95.126], [126.894, 98.794], [195.168, 136.378], [185.457, 163.843], [231.517, 142.838], [264.612, 127.066], [277.931, 104.438], [231.517, 82.618], [192.731, 108.854], [284.792, 66.854], [328.405, 90.701], [359.927, 119.386], [388.599, 158.957], [314.683, 154.934], [123.411, 51.552], [144.752, 4.483], [208.727, 31.651], [186.388, 64.637], [245.536, 2.093]], [[237, 113, 294, 148], [297, 123, 346, 172], [163, 112, 222, 154]]),
    ('cashew_nut_5800', 'cashew nut', '5800.jpg', 27466, '7b64d513eb101d123e7b88f44882a52495533d0132d7b2daf8cbcaf83af0f48b', 680, 453, 0.847059, 0.847682, 12, [[113.557, 15.258], [211.231, 47.835], [114.141, 100.781], [74.024, 150.235], [374.019, 204.927], [403.674, 256.127], [453.668, 186.888], [345.532, 246.235], [353.088, 307.327], [313.556, 282.888], [380.999, 346.312], [524.6, 348.635]], [[181, 9, 259, 70], [376, 216, 446, 283], [415, 160, 492, 208]]),
    ('cashew_nut_5804', 'cashew nut', '5804.jpg', 23404, 'c2d477338cf941d07ca784a7077b112a04cab40caf59338ad975d408dbab8358', 2048, 3648, 0.199219, 0.105263, 43, [[185.768, 143.517], [203.026, 155.441], [232.674, 152.869], [241.083, 141.646], [281.351, 169.237], [263.65, 163.625], [300.38, 174.848], [260.11, 180.226], [263.208, 191.683], [239.754, 175.549], [207.45, 172.744], [216.3, 196.125], [189.75, 185.371], [166.296, 190.982], [153.904, 179.993], [179.572, 165.729], [184.439, 154.974], [139.302, 159.182], [152.135, 166.197], [116.732, 182.798], [126.91, 196.594], [183.554, 205.245], [143.284, 217.637], [127.797, 210.388], [163.64, 230.497], [197.272, 219.274], [199.486, 239.148], [233.558, 229.095], [233.116, 237.98], [203.91, 230.732], [284.891, 210.622], [270.73, 227.925], [300.822, 197.528], [270.73, 202.205], [240.64, 207.349], [397.734, 243.357], [387.556, 266.739], [292.858, 300.409], [318.523, 333.144], [368.971, 310.463], [387.114, 323.792], [401.725, 339.926], [236.214, 360.735]], [[187, 165, 225, 183], [183, 185, 234, 203], [221, 158, 249, 186]]),
    ('egg_5919', 'egg', '5919.jpg', 40178, '6cf485e523ffad8ac8b14dbd9ccd3892daa6d219a7445b5cfddefedf6b6cccbc', 640, 427, 0.9, 0.899297, 39, [[240.255, 354.917], [196.002, 333.972], [230.355, 303.135], [294.993, 334.557], [346.23, 308.954], [290.331, 276.372], [371.853, 249.609], [387.567, 318.846], [432.405, 286.264], [437.067, 231.578], [418.428, 178.043], [386.406, 213.538], [328.761, 236.812], [277.524, 221.101], [235.017, 250.778], [183.195, 290.338], [179.118, 239.141], [201.24, 185.03], [239.67, 194.923], [253.062, 153.609], [281.016, 185.03], [313.038, 179.212], [369.522, 160.012], [346.815, 193.178], [375.345, 190.849], [391.068, 121.612], [350.892, 111.72], [335.745, 75.064], [287.424, 57.609], [304.308, 101.827], [315.954, 136.738], [267.039, 114.624], [247.824, 78.554], [205.317, 67.501], [201.825, 101.827], [164.556, 97.754], [155.241, 160.012], [193.095, 143.717], [233.271, 114.049]], [[213, 126, 287, 180], [340, 133, 391, 185], [342, 220, 406, 285]]),
    ('egg_5924', 'egg', '5924.jpg', 18649, 'b189e62cb451f3de75d6c36ade2d5bff8577066d2261a25068c3f02ffead3959', 376, 375, 1.023936, 1.024, 16, [[145.614, 143.708], [156.672, 43.172], [58.149, 47.196], [46.087, 138.68], [40.056, 246.262], [27.984, 355.85], [139.583, 355.85], [138.569, 250.276], [254.182, 250.276], [247.147, 355.85], [365.781, 362.885], [373.819, 250.276], [378.846, 137.677], [370.798, 35.133], [260.213, 41.165], [259.209, 144.712]], [[205, 197, 314, 306], [8, 90, 99, 186], [101, 93, 204, 196]]),
    ('egg_5925', 'egg', '5925.jpg', 18845, 'eaf99a9d00e4c855c993e0381be5e6c62b805bc342e0d035a40e605c06ca3513', 375, 500, 1.088, 0.768, 22, [[387.72, 331.638], [231.45, 327.444], [68.25, 295.334], [235.4, 210.156], [367.94, 219.924], [97.92, 183.621], [4.95, 256.934], [9.89, 177.339], [374.87, 118.694], [253.21, 127.764], [144.41, 119.386], [30.66, 121.482], [47.48, 78.897], [140.45, 76.101], [246.28, 80.294], [350.14, 71.916], [338.27, 23.04], [239.36, 34.214], [152.32, 14.661], [72.2, 20.943], [230.46, 3.494], [292.77, 5.583]], [[32, 140, 175, 232], [150, 250, 327, 373], [188, 88, 309, 168]]),
    ('egg_5931', 'egg', '5931.jpg', 22627, 'a8b7ffb1314ddaa7f6203dbded0ea187272b27090180ac89d00274d2d5176b4e', 1600, 1066, 0.36, 0.360225, 9, [[392.486, 111.832], [84.582, 203.704], [197.791, 119.674], [303.534, 190.631], [296.694, 62.413], [523.735, 188.765], [434.16, 270.302], [369.472, 343.125], [201.521, 290.842]], [[336, 56, 448, 181], [244, 115, 377, 255], [127, 45, 267, 191]]),
    ('egg_5938', 'egg', '5938.jpg', 25357, '06198b66baef67c79d79a6ddb1b356b00d2276bf0cef00300dc382a1a0aca11d', 3000, 1996, 0.192333, 0.192385, 31, [[54.238, 10.774], [132.518, 5.194], [64.816, 61.756], [157.713, 46.942], [236.57, 40.401], [307.926, 27.511], [400.63, 19.046], [502.759, 19.046], [533.34, 67.335], [541.611, 120.24], [439.674, 85.034], [333.891, 100.81], [256.957, 101.579], [141.942, 121.972], [75.01, 133.13], [68.855, 216.433], [159.252, 201.042], [259.265, 191.423], [354.855, 174.685], [452.176, 153.331], [555.074, 216.048], [512.376, 328.401], [481.603, 238.749], [380.051, 254.91], [290.616, 266.838], [174.446, 292.232], [82.319, 305.699], [78.28, 373.804], [178.485, 375.15], [277.345, 364.377], [401.592, 360.337]], [[387, 39, 465, 118], [221, 143, 306, 227], [325, 207, 422, 307]]),
    ('egg_6733', 'egg', '6733.jpg', 33424, '80ef7d5744b59ec82eb27d781fa92048753dd1325d0352c34f0503d5e74ca511', 940, 540, 0.710638, 0.711111, 24, [[304.892, 216.263], [268.778, 177.913], [486.666, 108.857], [267.044, 362.354], [593.085, 224.249], [423.142, 39.858], [341.05, 15.744], [417.706, 109.966], [43.242, 31.097], [589.574, 21.234], [450.552, 186.667], [342.158, 96.832], [466.612, 358.99], [251.275, 12.473], [321.336, 267.755], [211.841, 69.447], [81.425, 264.676], [272.047, 93.554], [405.632, 251.292], [379.381, 184.455], [173.517, 12.473], [191.062, 139.563], [222.806, 219.534], [151.636, 88.071]], [[335, 138, 425, 219], [544, 187, 630, 261], [37, 230, 131, 308]]),
    ('egg_6804', 'egg', '6804.jpg', 23110, 'c243bbbcf91a76190a8253b6967c4b168702a0cee31d46652f72e9ee91f5dd8c', 1500, 1048, 0.366667, 0.366412, 12, [[467.995, 246.053], [92.994, 163.248], [292.901, 258.687], [144.027, 226.871], [451.139, 300.322], [60.691, 210.031], [316.778, 213.307], [387.937, 222.196], [235.316, 191.318], [151.048, 181.026], [364.529, 276.465], [210.503, 242.781]], [[176, 216, 255, 286], [326, 246, 405, 316], [434, 219, 499, 281]]),
    ('egg_6859', 'egg', '6859.jpg', 25133, '7736362b602664dc6d09d34e8efe6bedffd3a65ae3e7401ba976281897d586b0', 300, 225, 1.706667, 1.706667, 24, [[254.669, 72.499], [328.09, 340.617], [223.676, 312.337], [426.445, 195.789], [289.69, 23.996], [232.431, 190.396], [278.921, 136.499], [378.624, 257.758], [458.786, 105.506], [136.772, 93.389], [484.386, 163.447], [156.979, 235.537], [148.224, 157.389], [385.365, 126.396], [327.424, 92.041], [202.786, 123.017], [405.572, 73.847], [293.734, 251.699], [344.269, 185.003], [231.083, 27.358], [73.438, 134.485], [344.269, 40.841], [187.29, 54.989], [87.586, 211.285]], [[169, 89, 246, 166], [334, 95, 417, 163], [251, 213, 339, 306]]),
    ('egg_6928', 'egg', '6928.jpg', 14650, 'a98aaa9d129005c54640fa02d1ea59aaa34f36d568d279db81f2da3092470f7a', 680, 680, 0.564706, 0.564706, 30, [[64.23, 148.907], [168.328, 145.4], [264.819, 88.088], [279.433, 146.569], [312.186, 86.92], [219.202, 186.336], [223.296, 228.44], [119.785, 112.066], [39.671, 226.103], [272.42, 116.16], [222.127, 151.251], [171.834, 110.315], [226.221, 88.088], [115.691, 144.813], [108.678, 185.167], [316.28, 118.498], [166.571, 183.998], [161.89, 228.44], [283.528, 234.291], [70.08, 117.329], [82.944, 85.169], [282.359, 191.017], [326.219, 151.251], [330.901, 186.336], [220.958, 117.916], [100.489, 226.103], [340.258, 233.122], [115.691, 88.088], [49.609, 186.336], [171.247, 85.751]], [[143, 129, 195, 172], [134, 202, 191, 265], [253, 163, 311, 211]]),
    ('egg_7605', 'egg', '7605.jpg', 22146, '8b8703478ec77403da3d728395ff294e5e6fe9b74763c7f72fa7fc30f4be85a7', 300, 199, 1.93, 1.929648, 20, [[477.559, 114.197], [19.454, 130.985], [94.956, 95.942], [241.25, 35.428], [177.155, 66.978], [332.809, 118.172], [181.864, 186.906], [361.084, 250.217], [566.841, 142.254], [517.317, 177.74], [343.579, 179.496], [393.72, 77.533], [270.142, 222.604], [314.165, 61.556], [420.624, 147.329], [94.956, 161.28], [447.528, 214.384], [251.267, 145.804], [255.281, 91.234], [171.056, 122.224]], [[135, 157, 233, 263], [314, 224, 422, 339], [291, 93, 380, 151]]),
    ('finger_food_3268', 'finger food', '3268.jpg', 34129, '6335c031a57a059ce33839918d8ba828398e1f4041df1ba57cfce1f3a73d88df', 800, 600, 0.64, 0.64, 19, [[142.874, 69.574], [132.25, 104.634], [130.125, 145.53], [106.752, 204.48], [92.947, 293.709], [195.981, 288.397], [220.416, 226.79], [226.79, 150.304], [224.134, 108.877], [219.354, 53.645], [301.146, 58.426], [304.333, 81.792], [304.333, 132.781], [330.886, 202.886], [328.762, 281.491], [429.677, 275.123], [419.053, 219.354], [400.998, 156.678], [391.968, 102.509]], [[159, 258, 255, 339], [195, 118, 269, 177], [292, 242, 394, 332]]),
    ('finger_food_3276', 'finger food', '3276.jpg', 22309, 'b94f24782e09ea2396953ee30334eeee5c1e2cb721016ba43dce4432a0222da1', 800, 600, 0.64, 0.64, 18, [[20.653, 116.947], [16.0, 166.4], [13.088, 239.13], [91.053, 191.418], [84.653, 103.565], [171.347, 106.47], [184.73, 175.13], [115.488, 247.853], [248.147, 253.67], [372.653, 256.582], [482.035, 258.33], [505.312, 174.547], [509.382, 114.035], [447.13, 119.27], [354.618, 123.347], [260.947, 123.347], [306.912, 183.853], [418.618, 183.27]], [[186, 196, 301, 310], [317, 198, 426, 290], [133, 137, 226, 210]]),
    ('finger_food_3279', 'finger food', '3279.jpg', 27040, 'fcc060dfca343e78cad47310e67903040364c18fa2a7f09508bcb56a40ce1bd1', 799, 533, 0.720901, 0.72045, 23, [[353.227, 337.473], [435.316, 303.727], [520.894, 260.09], [495.857, 197.252], [437.061, 217.036], [367.198, 245.544], [260.656, 284.527], [187.305, 235.652], [127.34, 202.49], [62.718, 173.398], [20.214, 158.852], [92.405, 137.909], [137.815, 150.704], [211.174, 176.309], [281.62, 208.887], [357.884, 180.963], [417.265, 161.179], [355.556, 136.741], [290.934, 156.525], [216.415, 134.414], [171.582, 115.214], [228.057, 101.252], [292.095, 112.887]], [[262, 129, 310, 180], [215, 256, 300, 317], [333, 215, 399, 279]]),
    ('finger_food_3283', 'finger food', '3283.jpg', 31890, 'df19a4c9c866326998138f6d9d2169ca0ba3c9a4ae6e489cc09d52c5973dcadb', 800, 600, 0.64, 0.64, 37, [[146.618, 109.965], [185.018, 125.088], [268.8, 137.312], [318.253, 148.365], [373.53, 164.653], [428.218, 170.47], [479.418, 190.835], [435.2, 204.8], [379.347, 197.235], [328.147, 192.582], [287.418, 173.382], [230.982, 160.582], [153.018, 162.33], [132.07, 141.382], [103.565, 132.07], [90.765, 160.582], [64.582, 185.018], [29.088, 212.365], [5.818, 254.253], [63.418, 272.288], [94.253, 233.888], [116.365, 195.488], [145.453, 201.888], [153.6, 294.4], [186.182, 236.218], [235.635, 247.27], [211.782, 200.147], [275.782, 211.782], [342.112, 237.965], [308.947, 281.018], [239.712, 300.8], [360.73, 338.035], [385.165, 292.653], [412.512, 247.853], [479.418, 259.488], [446.835, 310.112], [439.853, 363.635]], [[253, 125, 283, 151], [164, 114, 202, 136], [333, 317, 389, 365]]),
    ('finger_food_4112', 'finger food', '4112.jpg', 39750, '5a21dbf9e79ce2f0331c77ae45dbfb16c775ef4587658e4eaa73ca6a0685e42c', 2000, 1348, 0.285, 0.284866, 11, [[372.669, 118.804], [322.378, 47.59], [217.6, 90.317], [137.133, 130.534], [49.12, 176.614], [40.738, 262.071], [158.927, 212.639], [179.043, 317.367], [314.834, 272.962], [433.02, 191.695], [259.512, 164.046]], [[124, 254, 238, 360], [271, 0, 373, 81], [325, 68, 427, 157]]),
    ('finger_food_4113', 'finger food', '4113.jpg', 47560, '9dfb6d9bf3400a44d00d581514f5e0483aaa44afc04ce5cedaca92e8fd22a2c1', 1500, 1101, 0.348667, 0.348774, 37, [[21.262, 110.635], [72.446, 96.673], [118.397, 85.035], [170.742, 67.582], [217.854, 55.943], [262.061, 45.473], [296.377, 30.343], [330.111, 22.782], [363.848, 11.726], [403.979, 8.235], [45.107, 178.708], [95.712, 160.091], [152.13, 140.308], [205.058, 121.691], [248.101, 110.635], [295.795, 98.999], [325.459, 86.782], [380.134, 68.743], [414.45, 57.691], [447.022, 43.726], [494.134, 105.982], [461.561, 115.291], [422.591, 129.252], [388.857, 143.799], [342.328, 164.161], [291.722, 178.708], [242.285, 199.652], [202.15, 213.035], [140.499, 236.308], [78.844, 251.435], [166.091, 306.126], [93.966, 336.961], [223.091, 300.891], [284.746, 283.435], [331.275, 264.235], [381.877, 243.291], [432.479, 221.182]], [[40, 72, 107, 126], [150, 40, 196, 97], [122, 113, 174, 173]]),
    ('finger_food_4120', 'finger food', '4120.jpg', 43285, '1c10434d36b315682b1a0c313eb1d90f48e22a559895669e448417988de23024', 390, 280, 1.371795, 1.371429, 46, [[425.366, 3.291], [101.54, 237.12], [98.742, 301.591], [145.273, 340.827], [153.696, 285.888], [191.818, 298.217], [213.122, 345.312], [252.945, 290.935], [305.087, 342.514], [354.431, 324.576], [310.698, 279.717], [361.166, 278.606], [421.155, 315.607], [428.453, 271.31], [472.186, 322.889], [481.157, 267.95], [531.625, 294.857], [489.566, 210.775], [429.015, 210.775], [382.47, 213.01], [321.343, 224.229], [265.84, 230.949], [195.741, 236.558], [146.96, 225.902], [146.398, 170.976], [145.273, 109.303], [152.571, 64.457], [205.838, 61.659], [205.838, 126.117], [201.914, 179.383], [253.494, 170.414], [262.466, 115.474], [263.028, 67.269], [318.544, 166.491], [361.715, 154.711], [430.689, 151.913], [494.614, 141.819], [427.328, 98.098], [377.422, 97.536], [321.905, 104.256], [319.107, 65.019], [368.45, 46.519], [420.044, 45.394], [378.876, 5.431], [326.227, 3.305], [266.457, 7.845]], [[230, 84, 289, 145], [296, 196, 352, 247], [172, 210, 237, 267]]),
    ('finger_food_5383', 'finger food', '5383.jpg', 26099, '8ca289084a97296549bb9c6cf065155b73baf82dca1052ee4fdbcfc581c93383', 600, 793, 0.68, 0.484237, 10, [[90.433, 281.012], [236.681, 278.683], [185.212, 218.754], [325.74, 212.938], [273.448, 157.082], [142.725, 162.321], [154.979, 109.956], [40.596, 126.827], [30.79, 182.683], [17.721, 237.954]], [[125, 143, 225, 244], [298, 115, 395, 258], [116, 23, 209, 141]]),
    ('finger_food_5388', 'finger food', '5388.jpg', 42521, 'ddbfa204142fb6d570d62c7e21a8f831e06aba56650d20ec921324766cb42f0c', 1334, 1779, 0.305847, 0.215852, 13, [[284.0, 328.142], [327.694, 299.634], [255.973, 290.908], [132.313, 232.725], [62.237, 233.308], [5.355, 196.071], [45.752, 183.271], [114.176, 192.578], [119.947, 146.034], [180.126, 189.088], [218.051, 143.708], [163.64, 115.778], [143.029, 76.215]], [[31, 211, 90, 254], [297, 275, 356, 319], [188, 123, 234, 165]]),
    ('finger_food_5568', 'finger food', '5568.jpg', 37011, 'ac9a7ecbbc5c5a37efaedf093a8069a7572ac5b0a6034537b5f79b36665994dd', 800, 508, 0.75625, 0.755906, 30, [[124.97, 134.891], [114.534, 171.621], [100.014, 214.254], [83.679, 266.857], [163.078, 267.311], [172.153, 218.789], [178.052, 174.796], [190.756, 134.884], [266.071, 128.088], [258.811, 167.992], [252.913, 220.603], [245.199, 271.393], [332.311, 269.579], [325.959, 218.789], [330.95, 179.331], [329.135, 129.902], [394.021, 128.542], [399.466, 171.621], [398.559, 217.882], [406.726, 271.393], [459.808, 118.564], [470.244, 149.851], [474.328, 180.238], [472.513, 219.243], [474.328, 261.868], [550.55, 263.683], [544.197, 219.696], [533.308, 185.681], [529.678, 153.026], [523.325, 124.006]], [[519, 241, 573, 290], [445, 165, 494, 199], [128, 236, 205, 299]]),
    ('marble_4065', 'marble', '4065.jpg', 26470, '5d67cfac0bf52c4cc7e3522ed28406b479d066afdd104897255357b5ab5e7c3c', 900, 1200, 0.453333, 0.32, 37, [[59.051, 281.299], [120.773, 312.298], [136.199, 269.571], [191.987, 301.408], [283.379, 286.326], [334.415, 269.571], [316.613, 220.138], [375.958, 194.166], [260.825, 215.949], [209.789, 244.435], [151.631, 225.165], [82.792, 240.246], [36.502, 223.491], [53.117, 177.411], [114.838, 187.466], [197.921, 195.843], [234.713, 170.707], [307.115, 182.438], [337.974, 161.491], [297.622, 138.87], [322.547, 102.006], [228.779, 139.709], [259.638, 110.384], [177.743, 132.17], [169.433, 169.869], [107.717, 152.275], [64.985, 126.304], [38.873, 105.357], [83.975, 86.925], [136.199, 110.384], [132.641, 62.63], [176.555, 94.467], [183.677, 60.954], [193.174, 33.306], [237.089, 38.333], [299.993, 65.142], [229.967, 88.602]], [[293, 199, 355, 244], [103, 99, 154, 135], [211, 117, 260, 157]]),
    ('marble_4067', 'marble', '4067.jpg', 35187, '122d882ba587dbe8c1eba853f9abc9015e68f45c2b81a6e1d79e948dcae0c089', 533, 399, 0.962477, 0.962406, 42, [[81.762, 196.793], [90.983, 143.168], [118.635, 95.413], [175.604, 78.657], [195.72, 37.601], [232.582, 71.959], [251.014, 27.554], [291.236, 35.927], [335.644, 49.333], [276.154, 72.796], [276.154, 72.796], [320.562, 95.413], [382.565, 92.064], [401.834, 136.469], [352.401, 138.981], [307.992, 144.842], [259.397, 130.598], [209.117, 122.226], [209.117, 122.226], [161.359, 134.795], [138.741, 171.655], [189.011, 168.306], [123.659, 228.629], [168.068, 218.572], [107.74, 283.919], [165.556, 278.896], [151.311, 328.325], [203.256, 311.569], [219.175, 266.327], [219.175, 214.386], [256.048, 170.817], [307.155, 200.142], [272.804, 242.035], [266.096, 291.465], [240.119, 351.788], [309.667, 340.057], [320.562, 280.57], [365.808, 310.732], [355.751, 238.677], [408.533, 252.92], [419.428, 188.41], [355.751, 188.41]], [[286, 175, 335, 224], [197, 189, 247, 244], [181, 98, 235, 150]]),
    ('marble_4073', 'marble', '4073.jpg', 29266, 'ca7cbc97e3243c99d0b7c2723c0bcea2f23a10cab528bdb5887cfc4ac2b573fa', 2856, 3029, 0.142857, 0.126775, 25, [[51.441, 329.754], [49.553, 271.107], [37.28, 192.352], [37.28, 134.542], [62.77, 74.22], [131.69, 72.544], [119.417, 137.056], [123.193, 199.893], [123.193, 267.755], [115.64, 332.267], [196.833, 339.808], [198.721, 262.729], [196.833, 197.379], [193.057, 130.353], [199.666, 72.544], [273.306, 72.544], [272.361, 132.029], [280.859, 199.893], [265.753, 265.243], [273.306, 340.646], [354.499, 343.998], [345.057, 265.243], [346.946, 205.758], [348.834, 133.705], [349.779, 73.382]], [[164, 169, 230, 227], [160, 99, 236, 166], [246, 173, 311, 228]]),
    ('marble_4079', 'marble', '4079.jpg', 27319, '23cc5bb0d2fe100a0cf3b3f0c546ac51d6fde9e6157546455f40718716764755', 320, 240, 1.6, 1.6, 24, [[43.952, 69.888], [56.016, 126.192], [37.92, 187.52], [116.336, 61.856], [119.36, 118.16], [127.408, 196.576], [142.48, 308.176], [209.84, 219.696], [180.688, 127.2], [176.672, 61.856], [234.976, 67.888], [257.104, 125.184], [312.4, 73.92], [382.768, 95.024], [435.056, 100.048], [336.528, 137.248], [297.312, 213.664], [262.128, 315.2], [364.672, 314.208], [372.72, 227.744], [410.912, 166.416], [484.32, 177.472], [437.056, 241.808], [458.176, 348.384]], [[270, 186, 324, 243], [97, 168, 159, 234], [231, 97, 289, 156]]),
    ('marble_5356', 'marble', '5356.jpg', 28039, 'fc6fa296851a8cabb08f25ecf50e5be0510fedf5f0290c18ca0c7a16368b5f57', 440, 330, 1.163636, 1.163636, 16, [[192.675, 216.925], [115.107, 218.38], [152.436, 174.743], [117.527, 134.493], [143.709, 104.925], [221.289, 99.107], [193.164, 128.198], [235.345, 168.925], [275.584, 135.948], [315.834, 99.107], [372.561, 119.471], [395.345, 89.402], [422.016, 165.527], [328.925, 180.073], [298.38, 213.039], [394.857, 212.073]], [[294, 78, 334, 119], [252, 107, 298, 156], [89, 194, 141, 246]]),
    ('marble_5371', 'marble', '5371.jpg', 28146, 'ca6f4a89900d1ace0ddfe9675bac23631be3e13705a21fb47b2e184dadd641c4', 1000, 1000, 0.384, 0.384, 44, [[74.807, 355.212], [30.57, 310.975], [9.458, 259.699], [55.707, 213.45], [67.772, 150.113], [20.517, 125.983], [48.668, 46.556], [75.813, 109.897], [97.932, 24.438], [106.982, 72.699], [127.089, 115.93], [123.068, 170.22], [114.017, 226.522], [86.872, 282.824], [121.056, 322.034], [165.293, 372.3], [191.432, 324.042], [238.687, 359.232], [275.885, 318.01], [228.634, 276.791], [152.221, 270.758], [166.299, 215.462], [202.491, 179.267], [178.364, 124.977], [171.325, 83.758], [157.252, 37.509], [220.589, 13.379], [205.509, 51.583], [255.779, 54.601], [227.628, 89.791], [237.681, 135.03], [258.793, 186.305], [292.977, 147.095], [282.924, 96.826], [320.122, 61.64], [347.267, 109.897], [377.43, 145.087], [348.273, 173.234], [378.436, 211.442], [382.456, 262.714], [338.22, 291.871], [335.201, 226.522], [276.891, 248.64], [219.583, 227.528]], [[231, 26, 284, 74], [178, 148, 228, 197], [129, 241, 184, 299]]),
    ('marble_5576', 'marble', '5576.jpg', 44318, '485904e569e9bb0f002f6d53d63d7c798ac4d24f4545a23837cc8c58e859b5e6', 1588, 1191, 0.322418, 0.322418, 67, [[15.795, 103.067], [4.74, 136.231], [27.431, 178.12], [2.995, 240.959], [40.812, 268.884], [62.34, 214.776], [97.248, 175.212], [1.831, 76.303], [72.812, 316.012], [135.067, 343.359], [176.959, 288.667], [233.395, 327.067], [116.448, 263.648], [62.34, 77.467], [71.648, 105.976], [54.776, 136.231], [119.94, 139.14], [125.759, 101.32], [151.359, 72.231], [99.576, 48.376], [160.084, 168.812], [130.412, 213.612], [193.248, 237.467], [242.703, 268.303], [303.795, 310.776], [475.431, 299.14], [380.012, 293.32], [393.395, 235.14], [317.759, 250.848], [466.12, 250.267], [509.176, 212.448], [510.34, 270.048], [273.54, 8.812], [204.303, 34.412], [154.267, 43.72], [192.667, 94.34], [180.448, 129.831], [227.576, 154.848], [208.376, 193.831], [257.248, 217.684], [277.031, 180.448], [325.903, 207.212], [445.759, 23.94], [396.303, 8.812], [347.431, 31.503], [300.884, 40.812], [303.795, 69.903], [203.72, 64.667], [257.248, 51.284], [245.031, 82.703], [236.884, 116.448], [284.012, 137.976], [289.831, 102.484], [338.12, 126.34], [347.431, 89.684], [383.503, 194.995], [336.376, 164.159], [397.467, 153.103], [450.995, 201.976], [471.94, 165.903], [509.759, 139.14], [450.412, 128.667], [501.031, 69.32], [455.648, 90.848], [405.031, 76.303], [392.812, 114.12], [355.576, 58.848]], [[326, 35, 381, 78], [246, 144, 306, 187], [366, 122, 427, 179]]),
    ('marble_5581', 'marble', '5581.jpg', 22213, '849593d46f892adbbc274ac1d23083b34e0dd540e9c4c5bf3da0d12de8f0b86e', 750, 522, 0.736, 0.735632, 8, [[73.541, 117.37], [129.337, 196.509], [186.193, 128.53], [266.962, 185.357], [212.233, 279.893], [333.386, 112.059], [409.908, 179.516], [468.891, 99.318]], [[365, 129, 470, 234], [150, 63, 258, 168], [165, 225, 262, 318]]),
    ('marble_5582', 'marble', '5582.jpg', 32935, '420f507674030744f49c5d7704a70e95653b98b8e95377027cb92b8222e1d336', 400, 400, 0.96, 0.96, 47, [[125.578, 57.216], [144.97, 88.243], [172.608, 123.638], [207.034, 138.662], [205.094, 47.03], [186.662, 85.334], [225.456, 77.088], [263.76, 70.301], [307.392, 77.088], [242.429, 114.912], [282.182, 112.973], [329.213, 124.608], [295.757, 158.544], [255.034, 152.246], [125.088, 128.966], [103.277, 87.754], [60.605, 128.486], [88.723, 152.246], [123.168, 182.304], [157.094, 157.094], [158.544, 197.338], [191.51, 175.517], [228.365, 183.274], [199.277, 212.851], [243.398, 229.814], [265.699, 192.97], [336.49, 173.088], [326.304, 214.301], [286.061, 230.304], [274.426, 266.669], [290.909, 304.973], [251.635, 301.094], [227.875, 341.338], [177.936, 323.395], [132.365, 323.875], [209.942, 292.368], [170.179, 281.693], [202.186, 253.094], [165.331, 234.662], [126.547, 218.669], [124.608, 274.426], [86.304, 301.574], [66.912, 270.058], [94.541, 240.49], [51.878, 222.058], [86.304, 192.0], [56.246, 172.118]], [[250, 176, 287, 214], [107, 201, 147, 242], [191, 122, 226, 158]]),
    ('marble_5594', 'marble', '5594.jpg', 46678, 'cfa9436032cec3b2bc4753475fa76d4e7e45276dc426d9c113f4b3a77de01e6c', 570, 427, 0.9, 0.899297, 15, [[57.366, 153.609], [81.819, 221.101], [114.426, 108.23], [165.672, 164.086], [179.64, 82.052], [241.362, 118.123], [286.2, 73.904], [241.362, 178.627], [193.617, 240.886], [289.107, 250.778], [385.182, 239.141], [358.398, 154.778], [376.452, 96.009], [453.897, 130.335], [452.727, 191.433]], [[74, 72, 155, 153], [200, 141, 284, 216], [321, 121, 396, 196]]),
    ('stamp_2907', 'stamp', '2907.jpg', 31370, 'c50cfbe0664e059efd39863f7cd589f7b48ad0269dff0d700a14d52b2ace721e', 474, 355, 1.082278, 1.08169, 28, [[60.326, 132.085], [102.816, 132.085], [142.406, 131.501], [180.827, 132.085], [220.406, 134.411], [52.761, 175.721], [96.42, 172.811], [139.495, 173.395], [184.323, 169.901], [223.902, 172.811], [221.575, 214.121], [180.827, 214.705], [136.584, 214.705], [95.836, 217.03], [48.681, 217.614], [43.443, 264.744], [84.19, 268.238], [127.265, 265.901], [175.004, 255.43], [80.121, 314.199], [129.603, 312.446], [178.5, 312.446], [220.99, 312.446], [407.932, 213.58], [71.961, 88.006], [214.941, 89.131], [310.333, 86.881], [461.776, 86.881]], [[203, 194, 242, 229], [79, 150, 115, 191], [110, 241, 151, 287]]),
    ('stamp_2927', 'stamp', '2927.jpg', 70396, '562fcafdba25c010161e3d180e7b371897d8cb088af55dd100513568dc8fc247', 750, 482, 0.797333, 0.79668, 49, [[155.727, 41.81], [79.996, 90.814], [149.954, 95.14], [204.763, 94.415], [262.466, 92.256], [264.627, 151.345], [201.885, 151.345], [143.464, 157.113], [78.553, 167.199], [74.232, 246.469], [134.813, 244.302], [201.159, 243.585], [263.184, 239.259], [159.331, 301.958], [76.393, 304.117], [77.11, 347.353], [154.284, 348.07], [345.405, 45.419], [346.122, 83.604], [339.632, 128.289], [338.915, 169.358], [336.028, 216.203], [335.303, 263.04], [329.538, 327.173], [393.723, 327.898], [461.512, 325.014], [544.451, 333.658], [541.573, 279.611], [533.639, 238.542], [533.639, 195.306], [527.867, 152.787], [527.149, 113.87], [524.191, 82.528], [523.465, 48.805], [487.625, 43.977], [446.586, 51.609], [400.716, 51.975], [396.538, 96.072], [436.205, 99.171], [475.506, 99.896], [477.523, 142.63], [438.294, 143.347], [396.171, 143.347], [391.993, 188.893], [426.757, 188.526], [474.429, 190.255], [473.417, 230.902], [409.088, 271.254], [475.219, 276.735]], [[444, 211, 500, 254], [378, 123, 417, 167], [244, 69, 289, 121]]),
    ('stamp_2928', 'stamp', '2928.jpg', 29415, 'fe2a5eb6e3fd4dfccc7b66665a1030028cd7bc0bbfdf5eda80f496cea0426013', 600, 849, 0.68, 0.452297, 45, [[324.884, 315.002], [236.538, 310.348], [153.442, 311.515], [72.087, 322.569], [70.339, 349.915], [65.967, 367.948], [154.312, 346.423], [325.761, 351.077], [346.752, 237.04], [267.152, 238.786], [201.552, 239.369], [133.321, 238.786], [59.84, 237.623], [23.106, 180.023], [96.58, 163.731], [169.184, 166.64], [240.04, 167.802], [307.394, 173.623], [365.996, 178.277], [360.747, 112.531], [293.393, 113.115], [233.913, 111.369], [163.064, 114.861], [105.332, 102.64], [41.473, 108.461], [38.848, 130.569], [105.332, 128.823], [167.436, 128.823], [234.79, 129.402], [303.892, 129.986], [305.64, 195.148], [235.661, 186.423], [168.307, 187.586], [100.953, 188.169], [30.104, 199.223], [295.147, 65.986], [228.664, 65.986], [165.682, 67.148], [114.077, 63.661], [107.08, 82.861], [168.307, 81.694], [232.166, 82.861], [299.52, 85.186], [360.747, 130.569], [371.246, 197.477]], [[168, 206, 229, 269], [204, 274, 277, 343], [78, 42, 136, 88]]),
    ('stamp_2929', 'stamp', '2929.jpg', 31216, 'b58fd1a8f72f921d60da9c10dcfcb54e20593fc4fb2efb1a587de2d4c2a9d17d', 525, 700, 0.777143, 0.548571, 77, [[345.759, 343.801], [301.112, 342.347], [257.157, 349.133], [209.759, 348.65], [168.547, 345.737], [105.357, 349.616], [48.346, 348.162], [46.287, 307.436], [80.629, 305.982], [128.019, 308.407], [165.803, 308.407], [204.264, 310.343], [250.287, 311.797], [279.818, 313.256], [327.9, 314.222], [356.747, 315.193], [373.238, 276.891], [333.394, 279.316], [288.747, 279.316], [246.168, 276.891], [211.134, 277.374], [174.041, 277.374], [132.145, 275.92], [92.993, 275.438], [56.592, 273.013], [49.722, 239.556], [92.309, 239.556], [128.019, 239.073], [172.666, 239.073], [211.134, 239.073], [250.971, 239.556], [290.807, 241.009], [333.394, 241.009], [373.238, 241.98], [361.558, 202.708], [312.792, 206.587], [269.521, 206.587], [234.487, 206.587], [176.792, 205.616], [122.524, 205.616], [67.58, 208.04], [57.275, 169.256], [106.041, 172.164], [152.064, 170.222], [195.335, 171.675], [239.982, 172.646], [288.064, 172.164], [334.086, 171.675], [374.264, 138.756], [330.581, 137.061], [268.557, 141.861], [208.935, 140.747], [151.722, 139.579], [107.968, 139.337], [58.651, 137.642], [74.652, 105.254], [126.651, 103.219], [178.642, 103.219], [222.053, 102.879], [270.757, 103.219], [317.463, 104.234], [360.392, 104.574], [338.135, 69.279], [289.439, 70.491], [241.357, 71.314], [197.324, 72.137], [149.25, 72.137], [103.982, 72.137], [66.819, 72.137], [58.783, 40.188], [97.733, 39.409], [138.324, 38.197], [182.419, 37.374], [227.066, 35.772], [273.982, 36.551], [322.677, 36.985], [372.477, 37.764]], [[218, 194, 252, 228], [129, 59, 174, 95], [272, 231, 312, 266]]),
    ('stamp_2939', 'stamp', '2939.jpg', 23444, '50b663b360dff655a8a6432bcd45dd69cb0611ec8b388439555ca65f5c67d5b7', 1600, 1200, 0.32, 0.32, 52, [[2.262, 60.445], [25.696, 60.848], [60.445, 60.042], [78.627, 93.171], [42.666, 93.981], [5.898, 94.787], [15.19, 127.92], [60.445, 129.939], [7.92, 168.323], [4.282, 213.171], [34.989, 214.384], [68.525, 215.19], [33.779, 265.293], [1.456, 259.635], [218.829, 240.646], [160.243, 190.95], [194.182, 194.586], [229.334, 190.141], [262.867, 190.95], [275.395, 148.525], [225.696, 148.525], [196.605, 150.544], [156.605, 154.586], [194.989, 110.544], [229.334, 110.141], [270.544, 108.525], [191.354, 54.787], [237.011, 51.152], [278.627, 51.555], [237.414, 75.395], [406.707, 282.262], [438.221, 283.878], [470.544, 283.072], [415.19, 247.514], [464.89, 243.878], [506.506, 239.03], [490.749, 199.434], [457.616, 202.666], [426.102, 203.475], [407.514, 164.282], [453.171, 161.05], [499.635, 161.456], [358.627, 132.768], [403.475, 134.384], [502.061, 105.696], [464.08, 104.89], [503.274, 77.414], [491.958, 48.323], [433.779, 72.566], [412.365, 49.133], [361.859, 49.133], [338.829, 48.726]], [[185, 133, 213, 170], [388, 152, 428, 179], [394, 268, 421, 300]]),
    ('stamp_2948', 'stamp', '2948.jpg', 32645, '1472d82ea308e5d7a5b06472befd991579f96c217acb4e86a1dd29162377fccf', 581, 800, 0.702238, 0.48, 11, [[251.696, 190.954], [329.16, 190.954], [166.578, 196.771], [81.46, 189.787], [350.438, 300.336], [262.763, 299.17], [172.54, 299.75], [83.159, 299.75], [227.862, 72.845], [340.22, 73.426], [112.955, 71.678]], [[208, 155, 284, 232], [217, 262, 306, 340], [308, 263, 397, 342]]),
    ('stamp_2951', 'stamp', '2951.jpg', 34297, '86f98b108ceae24f9509683144fd594de39c99260f07add9fcfcf5e0f0b3bb81', 700, 491, 0.781429, 0.782077, 14, [[45.19, 88.038], [52.723, 165.957], [44.354, 243.875], [45.19, 320.112], [105.462, 245.549], [109.65, 165.12], [159.036, 245.549], [163.225, 166.794], [103.789, 85.528], [201.734, 85.528], [264.514, 81.336], [250.284, 164.283], [308.883, 167.63], [366.646, 165.957]], [[343, 133, 398, 201], [283, 134, 337, 204], [173, 51, 229, 117]]),
    ('stamp_6986', 'stamp', '6986.jpg', 42942, '1351383414a65b73ecc1ddd0994fb1c343c2de8ec7bbea8789d49ab152bd7e37', 786, 556, 0.69084, 0.690647, 32, [[386.248, 145.955], [41.354, 343.383], [516.347, 45.838], [237.904, 245.138], [499.498, 340.572], [247.728, 147.826], [177.069, 337.768], [239.777, 337.298], [237.435, 50.044], [45.098, 246.54], [443.809, 145.015], [370.808, 244.669], [105.001, 338.238], [112.02, 145.955], [433.985, 245.601], [175.667, 145.955], [519.622, 138.938], [172.855, 245.601], [496.693, 243.729], [315.589, 145.485], [112.02, 48.642], [174.727, 48.173], [305.289, 48.642], [431.644, 337.298], [440.065, 49.112], [112.489, 244.669], [303.887, 340.109], [50.715, 50.044], [46.507, 150.63], [302.954, 244.669], [374.553, 49.582], [362.38, 340.572]], [[75, 105, 143, 184], [267, 203, 338, 283], [276, 99, 354, 184]]),
    ('stamp_7216', 'stamp', '7216.jpg', 41459, '88264c963e156632434985f467120149d626b045a762a982aaf0d1abdb2359af', 1000, 906, 0.424, 0.423841, 25, [[43.608, 195.548], [223.329, 350.402], [390.877, 200.227], [132.063, 349.465], [295.871, 39.29], [48.289, 36.484], [314.12, 196.95], [311.784, 351.339], [220.989, 273.208], [134.404, 272.742], [220.518, 118.824], [310.847, 271.339], [390.406, 274.145], [132.534, 116.95], [223.329, 198.353], [135.807, 196.014], [42.676, 351.805], [213.501, 39.29], [316.932, 118.353], [45.949, 116.95], [378.242, 36.014], [383.856, 352.742], [41.272, 273.208], [389.94, 108.063], [134.874, 37.887]], [[359, 63, 421, 147], [185, 162, 268, 228], [9, 80, 95, 146]]),
    ('stamp_7273', 'stamp', '7273.jpg', 31668, 'a96ed085e599b474ac5a7b8a06cbbe5c96ee36cfd5ca25cf7a50882a245726f4', 333, 500, 1.225225, 0.768, 32, [[239.09, 325.716], [136.466, 50.857], [54.265, 174.252], [94.477, 90.04], [143.927, 132.15], [67.424, 52.616], [130.768, 171.917], [124.238, 326.884], [49.695, 283.607], [53.432, 246.766], [119.668, 281.272], [261.39, 91.208], [314.564, 243.256], [240.867, 249.692], [315.594, 136.827], [145.79, 207.583], [216.608, 161.971], [56.226, 129.224], [209.232, 211.676], [313.633, 289.459], [218.568, 136.243], [310.827, 170.158], [71.161, 206.415], [156.988, 245.015], [190.572, 96.476], [231.531, 54.367], [223.224, 285.949], [44.941, 321.039], [314.564, 324.549], [316.427, 208.174], [37.48, 92.966], [364.1, 94.717]], [[102, 310, 189, 347], [186, 153, 274, 189], [102, 36, 189, 73]]),
    ('strawberry_298', 'strawberry', '298.jpg', 50090, 'aee3c8946b6bfa0caf98a7e8c09e06bf8e93b73f5df019ad36307e118f297ce2', 799, 533, 0.720901, 0.72045, 33, [[344.598, 287.633], [272.046, 298.374], [181.581, 269.729], [138.586, 235.71], [116.188, 178.42], [122.459, 128.291], [147.54, 90.697], [185.163, 53.991], [242.49, 28.926], [297.127, 36.087], [348.181, 45.936], [396.553, 62.052], [430.587, 105.913], [448.501, 155.149], [446.714, 210.645], [409.984, 256.3], [353.559, 223.181], [280.106, 232.129], [220.992, 221.387], [177.097, 178.42], [183.368, 130.978], [206.661, 94.278], [255.026, 71.0], [312.352, 73.688], [364.307, 93.378], [383.116, 130.085], [390.281, 183.794], [280.106, 194.536], [232.635, 162.31], [254.132, 124.71], [280.106, 142.613], [305.186, 121.129], [333.849, 160.516]], [[184, 68, 236, 116], [307, 148, 351, 189], [256, 204, 309, 258]]),
    ('strawberry_310', 'strawberry', '310.jpg', 55132, 'ad9cffc9ed91fc2cf6a7f916789d51fc28de065fab4140046f1bf8bef0d48ae3', 800, 600, 0.64, 0.64, 65, [[378.125, 140.819], [400.346, 80.55], [370.72, 78.867], [336.71, 80.211], [346.106, 98.496], [507.994, 176.006], [491.93, 136.915], [470.989, 118.765], [450.01, 101.293], [506.579, 95.027], [4.269, 326.208], [259.539, 313.997], [180.781, 309.805], [215.968, 260.378], [274.618, 253.67], [175.757, 238.592], [226.022, 200.051], [276.294, 195.027], [189.158, 194.189], [184.134, 154.81], [233.568, 161.51], [275.456, 144.755], [235.238, 122.138], [200.051, 127.162], [210.944, 98.675], [264.563, 99.514], [269.594, 114.592], [248.646, 101.19], [143.078, 97.837], [141.408, 129.677], [129.677, 168.218], [115.43, 211.782], [84.435, 255.347], [72.704, 319.021], [22.432, 267.078], [53.434, 196.704], [83.597, 155.648], [41.702, 150.624], [8.192, 199.213], [54.272, 123.808], [8.192, 130.515], [25.786, 116.269], [101.19, 115.43], [116.269, 96.998], [482.4, 286.349], [439.667, 304.781], [379.347, 300.589], [358.4, 244.454], [428.774, 236.915], [462.291, 240.269], [390.912, 182.784], [343.322, 202.566], [338.291, 162.349], [338.291, 124.646], [375.155, 112.922], [406.995, 122.138], [425.427, 173.242], [440.506, 200.89], [34.086, 9.037], [72.467, 3.782], [96.307, 6.208], [177.517, 6.208], [234.893, 7.821], [317.318, 5.395], [364.992, 7.014]], [[217, 94, 255, 143], [339, 244, 406, 333], [182, 233, 252, 291]]),
    ('strawberry_336', 'strawberry', '336.jpg', 33989, 'aea3767562c09cc516f972743428152d6c796394624f68e4a9f5507394bae2c9', 800, 671, 0.5725, 0.57228, 16, [[230.431, 117.529], [179.794, 107.64], [136.724, 109.386], [161.748, 150.109], [118.679, 161.166], [164.662, 202.473], [128.572, 212.945], [207.732, 204.218], [207.732, 165.24], [208.894, 244.947], [164.662, 246.693], [127.994, 249.022], [144.871, 281.018], [185.032, 290.913], [220.538, 285.093], [202.493, 317.095]], [[141, 226, 179, 268], [204, 91, 243, 141], [182, 177, 232, 220]]),
    ('strawberry_4911', 'strawberry', '4911.jpg', 57646, '91a6441724dbcf46c7cc1989389e6a9d9fa7a0862e98c3934c9df76a755f38e4', 450, 320, 1.2, 1.2, 30, [[99.12, 74.808], [221.004, 110.376], [89.796, 170.448], [104.952, 284.736], [186.012, 210.684], [15.732, 230.508], [11.064, 323.808], [12.228, 155.868], [19.236, 60.804], [91.08, 9.204], [162.804, 13.872], [231.612, 28.44], [319.092, 38.364], [296.928, 11.532], [398.28, 24.708], [328.764, 132.54], [408.372, 101.28], [483.948, 82.152], [294.54, 226.248], [382.188, 225.264], [450.708, 182.928], [525.288, 152.712], [517.188, 254.472], [456.768, 347.196], [433.56, 278.676], [352.968, 311.916], [230.04, 323.988], [146.364, 353.208], [467.964, 9.708], [527.736, 29.628]], [[170, 53, 263, 180], [371, 60, 442, 148], [31, 124, 138, 233]]),
    ('strawberry_4915', 'strawberry', '4915.jpg', 47859, '216df0714f62843ec10d08144ad20ee6e43b4de6a79ee7c15fc0b67f2025393c', 866, 1300, 0.471132, 0.295385, 87, [[375.421, 259.552], [8.367, 222.534], [13.823, 36.512], [54.637, 13.954], [130.895, 26.074], [192.655, 4.862], [220.042, 30.788], [294.151, 32.135], [343.559, 15.638], [390.281, 13.28], [373.739, 53.146], [330.777, 47.521], [273.049, 64.928], [219.773, 60.149], [172.354, 48.936], [196.038, 76.141], [142.334, 88.5], [112.37, 55.671], [81.76, 40.855], [45.238, 79.24], [85.515, 74.526], [15.166, 124.694], [104.313, 124.02], [177.348, 116.612], [42.557, 134.122], [238.035, 89.342], [234.275, 122.673], [298.179, 100.788], [374.978, 101.798], [347.587, 75.872], [362.625, 128.061], [289.053, 143.885], [11.1, 164.73], [47.448, 162.789], [95.39, 164.73], [143.337, 149.698], [189.739, 149.214], [243.872, 167.152], [304.191, 178.306], [343.629, 156.486], [401.626, 151.639], [386.163, 200.608], [322.749, 200.608], [262.43, 210.789], [201.338, 189.457], [133.283, 186.547], [66.778, 199.639], [45.898, 226.79], [124.779, 226.306], [210.619, 222.428], [164.99, 232.122], [351.361, 230.184], [400.853, 241.82], [310.377, 247.154], [259.334, 240.366], [214.483, 253.458], [158.032, 267.518], [105.444, 267.518], [52.088, 266.062], [13.418, 268.002], [15.741, 308.246], [57.502, 301.458], [128.647, 304.852], [188.966, 302.911], [249.285, 301.942], [261.657, 268.487], [291.814, 300.97], [330.48, 276.728], [395.44, 300.486], [351.361, 322.306], [399.308, 334.91], [313.467, 342.67], [256.244, 325.213], [219.123, 347.033], [173.499, 331.516], [93.072, 340.245], [52.861, 337.335], [17.286, 374.669], [63.688, 370.306], [110.085, 373.213], [181.23, 374.184], [268.616, 376.607], [311.922, 371.759], [224.537, 366.912], [362.96, 365.94], [391.576, 168.124], [6.926, 195.37]], [[347, 29, 402, 72], [168, 166, 228, 210], [164, 59, 215, 98]]),
    ('strawberry_4917', 'strawberry', '4917.jpg', 30719, '6664f606bd750f087fcece91f83f94f087236c550390f4a3e7a86f9891aad656', 1480, 1991, 0.275676, 0.192868, 37, [[258.89, 20.949], [315.439, 16.295], [344.548, 57.604], [396.107, 90.767], [343.715, 107.059], [296.313, 109.967], [251.405, 100.077], [223.129, 116.367], [265.542, 135.567], [363.674, 153.021], [253.068, 188.513], [178.222, 172.221], [180.716, 146.04], [122.505, 169.313], [84.249, 129.167], [36.847, 122.768], [143.293, 103.568], [184.044, 89.605], [222.299, 57.021], [194.023, 22.695], [120.84, 32.005], [144.126, 65.168], [80.922, 65.168], [47.659, 200.731], [112.525, 255.423], [189.034, 264.15], [73.437, 285.676], [98.386, 313.023], [67.618, 345.604], [188.201, 306.621], [255.562, 343.858], [259.722, 370.623], [342.053, 342.113], [360.347, 372.368], [391.95, 300.805], [393.612, 246.113], [300.47, 225.167]], [[270, 207, 326, 241], [124, 86, 170, 117], [211, 166, 278, 211]]),
    ('strawberry_4931', 'strawberry', '4931.jpg', 60413, '03fc38f2ff43a639e40c0ad1a1433d981aaf0fce38662365b1961fadf65e9164', 1300, 866, 0.443077, 0.443418, 58, [[561.897, 357.244], [493.796, 377.38], [391.64, 358.792], [447.361, 329.362], [345.206, 375.832], [295.674, 364.986], [246.147, 354.145], [191.972, 361.891], [100.649, 366.538], [13.975, 377.38], [23.262, 332.457], [38.738, 265.851], [113.033, 261.204], [109.936, 313.869], [185.782, 301.48], [264.721, 290.634], [345.206, 309.222], [366.872, 242.616], [434.977, 258.105], [496.889, 230.227], [549.517, 287.539], [560.351, 225.58], [499.986, 296.212], [411.915, 296.212], [510.819, 160.673], [542.552, 200.021], [14.13, 169.35], [35.025, 100.727], [109.626, 49.92], [61.955, 27.461], [75.421, 148.439], [118.607, 168.574], [163.491, 100.727], [173.863, 190.262], [203.736, 143.171], [208.999, 227.593], [285.151, 200.793], [300.783, 256.092], [334.368, 190.262], [308.213, 120.091], [254.503, 82.76], [394.888, 155.254], [438.075, 151.534], [467.171, 99.955], [545.645, 103.671], [409.749, 42.333], [357.585, 88.648], [347.838, 21.422], [275.399, 24.517], [203.736, 40.936], [5.459, 11.041], [163.491, 20.49], [515.002, 31.332], [571.184, 49.61], [565.924, 10.731], [567.316, 162.69], [533.42, 343.148], [397.52, 204.974]], [[140, 65, 209, 127], [277, 89, 342, 158], [363, 115, 430, 189]]),
    ('strawberry_4933', 'strawberry', '4933.jpg', 43524, '8dced0c04f8615b4b1a54622f37fc9ab7425a267987b56c002c037161055160e', 750, 1334, 0.544, 0.287856, 54, [[11.527, 9.997], [74.202, 15.233], [153.37, 9.416], [254.527, 29.197], [334.794, 16.981], [354.585, 57.707], [402.963, 32.689], [255.626, 67.597], [175.358, 76.907], [159.963, 43.743], [90.696, 49.563], [86.295, 87.379], [21.423, 79.235], [20.324, 43.161], [30.219, 110.652], [62.108, 144.979], [10.428, 144.979], [135.777, 120.543], [225.94, 117.054], [313.899, 102.508], [389.771, 96.688], [371.079, 147.889], [285.312, 163.016], [181.957, 166.507], [107.19, 184.545], [59.905, 186.289], [4.929, 187.452], [10.428, 233.998], [0.533, 276.471], [85.196, 234.58], [59.905, 279.963], [159.963, 211.888], [196.248, 249.707], [241.329, 210.725], [345.788, 203.745], [397.468, 184.545], [366.678, 257.853], [287.515, 254.943], [324.899, 301.489], [242.434, 293.343], [151.172, 292.18], [36.818, 327.088], [111.585, 327.088], [219.341, 334.071], [319.399, 342.799], [397.468, 315.453], [391.968, 358.507], [336.992, 375.845], [251.23, 373.519], [165.463, 368.865], [66.504, 367.699], [223.078, 3.656], [203.07, 30.654], [296.529, 61.198]], [[226, 40, 293, 91], [185, 82, 265, 145], [293, 170, 390, 232]]),
    ('strawberry_5851', 'strawberry', '5851.jpg', 46681, '60a373e0a3265f5c09eb6c0fd26d3cbce0213f62d4d4d062815584a2b8df4f5f', 680, 453, 0.847059, 0.847682, 43, [[270.11, 70.773], [319.646, 94.508], [249.882, 119.642], [224.064, 91.719], [203.142, 152.464], [125.695, 157.347], [113.142, 220.185], [174.536, 192.958], [160.577, 226.467], [154.995, 262.078], [110.346, 276.734], [178.721, 329.104], [139.646, 357.027], [135.462, 308.158], [228.257, 370.293], [241.513, 332.588], [226.859, 262.773], [277.784, 288.602], [305.695, 303.962], [286.162, 348.652], [330.116, 376.574], [368.488, 368.894], [348.954, 332.588], [406.859, 338.174], [445.231, 338.174], [418.024, 378.668], [429.179, 292.094], [470.346, 242.522], [399.88, 254.398], [374.764, 287.212], [341.975, 235.545], [268.721, 234.147], [305.695, 214.599], [238.024, 201.333], [277.784, 164.332], [323.136, 151.065], [371.977, 139.893], [406.859, 128.025], [444.528, 165.027], [431.272, 213.904], [374.764, 193.653], [428.485, 244.624], [360.813, 112.665]], [[446, 208, 496, 270], [306, 126, 350, 174], [103, 138, 151, 189]]),
    ('strawberry_5856', 'strawberry', '5856.jpg', 40764, '23b7ea8cea867376d56b7f4e5ecbf933d2053046eedc1e579b1e98586b6ddead', 640, 480, 0.8, 0.8, 78, [[101.24, 91.344], [107.056, 106.472], [141.96, 94.256], [154.184, 108.216], [169.888, 134.4], [124.512, 156.512], [130.912, 134.4], [88.44, 141.384], [53.528, 158.84], [87.856, 173.384], [87.856, 206.544], [58.184, 242.616], [117.528, 240.872], [144.872, 247.856], [158.256, 211.2], [124.512, 200.728], [118.688, 228.072], [67.488, 133.24], [200.728, 164.656], [229.816, 186.76], [194.912, 218.184], [256.584, 228.656], [190.84, 256.584], [222.256, 251.928], [261.24, 260.072], [302.544, 257.744], [296.728, 226.328], [283.344, 206.544], [288.0, 175.128], [257.744, 158.256], [237.96, 125.088], [203.056, 114.04], [183.856, 92.512], [165.24, 86.688], [221.088, 94.84], [243.2, 98.328], [269.96, 98.328], [256.0, 84.36], [288.0, 88.44], [318.256, 107.64], [294.984, 136.144], [273.456, 125.088], [322.328, 164.072], [299.64, 199.56], [357.24, 145.456], [378.184, 182.688], [343.856, 182.112], [351.416, 211.2], [328.144, 212.36], [336.872, 243.784], [372.36, 255.416], [381.672, 212.36], [409.6, 238.544], [429.384, 257.744], [450.328, 224.0], [429.96, 194.912], [454.984, 168.728], [402.616, 151.856], [385.16, 126.256], [424.728, 121.6], [403.784, 96.0], [371.784, 97.744], [357.24, 118.688], [349.088, 96.0], [341.528, 78.544], [364.216, 76.216], [376.44, 80.872], [399.712, 71.56], [426.472, 90.184], [89.6, 247.856], [48.872, 196.656], [151.856, 177.456], [171.64, 185.6], [197.816, 193.16], [229.24, 152.44], [203.64, 139.056], [129.16, 77.384], [329.888, 133.816]], [[360, 170, 397, 201], [147, 113, 183, 156], [232, 138, 275, 181]]),
    ('sunglasses_2017', 'sunglasses', '2017.jpg', 54150, '9342d74a65dcf8d09b5e6b4a5a742e91df5b43b0f346c70feaccefc09bc76d61', 384, 280, 1.372396, 1.371429, 104, [[200.891, 76.238], [334.796, 189.106], [419.226, 212.379], [266.094, 197.253], [268.427, 243.799], [324.901, 229.838], [379.632, 217.618], [375.556, 182.702], [376.723, 151.872], [331.886, 156.521], [331.31, 115.803], [204.377, 23.287], [203.801, 123.36], [209.043, 171.648], [209.043, 214.121], [204.967, 261.257], [206.71, 310.121], [200.891, 345.614], [126.947, 331.077], [121.704, 286.272], [121.114, 235.077], [117.038, 182.126], [124.614, 132.096], [326.068, 78.555], [330.144, 36.096], [278.898, 28.526], [277.745, 73.33], [278.322, 116.379], [278.322, 156.521], [268.427, 286.848], [274.822, 324.096], [320.825, 267.648], [324.325, 309.545], [367.404, 289.755], [374.966, 253.687], [408.164, 248.448], [410.484, 278.126], [408.741, 313.029], [371.48, 325.838], [25.636, 11.067], [23.893, 69.257], [19.817, 131.506], [23.303, 191.438], [19.817, 247.872], [21.56, 306.638], [22.727, 347.355], [116.462, 25.029], [110.052, 75.648], [299.868, 349.111], [389.527, 352.594], [411.65, 346.203], [447.744, 351.429], [444.835, 328.155], [446.591, 294.994], [449.501, 262.423], [448.334, 232.155], [446.591, 203.067], [416.316, 181.55], [415.726, 149.541], [328.977, 4.677], [377.299, 5.829], [380.209, 36.096], [379.632, 74.496], [372.647, 111.141], [415.726, 11.067], [418.636, 42.487], [416.893, 77.979], [418.636, 112.306], [445.425, 110.565], [450.667, 79.721], [454.153, 44.229], [450.667, 12.233], [487.928, 17.472], [514.703, 20.955], [487.338, 47.726], [485.018, 77.979], [482.685, 106.491], [479.776, 137.321], [450.077, 141.984], [450.077, 173.979], [482.685, 166.423], [473.957, 198.994], [474.533, 225.175], [473.367, 254.853], [466.958, 283.365], [475.7, 311.877], [471.034, 341.541], [497.823, 350.853], [502.475, 329.911], [497.246, 300.233], [505.385, 275.794], [502.475, 248.448], [507.141, 218.194], [508.308, 189.106], [503.065, 161.76], [502.475, 137.911], [507.718, 106.491], [511.217, 50.057], [511.217, 80.311], [520.522, 336.891], [522.855, 311.287], [523.445, 287.438], [524.022, 262.999], [524.598, 236.818]], [[167, 52, 241, 99], [401, 197, 436, 225], [307, 172, 357, 210]]),
    ('sunglasses_2019', 'sunglasses', '2019.jpg', 40214, '9a7935ff2f7dbe594fca84422e4b97a723a88d2238337c84b9cf739a7038ffbc', 1300, 956, 0.401538, 0.401674, 14, [[60.809, 62.689], [55.501, 145.542], [45.944, 209.28], [35.857, 279.388], [217.967, 213.526], [225.4, 128.017], [233.896, 47.285], [188.767, 7.451], [460.926, 61.376], [423.972, 6.921], [454.702, 153.801], [432.469, 248.05], [416.01, 330.373], [213.723, 296.383]], [[367, 214, 520, 301], [153, 105, 315, 187], [143, 194, 313, 268]]),
    ('sunglasses_2027', 'sunglasses', '2027.jpg', 35169, '68a3c6fb3fdc204c50104a6cc5245e504c84346c0521cc730d621cc43eae9051', 520, 347, 1.105769, 1.106628, 23, [[49.384, 305.64], [50.456, 197.179], [534.562, 295.603], [452.591, 321.697], [354.233, 344.814], [231.294, 378.378], [230.542, 273.979], [355.726, 253.838], [455.577, 247.132], [537.548, 232.956], [539.781, 163.604], [459.303, 165.098], [360.204, 175.544], [230.542, 177.78], [47.238, 71.887], [227.567, 75.616], [363.179, 80.839], [459.303, 95.004], [539.781, 91.275], [539.781, 23.416], [457.07, 10.734], [357.219, 1.04], [241.721, 6.264]], [[297, 64, 393, 125], [405, 230, 487, 288], [290, 161, 401, 215]]),
    ('sunglasses_2031', 'sunglasses', '2031.jpg', 38624, 'fd618f2674a7847ccc5364e5e6974ae5dca6330caea9a58515b2215507ea1e5c', 1300, 958, 0.400769, 0.400835, 31, [[60.252, 69.409], [99.363, 77.674], [154.998, 95.303], [203.471, 106.322], [239.828, 117.89], [320.8, 57.836], [270.122, 42.412], [231.685, 35.434], [190.802, 24.784], [176.479, 18.17], [134.614, 17.621], [23.345, 143.234], [6.821, 134.416], [77.881, 155.352], [115.337, 169.128], [168.219, 191.162], [238.173, 273.802], [267.369, 292.537], [314.74, 203.284], [366.519, 139.378], [340.63, 144.337], [375.332, 212.098], [416.648, 154.803], [316.944, 314.571], [376.988, 360.299], [426.014, 278.212], [436.806, 379.859], [469.197, 293.195], [514.864, 203.724], [475.861, 225.598], [453.939, 185.599]], [[122, 63, 207, 131], [397, 123, 449, 198], [288, 34, 356, 92]]),
    ('sunglasses_2035', 'sunglasses', '2035.jpg', 30724, 'f5f19bc336521c8bb8fa25a3ead8483eaeb79d83d745d12bd7c8ee8fefcae091', 1209, 1201, 0.320099, 0.319734, 28, [[91.251, 34.71], [189.691, 17.838], [187.943, 42.854], [306.77, 37.038], [197.261, 74.274], [72.029, 77.18], [74.942, 107.437], [69.116, 152.82], [190.273, 145.255], [178.039, 115.002], [303.857, 76.02], [307.932, 110.346], [299.779, 152.82], [181.535, 184.237], [82.512, 190.056], [82.512, 226.71], [176.291, 223.219], [289.875, 191.802], [274.732, 233.108], [73.194, 263.946], [168.139, 267.438], [286.383, 268.602], [187.36, 305.838], [295.701, 305.256], [58.629, 305.256], [58.921, 351.218], [197.786, 354.824], [323.31, 349.242]], [[117, 208, 233, 255], [27, 62, 132, 95], [236, 216, 343, 257]]),
    ('sunglasses_2042', 'sunglasses', '2042.jpg', 40009, '3f3206cfcdfd648d1c7ce7638565c49ba4c2f528b2033e1f1e9c046abd0183dc', 700, 420, 0.914286, 0.914286, 81, [[521.847, 13.019], [569.536, 1.966], [614.455, 3.346], [525.303, 39.287], [569.536, 33.755], [614.455, 28.224], [619.291, 58.642], [571.602, 62.784], [520.457, 68.315], [530.139, 97.344], [530.139, 125.678], [573.678, 93.888], [580.59, 124.992], [620.681, 91.813], [626.898, 120.841], [631.049, 152.631], [572.992, 150.565], [534.281, 183.049], [587.502, 184.43], [631.735, 184.43], [633.81, 221.751], [579.209, 214.839], [534.976, 210.688], [531.52, 237.166], [580.59, 244.069], [633.81, 249.6], [534.976, 268.955], [578.514, 277.248], [629.659, 282.085], [450.651, 316.297], [472.777, 312.146], [492.123, 304.549], [510.784, 292.105], [494.894, 328.741], [514.935, 319.753], [537.326, 309.851], [561.243, 300.398], [551.561, 339.109], [568.841, 329.426], [588.882, 319.753], [610.313, 311.461], [607.543, 351.543], [620.526, 339.84], [438.217, 366.409], [471.387, 366.409], [492.123, 367.095], [512.165, 365.019], [493.504, 379.538], [519.077, 377.463], [540.507, 378.843], [560.549, 376.777], [382.921, 331.154], [393.289, 324.937], [395.365, 319.406], [375.323, 318.025], [423.013, 287.616], [394.67, 283.465], [368.411, 277.943], [412.645, 268.261], [445.815, 272.411], [441.673, 239.232], [411.264, 238.546], [377.399, 228.178], [402.277, 165.833], [348.37, 254.025], [309.659, 112.338], [218.432, 94.373], [365.641, 130.176], [353.893, 81.097], [458.953, 16.613], [419.557, 23.525], [384.997, 26.295], [352.512, 34.587], [332.471, 38.738], [299.291, 41.499], [281.326, 44.955], [260.59, 50.487], [237.787, 50.487], [257.829, 91.95], [240.549, 130.999], [244.005, 167.634]], [[497, 313, 538, 326], [517, 228, 554, 250], [509, 115, 552, 132]]),
    ('sunglasses_6547', 'sunglasses', '6547.jpg', 51328, 'a0dd26251a64c8d177465fd598fcd6ba54428992b0e6a590d33d588700057909', 500, 376, 1.022, 1.021277, 72, [[67.207, 129.437], [9.913, 130.826], [5.723, 102.904], [67.207, 96.623], [176.898, 104.303], [265.628, 105.692], [353.663, 113.372], [461.954, 107.091], [461.259, 73.583], [355.063, 74.972], [264.228, 70.785], [178.993, 73.583], [67.207, 62.41], [11.314, 68.691], [464.049, 41.464], [350.863, 40.065], [268.418, 37.971], [183.183, 36.572], [71.397, 32.385], [8.513, 37.277], [5.723, 4.463], [70.702, 3.758], [180.393, 7.251], [268.418, 4.463], [356.453, 10.744], [463.354, 9.345], [8.513, 364.024], [67.902, 363.329], [175.498, 368.906], [262.133, 368.906], [352.958, 373.797], [457.764, 368.211], [459.164, 336.092], [352.263, 338.186], [262.133, 333.304], [174.803, 332.609], [70.702, 326.318], [8.513, 329.117], [5.723, 298.397], [68.597, 294.904], [176.193, 299.786], [264.228, 303.983], [351.568, 303.983], [461.954, 303.983], [462.659, 270.465], [351.568, 269.77], [264.933, 270.465], [173.403, 262.09], [68.597, 259.292], [8.513, 264.878], [5.723, 229.277], [71.397, 227.877], [177.593, 232.769], [264.933, 239.745], [353.663, 243.237], [462.659, 241.144], [461.259, 204.143], [352.263, 204.837], [262.133, 211.823], [173.403, 200.65], [68.597, 195.064], [9.913, 201.345], [7.113, 164.344], [67.902, 163.649], [174.098, 167.132], [264.933, 173.423], [353.663, 174.812], [463.354, 175.517], [462.659, 139.21], [350.863, 139.905], [265.628, 135.717], [176.193, 134.318]], [[228, 60, 307, 95], [427, 225, 505, 259], [313, 227, 401, 261]]),
    ('sunglasses_6785', 'sunglasses', '6785.jpg', 29566, '5d63e4f27af1d1316808af5913e24066824e70dc7cb56a6e0010ad0aca0ed311', 2174, 2358, 0.187672, 0.16285, 45, [[185.753, 160.015], [33.712, 73.463], [31.555, 264.81], [29.399, 227.85], [259.615, 277.441], [184.135, 234.869], [106.497, 338.728], [106.497, 269.489], [258.538, 351.36], [32.096, 112.762], [189.527, 77.675], [338.871, 280.716], [32.634, 154.4], [357.201, 37.441], [266.624, 118.845], [256.92, 312.997], [114.045, 77.207], [187.37, 197.441], [262.312, 160.482], [349.114, 120.716], [181.979, 273.231], [34.252, 299.43], [114.045, 117.908], [183.056, 310.657], [272.016, 33.699], [344.263, 83.288], [109.732, 194.633], [334.558, 357.441], [107.037, 305.044], [262.312, 240.015], [112.968, 156.271], [349.653, 166.095], [183.596, 345.745], [262.312, 200.248], [33.712, 334.049], [341.566, 204.927], [115.663, 32.762], [104.34, 230.657], [31.555, 192.762], [35.868, 31.358], [268.242, 81.886], [198.152, 36.504], [342.645, 245.161], [185.753, 118.845], [337.253, 321.886]], [[236, 111, 296, 134], [234, 265, 289, 291], [160, 188, 219, 210]]),
    ('sunglasses_7221', 'sunglasses', '7221.jpg', 58122, '245c2c0f352a08da4101759aaf33448d78784d5b2232c12547220a27ac7352a4', 800, 500, 0.7675, 0.768, 60, [[322.25, 293.184], [432.241, 141.059], [554.572, 211.546], [86.067, 4.093], [555.355, 294.628], [325.059, 370.099], [12.702, 129.362], [321.705, 94.472], [428.848, 293.768], [13.523, 58.207], [554.189, 329.71], [11.106, 363.786], [219.044, 13.647], [326.771, 175.327], [11.927, 165.734], [556.522, 252.088], [83.067, 202.698], [218.461, 53.491], [84.394, 82.813], [84.394, 241.098], [427.175, 331.392], [322.834, 17.004], [426.085, 17.549], [428.303, 255.598], [553.406, 369.83], [322.834, 54.036], [557.612, 16.65], [430.56, 177.001], [322.834, 134.354], [216.78, 168.584], [213.979, 281.994], [8.696, 13.763], [428.303, 215.754], [85.76, 41.718], [426.63, 56.847], [216.235, 249.984], [215.107, 211.231], [85.093, 360.353], [554.964, 173.376], [81.708, 123.901], [85.093, 321.293], [556.522, 96.538], [215.652, 323.512], [559.369, 55.872], [322.834, 327.997], [9.471, 204.526], [4.643, 288.622], [310.484, 225.27], [216.235, 128.74], [85.093, 280.865], [83.067, 162.301], [429.977, 98.404], [217.325, 368.425], [12.702, 94.587], [218.461, 91.661], [558.863, 133.962], [7.875, 246.551], [8.696, 321.761], [325.643, 255.053], [427.72, 368.97]], [[273, 161, 372, 203], [37, 112, 152, 153], [378, 45, 488, 82]]),
    ('sunglasses_7386', 'sunglasses', '7386.jpg', 45229, '4763c8c4802a6b08494435049c3d1b926be38773cb4383ef75fa97ca0d0a8cf3', 429, 640, 0.951049, 0.6, 64, [[225.703, 250.41], [350.728, 44.034], [377.871, 108.372], [67.306, 251.532], [375.198, 336.864], [191.265, 44.034], [306.685, 307.668], [67.306, 337.986], [225.703, 165.072], [144.731, 13.272], [376.986, 280.722], [144.731, 220.092], [373.42, 250.968], [226.597, 220.65], [146.509, 338.55], [232.826, 131.388], [221.252, 191.46], [376.986, 368.304], [303.128, 337.422], [302.234, 105.564], [150.96, 367.74], [141.164, 281.844], [61.97, 74.124], [71.757, 221.214], [380.543, 219.528], [378.765, 166.758], [268.776, 46.788], [144.731, 165.072], [147.394, 193.704], [237.81, 14.676], [142.943, 135.882], [303.128, 194.268], [142.058, 250.41], [143.837, 309.918], [386.145, 11.868], [102.723, 45.384], [64.633, 311.598], [68.2, 134.196], [371.641, 74.688], [304.906, 134.76], [65.527, 165.072], [380.543, 135.318], [231.932, 281.844], [371.641, 74.124], [71.671, 10.464], [68.2, 281.844], [381.428, 189.774], [228.375, 369.426], [58.413, 104.442], [61.97, 194.268], [231.048, 307.668], [296.004, 281.844], [302.234, 76.374], [300.455, 163.95], [305.791, 221.214], [140.28, 76.374], [146.509, 105.006], [306.418, 16.08], [70.863, 367.74], [376.092, 307.11], [231.048, 105.564], [296.004, 366.618], [302.234, 252.654], [223.924, 339.672]], [[165, 119, 243, 144], [245, 241, 322, 260], [86, 273, 163, 292]]),
)
CORPUS_BYTES = 2732222
SAMPLE_CATEGORIES = ["apple", "cashew nut", "egg", "finger food", "marble", "stamp", "strawberry", "sunglasses"]

SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 5, "validation": 2, "test": 3}  # per category; 8 categories -> 40 / 16 / 24
MIN_RECORDS = 4
MAX_RECORDS = 20_000
MIN_CLASSES = 1
MAX_CLASSES = 100
MAX_LABEL_CHARS = MAX_TEXT_CHARS
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")
_LABEL_RE = re.compile(r"^[A-Za-z0-9_ .'-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def image_url(file: str) -> str:
    """The mirror's served object for a pinned image (`images_384_VarV2/<file>` at the pinned revision)."""
    if not re.match(r"^[0-9]+\.jpg$", file):
        raise ValueError(f"unexpected FSC-147 file name {file!r}")
    return f"{CORPUS_BASE_URL}{file}"


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:
    """Return every pinned image (bytes keyed by record id) from the cache or the Hub mirror."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out = {}
    for rid, _category, file, size, digest, *_rest in SAMPLE_RECORDS:
        local = cache / file
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = image_url(file)
            if fetcher is not None:
                data = fetcher(url)
            else:
                request = urllib.request.Request(url, headers={"User-Agent": "dimer-countgd-tutorial/1.0"})
                with urllib.request.urlopen(request, timeout=120) as response:  # noqa: S310 (pinned https URL)
                    data = response.read()
            if len(data) != size or _sha256_bytes(data) != digest:
                raise ValueError(
                    f"{rid} ({file}): fetched {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, "
                    f"pinned {size} / {digest[:16]}…"
                )
            local.write_bytes(data)
        out[rid] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> list[dict[str, Any]]:
    """Decode the verified image bytes into `{id, image, label, count, points, exemplars}` records with their provenance."""
    out = []
    for rid, category, file, _size, _digest, orig_w, orig_h, ratio_w, ratio_h, count, points, boxes in SAMPLE_RECORDS:
        if rid not in files:
            raise ValueError(f"corpus is missing {rid}")
        image = Image.open(io.BytesIO(files[rid]))
        image.load()
        if len(points) != count:
            raise ValueError(f"{rid}: {len(points)} points pinned for a count of {count}")
        out.append(
            {
                "id": rid,
                "image": image.convert("RGB"),
                "label": category,
                "count": count,
                "points": [list(p) for p in points],
                "exemplars": [list(b) for b in boxes],
                "source_file": file,
                "source_url": image_url(file),
                "original_size": [orig_w, orig_h],
                "resize_ratio": [ratio_w, ratio_h],
            }
        )
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded stratified draw per category: `sizes` counts per class for train / validation / test."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    rng = random.Random(seed)
    by_label: dict[str, list[dict[str, Any]]] = {}
    for record in records:
        by_label.setdefault(str(record["label"]), []).append(dict(record))
    out: dict[str, list[dict[str, Any]]] = {name: [] for name in sizes}
    for label in sorted(by_label):
        pool = by_label[label]
        rng.shuffle(pool)
        needed = sum(sizes.values())
        if len(pool) < needed:
            raise ValueError(f"{label}: only {len(pool)} records available, need {needed}")
        cursor = 0
        for name, per_class in sizes.items():
            out[name].extend(pool[cursor : cursor + per_class])
            cursor += per_class
    for part in out.values():
        rng.shuffle(part)
    return out


def fetch_sample_dataset(
    *,
    seed: int = SAMPLE_SEED,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
) -> dict[str, list[dict[str, Any]]]:
    """`fetch_corpus` → `read_corpus` → `build_sample_dataset` in one call."""
    return build_sample_dataset(read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed)


def _check_boxes(value: Any, width: int, height: int, what: str) -> list[list[float]]:
    if isinstance(value, Mapping) or not isinstance(value, Sequence) or isinstance(value, (str, bytes)):
        raise ValueError(f"{what}: exemplars must be a list of [x0, y0, x1, y1] boxes")
    if len(value) > MAX_EXEMPLARS:
        raise ValueError(f"{what}: at most {MAX_EXEMPLARS} exemplar boxes")
    out = []
    for i, box in enumerate(value):
        if isinstance(box, Mapping) or not isinstance(box, Sequence) or len(box) != 4:
            raise ValueError(f"{what}: exemplar {i} must be [x0, y0, x1, y1]")
        try:
            x0, y0, x1, y1 = (float(v) for v in box)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{what}: exemplar {i} must hold numbers") from exc
        if not (0.0 <= x0 < x1 <= width and 0.0 <= y0 < y1 <= height):
            raise ValueError(f"{what}: exemplar {i} {box} is outside the image or empty ({width} x {height})")
        if x1 - x0 < MIN_EXEMPLAR_SIDE or y1 - y0 < MIN_EXEMPLAR_SIDE:
            raise ValueError(f"{what}: exemplar {i} sides must be at least {MIN_EXEMPLAR_SIDE} px")
        out.append([x0, y0, x1, y1])
    return out


def _check_points(value: Any, width: int, height: int, what: str) -> list[list[float]]:
    if isinstance(value, Mapping) or not isinstance(value, Sequence) or isinstance(value, (str, bytes)):
        raise ValueError(f"{what}: points must be a list of [x, y] pairs")
    if len(value) > MAX_COUNT:
        raise ValueError(f"{what}: at most {MAX_COUNT} points")
    out = []
    for i, point in enumerate(value):
        if isinstance(point, Mapping) or not isinstance(point, Sequence) or len(point) != 2:
            raise ValueError(f"{what}: point {i} must be [x, y]")
        try:
            x, y = float(point[0]), float(point[1])
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{what}: point {i} must hold numbers") from exc
        if not (0.0 <= x <= width and 0.0 <= y <= height):
            raise ValueError(f"{what}: point {i} {point} is outside the image ({width} x {height})")
        out.append([x, y])
    return out


def _check_object_boxes(value: Any, width: int, height: int, what: str) -> list[list[float]]:
    """Gold object boxes `[x0, y0, x1, y1]` (one per counted object, full extents), in image pixels."""
    if isinstance(value, Mapping) or not isinstance(value, Sequence) or isinstance(value, (str, bytes)):
        raise ValueError(f"{what}: boxes must be a list of [x0, y0, x1, y1] boxes")
    if len(value) > MAX_COUNT:
        raise ValueError(f"{what}: at most {MAX_COUNT} boxes")
    out = []
    for i, box in enumerate(value):
        if isinstance(box, Mapping) or not isinstance(box, Sequence) or len(box) != 4:
            raise ValueError(f"{what}: box {i} must be [x0, y0, x1, y1]")
        try:
            x0, y0, x1, y1 = (float(v) for v in box)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{what}: box {i} must hold numbers") from exc
        if not (0.0 <= x0 < x1 <= width and 0.0 <= y0 < y1 <= height):
            raise ValueError(f"{what}: box {i} {box} is outside the image or empty ({width} x {height})")
        out.append([x0, y0, x1, y1])
    return out


def _check_record(record: Any, index: int, *, require_annotations: bool = True) -> dict[str, Any]:
    what = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{what} must be a mapping with id/image/label")
    for key in ("id", "image", "label") + (("count",) if require_annotations else ()):
        if key not in record:
            raise ValueError(f"{what} is missing {key!r}")
    rid, image, label = record["id"], record["image"], record["label"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{what}: id must match {_ID_RE.pattern}")
    if isinstance(image, str | Path):
        path = Path(image)
        if not path.is_file():
            raise ValueError(f"{what}: image file not found: {path}")
        image = Image.open(path)
        image.load()
    if not isinstance(image, Image.Image):
        raise ValueError(f"{what}: image must be a PIL.Image.Image or a file path")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE or max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"{what}: image sides outside {MIN_IMAGE_SIDE}..{MAX_IMAGE_SIDE} px: {image.size}")
    if not isinstance(label, str) or not _LABEL_RE.match(label.strip()):
        raise ValueError(f"{what}: label must be a non-empty string of at most {MAX_LABEL_CHARS} plain characters")
    item: dict[str, Any] = {"id": rid, "image": image.convert("RGB"), "label": label.strip().lower()}
    if "exemplars" in record and record["exemplars"] is not None:
        item["exemplars"] = _check_boxes(record["exemplars"], width, height, what)
    else:
        item["exemplars"] = []
    if "points" in record and record["points"] is not None:
        item["points"] = _check_points(record["points"], width, height, what)
    if "boxes" in record and record["boxes"] is not None:
        item["boxes"] = _check_object_boxes(record["boxes"], width, height, what)
        centres = [[(b[0] + b[2]) / 2, (b[1] + b[3]) / 2] for b in item["boxes"]]
        if "points" not in item:
            item["points"] = centres
        elif len(item["points"]) != len(centres):
            raise ValueError(f"{what}: {len(item['points'])} points but {len(centres)} boxes")
    if "count" in record:
        count = record["count"]
        if isinstance(count, bool) or not isinstance(count, int) or not 0 <= count <= MAX_COUNT:
            raise ValueError(f"{what}: count must be an int in 0..{MAX_COUNT}")
        if "points" in item and len(item["points"]) != count:
            raise ValueError(f"{what}: count {count} does not equal the number of points {len(item['points'])}")
        item["count"] = count
    elif "points" in item:
        item["count"] = len(item["points"])
    for key in ("source_file", "source_url", "original_size", "resize_ratio", "distractors"):
        if key in record:
            item[key] = record[key]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
    require_annotations: bool = True,
) -> dict[str, Any]:
    """Structural validation of a counting dataset; raises ValueError before any model import. A gold count
    (and, when present, points that agree with it and 0..3 in-image exemplar boxes) is required for evaluation
    and adaptation (`require_annotations=True`); the inference contract needs only `{id, image, label}`."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of {id, image, label, count, points, exemplars} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    counts: dict[str, int] = {}
    for index, record in enumerate(records):
        item = _check_record(record, index, require_annotations=require_annotations)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        counts[item["label"]] = counts.get(item["label"], 0) + 1
        checked.append(item)
    if not MIN_CLASSES <= len(counts) <= MAX_CLASSES:
        raise ValueError(f"{len(counts)} distinct labels; {MIN_CLASSES}..{MAX_CLASSES} are required")
    sides = [max(r["image"].size) for r in checked]
    gold = [r["count"] for r in checked if "count" in r]
    return {
        "records": checked,
        "n_records": len(checked),
        "classes": sorted(counts),
        "label_counts": dict(sorted(counts.items())),
        "image_side": {"min": min(sides), "max": max(sides)},
        "gold_count": {"min": min(gold), "max": max(gold), "total": sum(gold)} if gold else None,
        "with_points": sum(1 for r in checked if "points" in r),
        "with_boxes": sum(1 for r in checked if "boxes" in r),
        "with_exemplars": sum(1 for r in checked if r["exemplars"]),
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def class_names(records: Sequence[Mapping[str, Any]]) -> list[str]:
    """The sorted category vocabulary of a dataset (the captions the counter is asked)."""
    names = sorted({str(r["label"]) for r in records})
    if not names:
        raise ValueError("no labels in records")
    return names


def image_digest(image: Image.Image) -> str:
    return hashlib.sha256(image.convert("RGB").tobytes()).hexdigest()


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    digest = hashlib.sha256()
    for record in sorted(records, key=lambda r: str(r["id"])):
        digest.update(f"{record['id']}\t{record['label']}\t{record.get('count', '')}\t{image_digest(record['image'])}\n".encode())
    return digest.hexdigest()


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no image (by decoded-pixel digest) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = image_digest(record["image"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def category_coverage(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Which categories each split holds (the sample is stratified, so every split holds all eight; a BYOD
    split may not) and the gold-count range per split — observations, not assertions."""
    out = {}
    for name, records in splits.items():
        gold = [r.get("count", 0) for r in records]
        out[name] = {
            "categories": sorted({str(r["label"]) for r in records}),
            "gold_count": {"min": min(gold), "max": max(gold), "mean": round(sum(gold) / len(gold), 1)} if gold else None,
        }
    return out


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.2,
    test_fraction: float = 0.3,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded stratified shuffle of a BYOD dataset into train/validation/test after de-duplicating images."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    by_label: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        key = image_digest(record["image"])
        if key not in seen:
            seen.add(key)
            by_label.setdefault(record["label"], []).append(record)
    rng = random.Random(seed)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for label in sorted(by_label):
        pool = by_label[label]
        rng.shuffle(pool)
        n_test = max(1, round(len(pool) * test_fraction))
        n_val = round(len(pool) * val_fraction)
        splits["test"].extend(pool[:n_test])
        splits["validation"].extend(pool[n_test : n_test + n_val])
        splits["train"].extend(pool[n_test + n_val :])
    for part in splits.values():
        rng.shuffle(part)
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required")
    return splits


def _parse_json_cell(value: str | None, what: str) -> Any:
    if value is None or not value.strip():
        return None
    try:
        return json.loads(value)
    except ValueError as exc:
        raise ValueError(f"{what} must be JSON (a list of numbers)") from exc


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read `{id, image, label, count, points, exemplars}` records from a directory or a zip holding `labels.csv`
    (columns `id`, `file`, `label`, `count`; optional `points` and `exemplars` as JSON lists) beside the image
    files; images are decoded, never extracted to disk."""
    source = Path(path)
    if source.is_dir():
        table = (source / "labels.csv").read_text(encoding="utf-8")
        loader = lambda name: Image.open(source / name)  # noqa: E731
    elif source.is_file() and source.suffix.lower() == ".zip":
        archive = zipfile.ZipFile(source)
        members = {Path(n).name: n for n in archive.namelist()}
        if "labels.csv" not in members:
            raise ValueError("BYOD zip must contain labels.csv")
        table = archive.read(members["labels.csv"]).decode("utf-8")
        loader = lambda name: Image.open(io.BytesIO(archive.read(members[name])))  # noqa: E731
    else:
        raise ValueError("BYOD datasets must be a directory or a .zip holding labels.csv and the image files")
    rows = list(csv.DictReader(io.StringIO(table)))
    missing = {"id", "file", "label", "count"} - set(rows[0].keys() if rows else set())
    if missing:
        raise ValueError(f"labels.csv is missing columns {sorted(missing)}")
    out = []
    for row in rows:
        image = loader(row["file"])
        image.load()
        try:
            count = int(row["count"])
        except ValueError as exc:
            raise ValueError(f"{row['id']}: count must be an integer") from exc
        record: dict[str, Any] = {"id": row["id"], "image": image.convert("RGB"), "label": row["label"], "count": count}
        points = _parse_json_cell(row.get("points"), f"{row['id']}: points")
        exemplars = _parse_json_cell(row.get("exemplars"), f"{row['id']}: exemplars")
        boxes = _parse_json_cell(row.get("boxes"), f"{row['id']}: boxes")
        if points is not None:
            record["points"] = points
        if exemplars is not None:
            record["exemplars"] = exemplars
        if boxes is not None:
            record["boxes"] = boxes
        out.append(record)
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write the labels table of a split (id, file, label, count, exemplars, points, source) in the shape BYOD expects."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "file", "label", "count", "exemplars", "points", "boxes", "source_url"])
        writer.writeheader()
        for record in records:
            writer.writerow(
                {
                    "id": record["id"],
                    "file": record.get("source_file") or f"{record['id']}.jpg",
                    "label": record["label"],
                    "count": record.get("count", ""),
                    "exemplars": json.dumps(record.get("exemplars", [])),
                    "points": json.dumps(record["points"]) if "points" in record else "",
                    "boxes": json.dumps(record["boxes"]) if "boxes" in record else "",
                    "source_url": record.get("source_url", ""),
                }
            )
    return out

**Module 8/8:** `src/countgd_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""The CountGD pipeline: open-world counting from a text prompt, visual exemplar boxes or both, its evaluation
on a counting set, a bounded continuation of the detection-style training on the last decoder layers, and
the adapter artifact that reloads onto a freshly verified base.

What the model does (Amini-Naieni, Han and Zisserman, NeurIPS 2024): GroundingDINO's Swin-B image encoder and
BERT text encoder fused by a feature enhancer, with the exemplar boxes' pooled image features inserted as extra
text-side tokens (`*`), and a DETR-style decoder of 900 queries each predicting a box and a per-token similarity
logit. Upstream counts the queries whose best token score exceeds 0.23; the predicted "boxes" are tiny — the
model was trained on 2 x 2 px boxes centred on FSC-147's points — so a prediction is read as a **point**, the
count is the number of points, and no box overlap is measured.
"""
# ruff: noqa: E501  -- fleet pipeline module written at the 110-column fleet width; this repo lints at 100

from __future__ import annotations

import hashlib
import json
import math
import random
import time
from collections.abc import Callable, Mapping, Sequence
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torchvision.transforms.functional as TF
from PIL import Image

# standalone rewrite (build_notebook.py): `from .config import (` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .metrics import (` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .model import (` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .samples import MAX_IMAGE_SIDE, MIN_IMAGE_SIDE, _check_boxes` removed — names are kernel globals defined by the carried modules

ImageInput = str | Path | bytes | Image.Image

MAX_BATCH = 16  # images per count() call (each is one forward pass; the batch is a convenience, not a tensor batch)
DEFAULT_TRAINABLE_LAYERS = 2  # the last two of the six decoder layers train beside the shared box head
EXEMPLAR_CARRIER_WORD = "object"  # the caption word the exemplar tokens are attached to when no text is given
PARAMETER_COUNT = 233_362_816
MAX_EVAL_RECORDS = 5_000
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.countgd.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
INPUT_SCHEMA = {
    "image": "local path, bytes or PIL image; sides in [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE]; resized to a shortest side of 800 px (longest at most 1333) like upstream's test transform",
    "text": "the category to count, at most 64 plain characters (upstream appends ' .'); optional when exemplars are given",
    "exemplars": "0..3 boxes [x0, y0, x1, y1] in the input image's pixels around single instances; optional when text is given",
    "threshold": "the score a query must exceed to count, in (0, 1); upstream's 0.23 by default",
}


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


# ------------------------------------------------------------------ input checks


def _coerce_image(value: ImageInput) -> Image.Image:
    if isinstance(value, Image.Image):
        image = value
    elif isinstance(value, bytes):
        import io

        image = Image.open(io.BytesIO(value))
    elif isinstance(value, str | Path):
        text = str(value)
        if text.lower().startswith(("http://", "https://")):
            raise ValueError("remote URLs are not accepted; pass a local path, bytes or a PIL image")
        path = Path(text)
        if not path.is_file():
            raise FileNotFoundError(f"image file not found: {path}")
        image = Image.open(path)
    else:
        raise ValueError("image must be a local path, bytes or a PIL image")
    image.load()
    return image.convert("RGB")


def _check_size(image: Image.Image, what: str) -> None:
    w, h = image.size
    if min(w, h) < MIN_IMAGE_SIDE or max(w, h) > MAX_IMAGE_SIDE:
        raise ValueError(f"{what}: sides must be in [{MIN_IMAGE_SIDE}, {MAX_IMAGE_SIDE}] px, got {w} x {h}")


def _check_text(text: Any) -> str | None:
    if text is None:
        return None
    if not isinstance(text, str) or not text.strip():
        raise ValueError("text must be a non-empty string")
    cleaned = " ".join(text.strip().lower().rstrip(".").split())
    if len(cleaned) > MAX_TEXT_CHARS:
        raise ValueError(f"text must be at most {MAX_TEXT_CHARS} characters")
    if not all(ch.isalnum() or ch in " _-'" for ch in cleaned):
        raise ValueError("text must be plain words (letters, digits, spaces, hyphens, apostrophes)")
    return cleaned


def _check_threshold(value: Any) -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 < float(value) < 1.0:
        raise ValueError("threshold must be a number in (0, 1)")
    return float(value)


def _check_prompt(image: Image.Image, text: Any, exemplars: Any, what: str) -> tuple[str | None, list[list[float]]]:
    cleaned = _check_text(text)
    boxes = _check_boxes(exemplars, image.size[0], image.size[1], what) if exemplars else []
    if cleaned is None and not boxes:
        raise ValueError(f"{what}: give a text prompt, exemplar boxes, or both")
    return cleaned, boxes


def validate_inputs(
    images: Sequence[ImageInput] | ImageInput,
    *,
    text: str | None = None,
    exemplars: Sequence[Sequence[Sequence[float]]] | Sequence[Sequence[float]] | None = None,
    threshold: float = CONFIDENCE_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: the input manifest (schema, per-image observations, verdict), raising exactly what
    `count` would raise. `exemplars` is one box list for a single image or one list per image."""
    single = isinstance(images, str | Path | bytes | Image.Image)
    items = [images] if single else list(images)
    if not items:
        raise ValueError("images must contain at least one image")
    if len(items) > MAX_BATCH:
        raise ValueError(f"at most {MAX_BATCH} images per call")
    ratio = _check_threshold(threshold)
    per_image = _split_exemplars(exemplars, len(items))
    if names is not None and len(names) != len(items):
        raise ValueError("names must have one entry per image")
    observations = []
    for i, item in enumerate(items):
        image = _coerce_image(item)
        _check_size(image, f"image {i}")
        cleaned, boxes = _check_prompt(image, text, per_image[i], f"image {i}")
        observations.append({"id": names[i] if names else f"image-{i}", "mode": image.mode, "size": list(image.size), "text": cleaned, "exemplars": len(boxes)})
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": observations,
        "threshold": ratio,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def _split_exemplars(exemplars: Any, n: int) -> list[Any]:
    if exemplars is None:
        return [None] * n
    seq = list(exemplars)
    if seq and all(isinstance(b, Sequence) and len(b) == 4 and not isinstance(b[0], Sequence) for b in seq):
        if n != 1:
            raise ValueError("pass one exemplar list per image when counting several images")
        return [seq]
    if len(seq) != n:
        raise ValueError("exemplars must hold one box list per image")
    return seq


# ------------------------------------------------------------------ the pipeline


class CountGDPipeline:
    def __init__(
        self,
        model: Any,
        criterion: Any,
        tokenizer: Any,
        *,
        device: str | torch.device = "cpu",
        checkpoint_path: Path | str | None = None,
        checkpoint_source: str | None = None,
        manifest_verified: bool = False,
        weight_sha256: str | None = None,
        weight_size_bytes: int | None = None,
    ) -> None:
        self.model = model
        self.criterion = criterion
        self.tokenizer = tokenizer
        self.device = torch.device(device)
        self.checkpoint_path = Path(checkpoint_path) if checkpoint_path is not None else None
        self.checkpoint_source = checkpoint_source
        self.manifest_verified = manifest_verified
        self.weight_sha256 = weight_sha256
        self.weight_size_bytes = weight_size_bytes
        self.adapter: dict[str, Any] | None = None
        if hasattr(self.model, "parameters"):  # injected fakes in the offline tests carry none
            self.model.eval()
            for param in self.model.parameters():
                param.requires_grad_(False)

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str | torch.device | None = None,
        cache_dir: str | Path | None = None,
        weights_path: str | Path | None = None,
        weights_dir: str | Path | None = None,
        tokenizer_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> CountGDPipeline:
        """Load the one supported checkpoint (fleet snapshot directories → stage absent entries when allowed →
        verify against the manifests and the pinned digests → convert the audited pickle once when the served file
        is absent → load the safetensors file strictly)."""
        manifest_verified = False
        if weights_dir is not None:
            if weights_path is not None:
                raise ValueError("pass either weights_dir or weights_path, not both")
            stage_missing_files(weights_dir, allow_download=allow_download)
            verify_snapshot(weights_dir)
            weights_path, manifest_verified = weights_dir, True
        if tokenizer_dir is not None:
            stage_missing_tokenizer_files(tokenizer_dir, allow_download=allow_download)
            verify_tokenizer_snapshot(tokenizer_dir)
        model, criterion, tokenizer, target_device, _, metadata = load_components(device=device, cache_dir=cache_dir, weights_path=weights_path, tokenizer_path=tokenizer_dir, manifest_verified=manifest_verified, return_metadata=True)
        return cls(
            model,
            criterion,
            tokenizer,
            device=target_device,
            checkpoint_path=metadata.get("checkpoint_path"),
            checkpoint_source=metadata.get("checkpoint_source"),
            manifest_verified=metadata.get("manifest_verified", False),
            weight_sha256=metadata.get("weight_sha256"),
            weight_size_bytes=metadata.get("weight_size_bytes"),
        )

    # ------------------------------------------------------------------ tensors

    @staticmethod
    def _scale(image: Image.Image) -> tuple[float, tuple[int, int]]:
        w, h = image.size
        scale = SHORT_SIDE / min(w, h)
        if max(w, h) * scale > MAX_SIDE:
            scale = MAX_SIDE / max(w, h)
        return scale, (max(1, round(w * scale)), max(1, round(h * scale)))

    def _prepare(self, image: Image.Image) -> tuple[torch.Tensor, float]:
        scale, (nw, nh) = self._scale(image)
        resized = image if (nw, nh) == image.size else image.resize((nw, nh), Image.BILINEAR)
        pixels = TF.normalize(TF.to_tensor(resized), list(IMAGE_MEAN), list(IMAGE_STD))
        return pixels.to(self.device), scale

    def _forward(self, pixels: torch.Tensor, exemplars: Sequence[Sequence[float]], scale: float, text: str | None, targets: Sequence[Mapping[str, Any]] | None = None) -> Any:
        boxes = torch.tensor([[b[0] * scale, b[1] * scale, b[2] * scale, b[3] * scale] for b in exemplars], dtype=torch.float32, device=self.device).reshape(-1, 4)
        # Exemplar-only: upstream's evaluation removes the class word from a caption that still names other
        # classes and inserts the exemplar tokens where it stood; with a single-class caption that leaves no word
        # to anchor them, so the neutral carrier word below stands in and the exemplar tokens follow it.
        caption = f"{text} ." if text else f"{EXEMPLAR_CARRIER_WORD} ."
        if targets is None:
            return self.model(pixels.unsqueeze(0), [boxes], [torch.tensor([0], device=self.device)], captions=[caption])
        return self.model(pixels.unsqueeze(0), [boxes], [torch.tensor([0], device=self.device)], targets=[{**targets[0], "caption": caption}])

    @staticmethod
    def _read(out: Any, threshold: float, size: tuple[int, int]) -> dict[str, Any]:
        logits = out["pred_logits"][0].sigmoid()
        boxes = out["pred_boxes"][0]
        scores = logits.max(dim=-1).values
        keep = scores > threshold
        w, h = size
        kept = boxes[keep].detach().cpu().numpy().astype(np.float64)
        kept_scores = scores[keep].detach().cpu().numpy().astype(np.float64)
        xyxy = np.stack([(kept[:, 0] - kept[:, 2] / 2) * w, (kept[:, 1] - kept[:, 3] / 2) * h, (kept[:, 0] + kept[:, 2] / 2) * w, (kept[:, 1] + kept[:, 3] / 2) * h], axis=1) if len(kept) else np.zeros((0, 4))
        points = np.stack([kept[:, 0] * w, kept[:, 1] * h], axis=1) if len(kept) else np.zeros((0, 2))
        order = np.argsort(-kept_scores)
        return {
            "count": int(keep.sum()),
            "points": points[order].round(2).tolist(),
            "boxes": xyxy[order].round(2).tolist(),
            "scores": kept_scores[order].round(4).tolist(),
            "max_score": float(scores.max()),
            "queries": int(scores.numel()),
        }

    # ------------------------------------------------------------------ inference contract

    def count(
        self,
        images: Sequence[ImageInput] | ImageInput,
        *,
        text: str | None = None,
        exemplars: Sequence[Sequence[Sequence[float]]] | Sequence[Sequence[float]] | None = None,
        threshold: float = CONFIDENCE_THRESHOLD,
    ) -> dict[str, Any]:
        """Count the objects a text prompt and/or 0..3 exemplar boxes describe: one result per image with the
        count, the predicted points (box centres in the input image's pixels, best first), the predicted boxes and
        the scores. Text alone, exemplars alone, or both, as upstream allows."""
        manifest = validate_inputs(images, text=text, exemplars=exemplars, threshold=threshold)
        single = isinstance(images, str | Path | bytes | Image.Image)
        items = [images] if single else list(images)
        per_image = _split_exemplars(exemplars, len(items))
        cleaned = _check_text(text)
        results = []
        for i, item in enumerate(items):
            image = _coerce_image(item)
            boxes = _check_boxes(per_image[i], image.size[0], image.size[1], f"image {i}") if per_image[i] else []
            pixels, scale = self._prepare(image)
            with torch.no_grad():
                out = self._forward(pixels, boxes, scale, cleaned)
            entry = self._read(out, manifest["threshold"], image.size)
            entry.update({"input_size": list(image.size), "model_input_size": [int(pixels.shape[-1]), int(pixels.shape[-2])], "text": cleaned, "exemplars": boxes})
            results.append(entry)
        return {
            "results": results,
            "threshold": manifest["threshold"],
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "model_license": MODEL_LICENSE,
        }

    # ------------------------------------------------------------------ evaluation

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        threshold: float = CONFIDENCE_THRESHOLD,
        use_text: bool = True,
        use_exemplars: bool = True,
        progress: Callable[[int, int], None] | None = None,
    ) -> dict[str, Any]:
        """Count every record of a validated set with its own label and exemplars and score the counts (MAE,
        RMSE, NAE, per class), the localisation of the predicted points where gold points exist and the extent of
        the predicted boxes (IoU matching) where gold object boxes exist."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if not use_text and not use_exemplars:
            raise ValueError("evaluate with text, exemplars, or both")
        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        ratio = _check_threshold(threshold)
        rows, loc_rows, box_rows, per_image = [], [], [], []
        started = time.perf_counter()
        for i, record in enumerate(checked):
            exemplars = record["exemplars"] if use_exemplars else []
            text = record["label"] if use_text else None
            if not exemplars and text is None:
                raise ValueError(f"{record['id']}: no exemplars to count with and text is off")
            pixels, scale = self._prepare(record["image"])
            with torch.no_grad():
                out = self._forward(pixels, exemplars, scale, text)
            entry = self._read(out, ratio, record["image"].size)
            rows.append({"id": record["id"], "label": record["label"], "gold": record["count"], "predicted": entry["count"]})
            loc = None
            if "points" in record:
                loc = match_points(entry["points"], record["points"], match_radius(record))
                loc_rows.append(loc)
            box = None
            if "boxes" in record:
                box = match_boxes(entry["boxes"], record["boxes"])
                box_rows.append(box)
            per_image.append({"id": record["id"], "label": record["label"], "gold": record["count"], "predicted": entry["count"], "max_score": entry["max_score"], "localisation": loc, "boxes": {k: box[k] for k in ("tp", "fp", "fn")} if box else None})
            if progress is not None:
                progress(i + 1, len(checked))
        result = counting_metrics(rows)
        result["localisation"] = localisation_metrics(loc_rows)
        result["boxes"] = box_metrics(box_rows)
        result["per_image"] = per_image
        result["threshold"] = ratio
        result["prompt"] = {"text": use_text, "exemplars": use_exemplars}
        result["seconds"] = round(time.perf_counter() - started, 2)
        result["verdict"] = "measured" if len(checked) >= MIN_SCORED_RECORDS else "small-sample"
        result["adapted"] = self.adapter is not None
        return result

    # ------------------------------------------------------------------ adaptation contract

    def _trainable_names(self, trainable_layers: int) -> list[str]:
        if isinstance(trainable_layers, bool) or not isinstance(trainable_layers, int) or not 0 <= trainable_layers <= DECODER_LAYERS:
            raise ValueError(f"trainable_layers must be an int in 0..{DECODER_LAYERS}")
        prefixes = tuple(f"transformer.decoder.layers.{i}." for i in range(DECODER_LAYERS - trainable_layers, DECODER_LAYERS))
        names = [n for n, _ in self.model.named_parameters() if n.startswith(prefixes) or n.startswith("bbox_embed.0.") or n.startswith("transformer.decoder.norm.")]
        return names

    def _targets(self, record: Mapping[str, Any]) -> dict[str, Any]:
        """Training boxes in normalised cx, cy, w, h: the gold object boxes when the record has them, else upstream's
        2 x 2 px boxes centred on the gold points (FSC-147 annotates points only)."""
        w, h = record["image"].size
        if record.get("boxes"):
            b = torch.tensor(record["boxes"], dtype=torch.float32)
            boxes = torch.stack([(b[:, 0] + b[:, 2]) / 2 / w, (b[:, 1] + b[:, 3]) / 2 / h, (b[:, 2] - b[:, 0]) / w, (b[:, 3] - b[:, 1]) / h], dim=1)
            return {"boxes": boxes.to(self.device), "labels": torch.zeros(len(b), dtype=torch.long, device=self.device)}
        points = record.get("points")
        if not points:
            raise ValueError(f"{record['id']}: adaptation needs gold points (count alone cannot place the training boxes)")
        pts = torch.tensor(points, dtype=torch.float32)
        boxes = torch.stack([pts[:, 0] / w, pts[:, 1] / h, torch.full((len(pts),), TARGET_BOX_PX / w), torch.full((len(pts),), TARGET_BOX_PX / h)], dim=1)
        return {"boxes": boxes.to(self.device), "labels": torch.zeros(len(pts), dtype=torch.long, device=self.device)}

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 3,
        lr: float = 1e-5,
        trainable_layers: int = DEFAULT_TRAINABLE_LAYERS,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded continuation of CountGD's training on a validated counting set with gold points.

        Trains the last `trainable_layers` decoder layers, the decoder's final norm and the shared box head on
        upstream's own objective — the token-level sigmoid focal loss and the L1 box loss of `SetCriterion`
        after Hungarian matching, over the final and every intermediate decoder output — with the caption
        `<label> .`, the record's exemplars and, as target boxes, the record's gold object boxes when it has them
        (the synthetic scenes) or upstream's 2 x 2 px boxes centred on the gold points (FSC-147), one image per
        step; AdamW at a fixed learning rate (weight decay 1e-4), gradient clipping at 0.1 (upstream's), seeded
        order, no scheduler, no augmentation. Epoch 0 records the frozen model's validation MAE; every epoch is
        scored on the validation set and the epoch with the lowest validation MAE is kept — the frozen model
        itself when nothing beats it. Transactional: any failure restores the base tensors."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        names = self._trainable_names(trainable_layers)
        train_checked = validate_dataset(train, min_records=1)["records"]
        val_checked = validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        for record in train_checked:
            if not record.get("points"):
                raise ValueError(f"{record['id']}: adaptation needs gold points")
        model = self.model
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
        started = time.perf_counter()
        weight_dict = self.criterion.weight_dict

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            result = self.evaluate(val_checked)
            return {k: result[k] for k in ("mae", "rmse", "nae", "n")} | {"localisation_f1": result["localisation"]["f1"], "box_f1": result["boxes"]["f1"]}

        def key(entry: dict[str, Any]) -> float:
            return -entry["val"]["mae"] if entry["val"] else -math.inf

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress is not None:
            progress(entry)
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        initial_state = {k: v.clone() for k, v in best_state.items()}
        best_epoch = 0
        rng = random.Random(seed)
        torch.manual_seed(seed)
        try:
            for epoch in range(1, epochs + 1):
                model.train()
                order = list(range(len(train_checked)))
                rng.shuffle(order)
                losses = []
                for index in order:
                    record = train_checked[index]
                    pixels, scale = self._prepare(record["image"])
                    target = self._targets(record)
                    optimiser.zero_grad(set_to_none=True)
                    out = self._forward(pixels, record["exemplars"], scale, record["label"], targets=[target])
                    loss_dict = self.criterion(out, [target], [[record["label"]]], [f"{record['label']} ."])
                    loss = sum(loss_dict[k] * weight_dict[k] for k in loss_dict if k in weight_dict)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 0.1)
                    optimiser.step()
                    losses.append(float(loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": float(np.mean(losses)), "val": score_val()}
                history.append(entry)
                if progress is not None:
                    progress(entry)
                if val_checked:
                    if key(entry) > key(history[best_epoch]):
                        best_epoch = epoch
                        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                else:
                    best_epoch = epoch
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        except BaseException:
            self._overlay(initial_state)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            raise
        self._overlay(best_state)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "objective": "CountGD's own: token sigmoid focal loss + L1 box loss after Hungarian matching, final and intermediate decoder outputs",
            "trainable_layers": trainable_layers,
            "n_trainable": int(n_trainable),
            "n_total": int(sum(p.numel() for p in model.parameters())),
            "epochs": epochs,
            "lr": lr,
            "batch_size": 1,
            "seed": seed,
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "best_epoch": best_epoch,
            "selection": "lowest validation MAE (validation split)" if val_checked else "final epoch (no validation split)",
            "seconds": round(time.perf_counter() - started, 2),
            "history": history,
            "trainable_names": names,
        }
        return dict(self.adapter)

    def _overlay(self, tensors: Mapping[str, torch.Tensor]) -> None:
        """Copy tensors into the model's parameters by name (shared modules — the tied box heads — follow)."""
        params = dict(self.model.named_parameters())
        with torch.no_grad():
            for name, value in tensors.items():
                if name not in params:
                    raise ValueError(f"tensor {name} is not a parameter of the base")
                if tuple(value.shape) != tuple(params[name].shape):
                    raise ValueError(f"tensor {name} has shape {tuple(value.shape)}, base has {tuple(params[name].shape)}")
                params[name].copy_(value.to(params[name].device, params[name].dtype))

    # ------------------------------------------------------------------ artifact

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted tensors as safetensors plus a base manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        params = dict(self.model.named_parameters())
        tensors = {k: params[k].detach().cpu().contiguous() for k in self.adapter["trainable_names"]}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": DEFAULT_MODEL_KEY, "weight_file": MODEL_FILENAME, "weight_sha256": MODEL_SHA256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [{"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": _sha256_file(weights_path)}],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite
        exactly the parameters it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path = _check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256_file(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        expected = sorted(self._trainable_names(manifest["adapter"]["trainable_layers"]))
        if sorted(manifest["tensors"]) != expected:
            raise ValueError("artifact tensor list does not match its recorded configuration")
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        self._overlay(tensors)
        self.model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": list(manifest["tensors"]), "history": manifest.get("history", [])}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | torch.device | None = None,
        weights_dir: str | Path | None = None,
        tokenizer_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> CountGDPipeline:
        """A fresh pipeline from the pinned base with an adapter overlaid."""
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, tokenizer_dir=tokenizer_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe


def _check_artifact_manifest(root: Path, manifest: Mapping[str, Any]) -> Path:
    """Refuse an artifact whose manifest is not exactly the one this package writes. Nothing is deserialised
    here; the digest check that follows detects corruption or drift relative to the adjacent manifest."""
    if manifest.get("format") != ARTIFACT_FORMAT:
        raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
    if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
        raise ValueError(f"artifact format_version {manifest.get('format_version')!r} is not the supported {ARTIFACT_FORMAT_VERSION!r}")
    base = manifest.get("base_model", {})
    if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (MODEL_ID, MODEL_REVISION, MODEL_SHA256):
        raise ValueError("artifact was adapted from a different base model, revision or weight file")
    if base.get("weight_file", MODEL_FILENAME) != MODEL_FILENAME:
        raise ValueError("artifact was adapted from a different base weight file")
    files = manifest.get("files")
    if not isinstance(files, list) or len(files) != 1:
        raise ValueError("artifact manifest must list exactly one file")
    entry = files[0]
    if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
        raise ValueError(f"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}")
    weights_path = (root / entry["path"]).resolve()
    if weights_path.parent != root.resolve():
        raise ValueError("artifact weight path must resolve inside the artifact directory")
    adapter = manifest.get("adapter")
    layers = adapter.get("trainable_layers") if isinstance(adapter, Mapping) else None
    if isinstance(layers, bool) or not isinstance(layers, int) or not 0 <= layers <= DECODER_LAYERS:
        raise ValueError("artifact manifest does not record an in-range integer trainable_layers")
    if not isinstance(manifest.get("tensors"), list):
        raise ValueError("artifact manifest must list its tensors")
    return weights_path


def load_pipeline(**kwargs: Any) -> CountGDPipeline:
    return CountGDPipeline.from_pretrained(**kwargs)


def evaluation_report(result: Mapping[str, Any], expected: Sequence[int] | None = None, *, sample_kind: str = "synthetic") -> dict[str, Any]:
    """The per-call reading of a `count` result: the counts, and against `expected` counts the absolute
    errors, with a `sample-sanity` verdict on drawings (plumbing evidence) or `measured` on a scored set."""
    rows = result.get("results", [])
    if not rows:
        raise ValueError("no results to report")
    counts = [int(r["count"]) for r in rows]
    metrics = [{"id": "count", "value": counts, "definition": "objects counted per image (queries above the threshold)"}]
    if expected is not None:
        if len(expected) != len(rows):
            raise ValueError("expected must have one count per result")
        errors = [abs(c - int(e)) for c, e in zip(counts, expected, strict=True)]
        metrics.append({"id": "mae", "value": float(np.mean(errors)), "definition": METRIC_DEFINITIONS_MAE})
        metrics.append({"id": "exact", "value": int(sum(1 for e in errors if e == 0)), "definition": "images counted exactly"})
    return {
        "metrics": metrics,
        "n": len(rows),
        "sample_kind": sample_kind,
        "verdict": "sample-sanity" if sample_kind.startswith("synthetic") or len(rows) < MIN_SCORED_RECORDS else "measured",
        "note": "a count above a fixed score threshold; nothing here is calibrated and a drawing is plumbing evidence, not a measurement",
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


METRIC_DEFINITIONS_MAE = "mean over images of |predicted count - expected count|; lower is better"

__all__ = [
    "ARTIFACT_FORMAT",
    "DEFAULT_TRAINABLE_LAYERS",
    "INPUT_SCHEMA",
    "MAX_BATCH",
    "MAX_EXEMPLARS",
    "MAX_IMAGE_SIDE",
    "MIN_IMAGE_SIDE",
    "PARAMETER_COUNT",
    "CountGDPipeline",
    "evaluation_report",
    "load_pipeline",
    "validate_inputs",
]

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `2`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at Space revision `6e82e59569a8…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `CountGDPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, tokenizer_dir=TOKENIZER_WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The package also pins a second snapshot `bert-base-uncased` (6 files), carried and verified the same way. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "countgd",
  "modelId": "nikigoli/countgd",
  "repoType": "space",
  "revision": "6e82e59569a84ee5c6aafa35d396f2d2bee57be2",
  "files": [
    {
      "path": "README.md",
      "bytes": 526,
      "sha256": "f67b0f10bb3d47ec0b810624f2e1c71ffcc026458936e90e5e203fe08408d150"
    },
    {
      "path": "checkpoint_best_regular.pth",
      "bytes": 1250122522,
      "sha256": "c1bab864b17db345b4c6e3aaabb5765bc2c0a90d0bc8defb5e664a74a50aa126"
    }
  ],
  "totalBytes": 1250123048
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})

TOKENIZER_MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "bert-base-uncased",
  "modelId": "google-bert/bert-base-uncased",
  "revision": "86b5e0934494bd15c9632b12f734a8a67f723594",
  "files": [
    {
      "path": "LICENSE",
      "bytes": 11356,
      "sha256": "43070e2d4e532684de521b885f385d0841030efa2b1a20bafb76133a5e1379c1"
    },
    {
      "path": "README.md",
      "bytes": 10517,
      "sha256": "9187b6018ea0010d884e78e098e328faa1b88b301570d0cce606bb35e4067e17"
    },
    {
      "path": "config.json",
      "bytes": 570,
      "sha256": "7160e1553ad2ca51d8c1cb066be533db31826e12d173824c1bb0cb1a4f187d20"
    },
    {
      "path": "tokenizer.json",
      "bytes": 466062,
      "sha256": "ce64fce797c24f68df90b40a3f74f579b336a493db14bd583fd520ea0d8c9a98"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 48,
      "sha256": "a025160ef0431f1a392f6f050c1310f4c5d9fb6f275932dbccba73c4d214bf10"
    },
    {
      "path": "vocab.txt",
      "bytes": 231508,
      "sha256": "07eced375cec144d27c900241f3e339478dec958f92fddbc551f295c992038a3"
    }
  ],
  "totalBytes": 720061
}

if (TOKENIZER_MANIFEST['modelId'], TOKENIZER_MANIFEST['revision']) != (TOKENIZER_MODEL_ID, TOKENIZER_REVISION):
    raise RuntimeError('inline bert-base-uncased manifest does not name the identity carried by the pipeline module')
TOKENIZER_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(TOKENIZER_WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(TOKENIZER_MANIFEST, handle, indent=2)
fetched_bert_base_uncased = stage_missing_tokenizer_files(TOKENIZER_WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(TOKENIZER_WEIGHTS_DIR), 'fetched': fetched_bert_base_uncased})
_extra = verify_tokenizer_snapshot(TOKENIZER_WEIGHTS_DIR)
_extra_files = _extra.get('files', []) if isinstance(_extra, dict) else []
print({'verified_files_bert_base_uncased': len(_extra_files) if isinstance(_extra_files, list) else _extra_files})
pipe = CountGDPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, tokenizer_dir=TOKENIZER_WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Synthetic scenes, FSC-147 photographs and splits

`build_synthetic_dataset` draws the seeded synthetic splits — scene `i` of a split is `synthetic_scene(base + i)` with disjoint seed ranges per split, 24 / 8 / 12 scenes of 512×384 pixels, each holding 6..40 **targets** of one colour and shape (the label, e.g. `blue circle`) and 4..20 **distractors** of another colour *and* shape, with one gold box per target and three of the targets' boxes as exemplars. `fetch_corpus` returns the 80 pinned FSC-147 photographs from the cache under `weights/fsc147-subset/` or the Hub mirror — every cached file re-hashed, every fetched file refused on any byte-size or SHA-256 mismatch — `read_corpus` decodes them into records with their gold points and three exemplar boxes, and `build_sample_dataset` draws a seeded stratified split per category (5 / 2 / 3 of 10 → 40 / 16 / 24); only its 24 test photographs are scored in this notebook. `validate_dataset` checks every record against the contract, `check_split_disjoint` asserts no image (by decoded-pixel digest) is shared between splits, and the training scenes' labels table is written to `outputs/countgd_object_counting_train.csv` in the shape BYOD expects.

Look for: 44 synthetic scenes with their gold-count range, 80 photographs over eight categories, one digest per split, and four refusal probes — a duplicate id, an image over the side ceiling, a count that disagrees with its boxes, and a box outside its image — each rejected before the model does anything.

In [ ]:
import json
import time

import numpy as np
from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
BYOD_ZIP_PATH = ''  # @param {type:"string"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD and BYOD_ZIP_PATH:
    byod_zip = Path(BYOD_ZIP_PATH)  # a location field: no upload dialog (NOTEBOOK_SPEC 2.1 EXE2)
    file_name = byod_zip.name
elif USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_zip = Path('work') / 'byod.zip'
    byod_zip.parent.mkdir(parents=True, exist_ok=True)
    byod_zip.write_bytes(payload)
if USE_BYOD:
    records = load_byod_dataset(byod_zip)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    fsc_test = []
else:
    splits = build_synthetic_dataset(SYNTHETIC_SPLIT)
    data_source = f'synthetic counting scenes (seeds {SYNTHETIC_SEED_BASE}, sizes {SYNTHETIC_SPLIT}, {SYNTHETIC_SIZE[0]}x{SYNTHETIC_SIZE[1]} px)'
    corpus_files = fetch_corpus(cache_dir='weights/fsc147-subset')
    corpus = read_corpus(corpus_files)
    fsc_splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    fsc_manifest = validate_dataset(fsc_splits['test'])
    fsc_test = fsc_manifest['records']
    print({'fsc147': {'photographs': len(corpus), 'bytes': sum(len(v) for v in corpus_files.values()), 'categories': sorted({r['label'] for r in corpus}), 'splits': check_split_disjoint(fsc_splits), 'test_gold_count': fsc_manifest['gold_count'], 'test_digest': fsc_manifest['digest'][:16] + '...'}})
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
write_dataset_csv(train_records, 'outputs/countgd_object_counting_train.csv')
print({'data_source': data_source, 'splits': disjoint})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'labels': len(manifest['classes']), 'gold_count': manifest['gold_count'], 'with_boxes': manifest['with_boxes'], 'digest': manifest['digest'][:16] + '...'}})

example = train_records[0]
probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:4]],
    'image over the side ceiling': [{**example, 'image': Image.new('RGB', (MAX_IMAGE_SIDE + 1, 64)), 'boxes': None, 'points': None, 'exemplars': None, 'count': 0}],
    'count disagrees with its boxes': [{**example, 'count': example['count'] + 1, 'points': None}],
    'box outside its image': [{**example, 'boxes': [[0, 0, 9999, 10]] + example['boxes'][1:], 'points': None}],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe, min_records=1)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Count one scene three ways through the inference contract

The demo scene (`synthetic_scene(DEMO_SEED)`) holds 35 blue circles among 16 green triangles. `validate_inputs` applies exactly the checks the public operations apply — image decoding and sides, the prompt's characters and length, the exemplars' count, extent and placement — and returns an input manifest, here with one entry per prompt mode; a remote URL is validated too and its rejection recorded as a finding. `count` then runs the scene three times: with the **text** `blue circle` alone, with the three **exemplar boxes** alone (the caption is then the neutral word `object` with the exemplar tokens attached), and with **both**. Each result carries the count, one box and one point per counted object, their scores and the highest score of the 900 queries. The predicted boxes are drawn onto the scene and saved as PNG. The per-call `evaluation_report` against the scene's gold count is `sample-sanity` — plumbing evidence, not a measurement; Section 6 measures.

Look for what each prompt counts. The build record counted 35, 51 and 35: text and text-plus-exemplars count the circles, while three exemplar boxes alone count 51 — every drawn shape, circles and triangles alike, because three boxes around small flat blobs describe *a small flat blob* at least as well as they describe *a blue circle*. That is the exemplar mode's known weakness on look-alike distractors, and the reason the fine-tune below trains with both prompts on scenes full of distractors.

In [ ]:
demo = synthetic_scene(DEMO_SEED)
print({'ceilings': {'MAX_EXEMPLARS': MAX_EXEMPLARS, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_BATCH': MAX_BATCH, 'MAX_COUNT': MAX_COUNT, 'SHORT_SIDE': SHORT_SIDE, 'MAX_SIDE': MAX_SIDE, 'threshold': CONFIDENCE_THRESHOLD, 'device': str(pipe.device)}})
PROMPTS = {'text': {'text': demo['label']}, 'exemplars': {'exemplars': demo['exemplars']}, 'both': {'text': demo['label'], 'exemplars': demo['exemplars']}}
input_manifest = validate_inputs(demo['image'], text=demo['label'], exemplars=demo['exemplars'], names=[demo['id']])
input_manifest['prompt_modes'] = {mode: validate_inputs(demo['image'], **kwargs)['inputs'][0] for mode, kwargs in PROMPTS.items()}
try:
    validate_inputs('https://example.invalid/not-allowed.png', text='cells')
except ValueError as exc:
    input_manifest['findings'].append({'input': 'remote-url-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/countgd_object_counting_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'scene': demo['id'], 'label': demo['label'], 'gold': demo['count'], 'distractors': demo['distractors'], 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})


def draw_counts(image, entry, colour=(255, 0, 255)):
    canvas = image.copy()
    pen = ImageDraw.Draw(canvas)
    for box in entry['boxes']:
        pen.rectangle(box, outline=colour, width=2)
    for x0, y0, x1, y1 in entry['exemplars']:
        pen.rectangle([x0, y0, x1, y1], outline=(0, 0, 0), width=3)
    return canvas


t0 = time.perf_counter()
demo_frozen = {mode: pipe.count(demo['image'], **kwargs)['results'][0] for mode, kwargs in PROMPTS.items()}
again = pipe.count(demo['image'], **PROMPTS['both'])['results'][0]
for mode, entry in demo_frozen.items():
    draw_counts(demo['image'], entry).save(f'outputs/countgd_object_counting_demo_frozen_{mode}.png')
    print(mode, '->', {'count': entry['count'], 'gold': demo['count'], 'max_score': round(entry['max_score'], 3), 'box_f1': round(box_metrics([match_boxes(entry['boxes'], demo['boxes'])])['f1'], 3)})
w, h = demo['image'].size
checks = {
    'one_box_and_point_per_count': all(len(e['boxes']) == len(e['points']) == e['count'] for e in demo_frozen.values()),
    'scores_above_threshold': all(min(e['scores'], default=1.0) > CONFIDENCE_THRESHOLD for e in demo_frozen.values()),
    'points_inside_the_image': all(0 <= x <= w and 0 <= y <= h for e in demo_frozen.values() for x, y in e['points']),
    'same_input_same_output': again['count'] == demo_frozen['both']['count'] and again['boxes'] == demo_frozen['both']['boxes'],
    'exemplars_echoed_in_input_pixels': demo_frozen['exemplars']['exemplars'] == [[float(v) for v in b] for b in demo['exemplars']],
}
if not all(checks.values()):
    raise RuntimeError(f'inference output failed a sanity check: {checks}')
demo_report = evaluation_report({'results': list(demo_frozen.values())}, [demo['count']] * 3, sample_kind='synthetic')
print({'checks': checks, 'seconds': round(time.perf_counter() - t0, 2), 'report': {m['id']: m['value'] for m in demo_report['metrics']}, 'verdict': demo_report['verdict']})

## 6. Baselines and the frozen model on held-out scenes and photographs

`pipe.evaluate` counts every record with its own label and its own exemplars (both prompts, as the fine-tune will train) and reports the **count error** (MAE, RMSE, the normalised absolute error, the fraction counted exactly), the **point localisation** (precision, recall and F1 of the predicted points matched one-to-one to the gold points within half the mean exemplar side) and, where gold object boxes exist, the **box IoU** reading (precision, recall and F1 of the predicted boxes matched one-to-one to the gold boxes at IoU ≥ 0.5, the mean IoU of the matched pairs, and the mean over gold boxes of the best IoU any prediction reaches). Two non-neural baselines are scored by the same code on the same records: the **mean-count baseline** answers every image with the rounded mean gold count of the training records (the counting analogue of a majority floor), and the **template matcher** correlates the mean of the image's own exemplar crops over the image and counts the peaks above 0.6 — a classical, learning-free use of the same three boxes the model gets.

Two sets, read separately. On the **12 held-out synthetic scenes** the build record measured the frozen model at MAE 7.42 (RMSE 9.97), box F1 0.453 and point F1 0.862, against MAE 10.00 for the mean count and 14.50 for the template matcher — the distractors are where its count error comes from. On the **24 FSC-147 test photographs** (categories it never saw in training) it measured MAE 4.54 against 16.21 and 34.00. Read the per-image rows: a few dense images carry most of an RMSE.

In [ ]:
CK = ('mae', 'rmse', 'nae', 'exact_fraction')


def brief(result):
    row = {k: round(result[k], 3) for k in CK}
    row['point_f1'] = None if result['localisation']['f1'] is None else round(result['localisation']['f1'], 3)
    boxes = result.get('boxes') or {}
    if boxes.get('f1') is not None:
        row.update({'box_f1': round(boxes['f1'], 3), 'box_mean_matched_iou': round(boxes['mean_matched_iou'] or 0.0, 3), 'box_mean_best_iou': round(boxes['mean_best_iou'], 3)})
    return row


t0 = time.perf_counter()
frozen_syn = pipe.evaluate(test_records)
frozen_fsc = pipe.evaluate(fsc_test) if fsc_test else None
frozen_seconds = round(time.perf_counter() - t0, 1)
baselines = {'synthetic': {'mean_count': mean_count_baseline(train_records, test_records), 'template_matching': template_matching_baseline(test_records)}}
if fsc_test:
    baselines['fsc147'] = {'mean_count': mean_count_baseline(fsc_splits['train'], fsc_test), 'template_matching': template_matching_baseline(fsc_test)}
print({'frozen_synthetic_test': brief(frozen_syn), 'n': frozen_syn['n'], 'verdict': frozen_syn['verdict'], 'seconds': frozen_seconds})
if frozen_fsc:
    print({'frozen_fsc147_test': brief(frozen_fsc), 'n': frozen_fsc['n'], 'verdict': frozen_fsc['verdict']})
for dataset, rows in baselines.items():
    for name, base in rows.items():
        print({dataset + '/' + name: {k: round(base[k], 3) for k in CK}, 'note': base['baseline']})
print({'definitions': {**frozen_syn['definitions'], **frozen_syn['boxes']['definitions']}})
for row in frozen_syn['per_image'][:6]:
    print({k: row[k] for k in ('id', 'label', 'gold', 'predicted')}, {'boxes': row['boxes']})
assert frozen_syn['boxes']['n'] == len(test_records) and frozen_syn['localisation']['n'] == len(test_records)

## 7. Bounded counting fine-tune

`pipe.adapt` continues CountGD's own training on the training scenes: the last `TRAINABLE_LAYERS` decoder layers, the decoder's final LayerNorm and the shared box head train — two layers by default, 3,619,584 of 233,362,816 parameters; the Swin-B backbone, BERT, the feature enhancer, the encoder and the first four decoder layers stay frozen — on upstream's objective: the token-level sigmoid focal loss and the L1 box loss after Hungarian matching, over the final and every intermediate decoder output, with the caption `<label> .`, the scene's three exemplars and the **gold object boxes** as targets (FSC-147 records, which have points only, would train on upstream's 2×2-pixel point boxes instead). One scene per step, AdamW at a fixed learning rate with weight decay 1e-4, gradient clipping at 0.1 (upstream's), seeded order, no scheduler, no augmentation. Epoch 0 records the frozen model's validation count error; every epoch is scored on the eight validation scenes and the epoch with the lowest validation MAE is kept — so the selection can return the frozen model itself (epoch 0) when nothing beats it, and never returns a worse one. The validation rows also print point and box F1: they are read, not selected on.

The build record's recipe sweep (`docs/release-verification.md`) found `1e-5` too small to move anything in a few epochs, `1e-4` for three epochs on twelve scenes moved the test MAE only from 7.25 to 5.62, and `2e-4` for four epochs on the 24 scenes moved the validation MAE from 9.38 to 0.75 at epoch 3 before it rose again at epoch 4 (2.75) — so the selector kept epoch 3. The default below is that configuration; expect the validation MAE to fall and then turn, and the kept epoch to be the lowest one, not the last.

In [ ]:
EPOCHS = 4  # @param {type:"integer"}
LEARNING_RATE = 2e-4  # @param {type:"number"}
TRAINABLE_LAYERS = 2  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 3)}
    if entry.get('val'):
        row.update({'val_' + k: None if entry['val'][k] is None else round(entry['val'][k], 3) for k in ('mae', 'rmse', 'localisation_f1', 'box_f1')})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, trainable_layers=TRAINABLE_LAYERS, seed=0, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'objective': adapt_result['objective'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})
val_history = {h['epoch']: h['val']['mae'] for h in adapt_result['history']}
assert val_history[adapt_result['best_epoch']] <= val_history[0]  # the selector never returns an epoch worse than the frozen model

## 8. Held-out evaluation after the fine-tune

The test scenes and the FSC-147 photographs were never used for training or epoch selection, and no image appears in two splits. The adapted model is scored exactly as the frozen model was in Section 6 — same records, same prompts, same threshold — and the systems are put side by side: on the synthetic test the mean count, the template matcher, the frozen model and the adapted one on count error, point F1 and box IoU; on FSC-147 the same on count error and point F1. The evaluation report is written as JSON. Read it in this order: the **validation MAE** that selected the epoch, then the **synthetic test** (the build record measured MAE 7.42 → 0.92 and box F1 0.453 → 0.550), then **FSC-147** (MAE 4.54 → 3.50) — the check that training on coloured shapes did not break counting on photographs. Twelve scenes and 24 photographs from one seeded draw give **no dispersion estimate**, a count error moves in steps of one object per image, and the synthetic scenes are easy in ways real images are not: this is sample-sanity evidence that the adaptation contract works, not a benchmark, and not a claim about your images until you measure them.

In [ ]:
adapted_syn = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
adapted_fsc = pipe.evaluate(fsc_test) if fsc_test else None


def side_by_side(dataset, frozen, adapted):
    table = {name: {k: round(base[k], 3) for k in CK} for name, base in baselines[dataset].items()}
    table.update({'frozen': brief(frozen), 'adapted': brief(adapted)})
    return table


comparison = {
    'synthetic_test': side_by_side('synthetic', frozen_syn, adapted_syn),
    'synthetic_delta_vs_frozen': {k: round(v - brief(frozen_syn)[k], 3) for k, v in brief(adapted_syn).items() if v is not None and brief(frozen_syn).get(k) is not None},
    'validation_mae': {'frozen': round(val_history[0], 3), 'selected_epoch': adapt_result['best_epoch'], 'selected': round(val_history[adapt_result['best_epoch']], 3), 'rescored': round(adapted_val['mae'], 3)},
}
if fsc_test:
    comparison['fsc147_test'] = side_by_side('fsc147', frozen_fsc, adapted_fsc)
    comparison['fsc147_delta_vs_frozen'] = {k: round(v - brief(frozen_fsc)[k], 3) for k, v in brief(adapted_fsc).items() if v is not None and brief(frozen_fsc).get(k) is not None}
for key, row in comparison.items():
    print({key: row})
strip = lambda result: None if result is None else {k: v for k, v in result.items() if k != 'definitions'}  # noqa: E731
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': DEFAULT_MODEL_KEY, 'weight_sha256': MODEL_SHA256},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'synthetic': {'frozen_test': strip(frozen_syn), 'adapted_test': strip(adapted_syn), 'adapted_validation': strip(adapted_val), 'baselines': baselines['synthetic']},
    'fsc147': {'frozen_test': strip(frozen_fsc), 'adapted_test': strip(adapted_fsc), 'baselines': baselines.get('fsc147')},
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/countgd_object_counting_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False, default=str)
assert abs(adapted_val['mae'] - val_history[adapt_result['best_epoch']]) < 1e-6  # the kept epoch re-scores to the number that selected it
print({'report': 'outputs/countgd_object_counting_evaluation_report.json'})

## 9. Re-count the demo scene, export the adapter and reload it

The demo scene from Section 5 is a *training-distribution* scene the fine-tune never saw (its seed lies outside every split's range); it is counted again by the adapted model with the same three prompts and drawn beside the frozen result. The exemplar-only count is the number to watch: the fine-tune trained with both prompts on scenes full of distractors, and in the build record the exemplar-only count moved from 51 to 35 — the circles only — while the text and text-plus-exemplar counts stayed at 35. A different count here is a finding to record, not a failure.

`pipe.save_artifact` writes the trained tensors — the last two decoder layers, the decoder norm and the shared box head, about 14.5 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `countgd.safetensors`, the objective, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `CountGDPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical counts and boxes on four test scenes (VER4).

In [ ]:
import shutil

demo_adapted = {mode: pipe.count(demo['image'], **kwargs)['results'][0] for mode, kwargs in PROMPTS.items()}
demo_rows = []
for mode in PROMPTS:
    before, after = demo_frozen[mode], demo_adapted[mode]
    draw_counts(demo['image'], after).save(f'outputs/countgd_object_counting_demo_adapted_{mode}.png')
    row = {'prompt': mode, 'gold': demo['count'], 'frozen': before['count'], 'adapted': after['count'], 'frozen_box_f1': round(box_metrics([match_boxes(before['boxes'], demo['boxes'])])['f1'], 3), 'adapted_box_f1': round(box_metrics([match_boxes(after['boxes'], demo['boxes'])])['f1'], 3)}
    demo_rows.append(row)
    print(row)
demo_adapted_report = evaluation_report({'results': list(demo_adapted.values())}, [demo['count']] * 3, sample_kind='synthetic')
with open('outputs/countgd_object_counting_demo.json', 'w', encoding='utf-8') as handle:
    json.dump({'scene': demo['id'], 'label': demo['label'], 'gold': demo['count'], 'distractors': demo['distractors'], 'rows': demo_rows, 'frozen_report': demo_report, 'adapted_report': demo_adapted_report}, handle, indent=2)

artifact_dir = Path('outputs/countgd_object_counting_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'countgd_object_counting', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = CountGDPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, tokenizer_dir=TOKENIZER_WEIGHTS_DIR, device=pipe.device)
parity_records = test_records[:4]
before = [pipe.count(r['image'], text=r['label'], exemplars=r['exemplars'])['results'][0] for r in parity_records]
after = [reloaded.count(r['image'], text=r['label'], exemplars=r['exemplars'])['results'][0] for r in parity_records]
parity = {
    'identical_counts': int(sum(a['count'] == b['count'] for a, b in zip(before, after, strict=True))),
    'max_abs_box_difference': float(max((abs(x - y) for a, b in zip(before, after, strict=True) for p, q in zip(a['boxes'], b['boxes'], strict=True) for x, y in zip(p, q, strict=True)), default=0.0)),
    'max_abs_score_difference': float(max((abs(x - y) for a, b in zip(before, after, strict=True) for x, y in zip(a['scores'], b['scores'], strict=True)), default=0.0)),
    'of': len(parity_records),
}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_counts'] == parity['of'] and parity['max_abs_box_difference'] < 0.05 and parity['max_abs_score_difference'] < 1e-4

write_provenance('outputs/provenance.json', pipeline=pipe)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'fetched_this_run': fetched, 'source_file': SOURCE_CKPT_NAME, 'source_sha256': SOURCE_CKPT_SHA256, 'pickle_audit_sha256': PICKLE_AUDIT_SHA256, 'weight_file': MODEL_FILENAME, 'weight_format': 'safetensors, converted once from the audited pickle, digest-verified', 'weight_sha256': MODEL_SHA256},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'license': CORPUS_LICENSE, 'base_url': CORPUS_BASE_URL, 'bytes': CORPUS_BYTES, 'pinned_photographs': len(SAMPLE_RECORDS), 'scored_photographs': len(fsc_test)},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'demo': demo_rows},
    'comparison': comparison,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'numpy': numpy.__version__, 'device': str(pipe.device), 'dtype': 'float32', 'checkpoint_source': pipe.checkpoint_source},
}
with open('outputs/countgd_object_counting_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False, default=str)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen model counts what a prompt names on scenes and photographs it never trained on — MAE 7.42 on the held-out synthetic scenes and 4.54 on 24 FSC-147 photographs in the build record, beside 10.00 / 16.21 for the mean count and 14.50 / 34.00 for the template matcher — and its boxes enclose the objects it counts (box F1 0.453 at IoU ≥ 0.5 on the scenes). What the three prompt modes on the demo scene show is where it errs: exemplar boxes alone describe *what an object looks like* and count look-alike distractors; the text is what separates a blue circle from a green triangle. A bounded fine-tune of 3.6 M of its 233 M parameters on 24 scenes with distractors, chosen on the validation count error, moves the held-out synthetic MAE from 7.42 to 0.92 and the FSC-147 MAE from 4.54 to 3.50, and exports a 14.5 MB adapter that reloads to identical counts. That is the claim: the adaptation contract works end to end, the selector keeps the frozen model when nothing beats it, and the numbers are read on count, points and boxes against non-neural baselines and the frozen model rather than in isolation.

The synthetic scenes are a teaching instrument, not a domain: flat colours, one scale per scene, no occlusion, no texture. They were chosen because they carry object boxes FSC-147 does not, and because their distractors make the prompt modes' behaviour visible. Where a fine-tune earns its place is a real domain with a counting failure you can name — cells, cars, seeds, colonies — with boxes or points for a few dozen images; that is what BYOD is for. The test sets are 12 scenes and 24 photographs from one seeded draw, the validation set that picks the epoch is 8 scenes, a count error moves in steps of one object, and the scores are uncalibrated similarities read against a fixed threshold.

Three things to carry to real data. **Baselines first:** the mean count and the template matcher on *your* images are the numbers to read before any adapted one. **Watch the other set:** a fine-tune that helps its own domain can hurt another; keep a held-out set from the original distribution, as FSC-147 is kept here. **Dense images need cropping:** a single 800-pixel pass caps what 900 queries can count; upstream tiles images with many small objects, which this repository does not carry.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can acquire and digest-verify the pinned snapshots, audit and convert a pickle checkpoint into a digest-pinned safetensors file, fetch and digest-verify a real photograph set, validate the demonstrated dataset contract without leakage, execute the inference contract and a bounded counting fine-tune, evaluate against non-neural baselines and the frozen model on held-out scenes and photographs, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, performance on the FSC-147 benchmark, counting quality on any other population or camera, or production fitness.

**Optional experiments (they do not affect the default path):** count the demo scene with `threshold=0.35` and watch the exemplar-only count fall; set `TRAINABLE_LAYERS = 0` to train the box head alone; fine-tune on `fsc_splits['train']` (points only, upstream's 2×2-pixel boxes) and read what happens to the synthetic box IoU; or bring your own images through BYOD and read the baselines before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/countgd-object-counting-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/countgd-object-counting-pipeline/blob/main/MODEL_CARD.md
- Weight provenance, the pickle audit and the conversion: https://github.com/kurtvalcorza/countgd-object-counting-pipeline/blob/main/docs/WEIGHTS.md
- Upstream Space (pinned checkpoint host): https://huggingface.co/spaces/nikigoli/countgd
- Upstream code: https://github.com/niki-amini-naieni/CountGD (MIT; the GroundingDINO parts carry IDEA's Apache-2.0 header)
- CountGD: Multi-Modal Open-World Counting (Amini-Naieni, Han and Zisserman, NeurIPS 2024): https://arxiv.org/abs/2407.04619
- Grounding DINO (Liu et al., 2023): https://arxiv.org/abs/2303.05499
- FSC-147 / Learning To Count Everything (Ranjan, Sharma, Nguyen and Hoai, CVPR 2021): https://github.com/cvlab-stonybrook/LearningToCountEverything — Hub mirror https://huggingface.co/datasets/isentropic/FSC147
- DIMER Notebook Specification 2.0 and Model Card Specification 1.2 (fleet specs in the ml-worker repository)